In [1]:
# ============================================================
# RNDC PASTO 2018–2024
# CELDA 1 — ENTORNO, LIBRERÍAS E IMPORTS
# ============================================================
#
# OBJETIVO:
# Crear un entorno completamente reproducible en Google Colab.
#
# REGLAS DEL NOTEBOOK:
# - NO depende de Google Drive.
# - NO depende de archivos locales del estudiante.
# - Los datos se obtendrán de fuentes públicas/reproducibles.
# - Los archivos temporales se guardarán en /content.
# - Cada etapa tendrá controles antes de continuar.
# ============================================================


# ------------------------------------------------------------
# 1. INSTALAR SOLO LAS DEPENDENCIAS NECESARIAS
# ------------------------------------------------------------

import sys
import subprocess
import importlib.util


PAQUETES = {
    "python-calamine": "python_calamine",
    "pyarrow": "pyarrow",
    "beautifulsoup4": "bs4",
}


for paquete, modulo in PAQUETES.items():

    if importlib.util.find_spec(modulo) is None:

        print(f"Instalando {paquete}...")

        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            paquete
        ])


# ------------------------------------------------------------
# 2. IMPORTS
# ------------------------------------------------------------

import os
import re
import gc
import json
import time
import hashlib
import warnings

from pathlib import Path
from urllib.parse import urljoin

import requests
import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq

from bs4 import BeautifulSoup
from IPython.display import display


# ------------------------------------------------------------
# 3. CONFIGURACIÓN GENERAL
# ------------------------------------------------------------

warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)

pd.set_option(
    "display.max_columns",
    100
)

pd.set_option(
    "display.width",
    160
)

pd.set_option(
    "display.max_rows",
    100
)


# ------------------------------------------------------------
# 4. DIRECTORIOS TEMPORALES
# ------------------------------------------------------------

ROOT = Path(
    "/content/RNDC_PASTO"
)

DIR_RAW = (
    ROOT
    /
    "raw"
)

DIR_CACHE = (
    ROOT
    /
    "cache"
)

DIR_OUTPUT = (
    ROOT
    /
    "output"
)

DIR_AUDIT = (
    ROOT
    /
    "audit"
)


for carpeta in [
    ROOT,
    DIR_RAW,
    DIR_CACHE,
    DIR_OUTPUT,
    DIR_AUDIT
]:

    carpeta.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# 5. PARÁMETROS CENTRALES DE LA INVESTIGACIÓN
# ------------------------------------------------------------

ANIO_INICIAL = 2018
ANIO_FINAL = 2024

CODIGO_PASTO = 52001

MUNICIPIO_OBJETIVO = "PASTO"

DEPARTAMENTO_OBJETIVO = "NARIÑO"


# Se incluye diciembre de 2017 exclusivamente para poder
# calcular correctamente el primer cambio:
#
# 2017-12 → 2018-01

PERIODO_BASE = "201712"


PERIODOS_ESTUDIO = [

    f"{anio}{mes:02d}"

    for anio in range(
        ANIO_INICIAL,
        ANIO_FINAL + 1
    )

    for mes in range(
        1,
        13
    )
]


PERIODOS_REQUERIDOS = [

    PERIODO_BASE

] + PERIODOS_ESTUDIO


# ------------------------------------------------------------
# 6. CONTROLES
# ------------------------------------------------------------

assert len(
    PERIODOS_ESTUDIO
) == 84, (
    "Deben existir exactamente 84 meses entre "
    "enero de 2018 y diciembre de 2024."
)


assert len(
    PERIODOS_REQUERIDOS
) == 85, (
    "Deben existir 85 periodos incluyendo diciembre de 2017."
)


assert (
    PERIODOS_ESTUDIO[0]
    ==
    "201801"
)


assert (
    PERIODOS_ESTUDIO[-1]
    ==
    "202412"
)


assert (
    PERIODOS_REQUERIDOS[0]
    ==
    "201712"
)


# ------------------------------------------------------------
# 7. VERSIONES
# ------------------------------------------------------------

print(
    "=" * 90
)

print(
    "RNDC PASTO 2018–2024"
)

print(
    "CELDA 1 — ENTORNO"
)

print(
    "=" * 90
)


print(
    "Python:",
    sys.version.split()[0]
)

print(
    "Pandas:",
    pd.__version__
)

print(
    "NumPy:",
    np.__version__
)

print(
    "PyArrow:",
    pyarrow.__version__
)


print(
    "\nDirectorio de trabajo:",
    ROOT
)


print(
    "Periodos de estudio:",
    PERIODOS_ESTUDIO[0],
    "→",
    PERIODOS_ESTUDIO[-1]
)


print(
    "Número de meses de estudio:",
    len(
        PERIODOS_ESTUDIO
    )
)


print(
    "Periodo adicional para rezago:",
    PERIODO_BASE
)


print(
    "\n"
    + "=" * 90
)

print(
    "PASS — CELDA 1"
)

print(
    "Entorno reproducible creado correctamente."
)

print(
    "No se utilizó Google Drive."
)

print(
    "=" * 90
)

RNDC PASTO 2018–2024
CELDA 1 — ENTORNO
Python: 3.13.18
Pandas: 2.2.3
NumPy: 2.1.3
PyArrow: 23.0.1

Directorio de trabajo: /content/RNDC_PASTO
Periodos de estudio: 201801 → 202412
Número de meses de estudio: 84
Periodo adicional para rezago: 201712

PASS — CELDA 1
Entorno reproducible creado correctamente.
No se utilizó Google Drive.


In [2]:
# ============================================================
# RNDC PASTO 2018–2024
# CELDA 2 — CONTRATO DE DATOS Y FUNCIONES DE VALIDACIÓN
# ============================================================

import unicodedata


# ------------------------------------------------------------
# 1. COLUMNAS RNDC NECESARIAS
# ------------------------------------------------------------

COLUMNAS_RNDC = [
    "CODMUNICIPIOORIGEN",
    "MUNICIPIOORIGEN",
    "DEPARTAMENTOORIGEN",

    "CODMUNICIPIODESTINO",
    "MUNICIPIODESTINO",
    "DEPARTAMENTODESTINO",

    "CODMERCANCIA",
    "MERCANCIA",

    "VIAJESTOTALES",
    "VIAJESVALORCERO",

    "KILOGRAMOS",
    "KILOMETROS",

    "VALORESPAGADOS"
]


COLUMNAS_NUMERICAS = [
    "VIAJESTOTALES",
    "VIAJESVALORCERO",
    "KILOGRAMOS",
    "KILOMETROS",
    "VALORESPAGADOS"
]


# ------------------------------------------------------------
# 2. NORMALIZACIÓN DE TEXTO
# ------------------------------------------------------------

def normalizar_texto(valor):

    if pd.isna(valor):
        return ""

    valor = str(valor).strip().lower()

    valor = unicodedata.normalize(
        "NFKD",
        valor
    )

    valor = "".join(
        c
        for c in valor
        if not unicodedata.combining(c)
    )

    valor = re.sub(
        r"\s+",
        " ",
        valor
    )

    return valor


# ------------------------------------------------------------
# 3. NORMALIZACIÓN DE CÓDIGOS MUNICIPALES
# ------------------------------------------------------------

def normalizar_codigo_municipio(serie):

    return pd.to_numeric(
        serie,
        errors="coerce"
    ).astype("Int64")


# ------------------------------------------------------------
# 4. FINALIZAR BASE MENSUAL DE PASTO
# ------------------------------------------------------------

def finalizar_base_pasto(df, periodo):

    df = df.copy()


    # --------------------------------------------------------
    # CONTROL DE COLUMNAS
    # --------------------------------------------------------

    faltantes = [
        c
        for c in COLUMNAS_RNDC
        if c not in df.columns
    ]

    if faltantes:

        raise ValueError(
            f"{periodo}: faltan columnas obligatorias: "
            f"{faltantes}"
        )


    # --------------------------------------------------------
    # PERIODO
    # --------------------------------------------------------

    df["PERIODO"] = str(
        periodo
    )


    # --------------------------------------------------------
    # CÓDIGOS MUNICIPALES
    # --------------------------------------------------------

    df[
        "CODMUNICIPIOORIGEN_NUM"
    ] = normalizar_codigo_municipio(
        df["CODMUNICIPIOORIGEN"]
    )


    df[
        "CODMUNICIPIODESTINO_NUM"
    ] = normalizar_codigo_municipio(
        df["CODMUNICIPIODESTINO"]
    )


    # --------------------------------------------------------
    # TEXTO NORMALIZADO
    # --------------------------------------------------------

    for columna in [
        "MUNICIPIOORIGEN",
        "DEPARTAMENTOORIGEN",
        "MUNICIPIODESTINO",
        "DEPARTAMENTODESTINO",
        "MERCANCIA"
    ]:

        df[
            f"{columna}_NORM"
        ] = (
            df[columna]
            .map(normalizar_texto)
        )


    # --------------------------------------------------------
    # VARIABLES NUMÉRICAS
    #
    # IMPORTANTE:
    # errors="coerce" NO es imputación.
    # Solo identifica valores no interpretables como número.
    # --------------------------------------------------------

    for columna in COLUMNAS_NUMERICAS:

        df[columna] = pd.to_numeric(
            df[columna],
            errors="coerce"
        )


    # --------------------------------------------------------
    # FILTRO PASTO
    #
    # Criterio principal:
    # DANE 52001
    #
    # Respaldo:
    # nombre Pasto + departamento Nariño
    # --------------------------------------------------------

    filtro_codigo = (
        df[
            "CODMUNICIPIODESTINO_NUM"
        ]
        ==
        CODIGO_PASTO
    )


    filtro_texto = (

        df[
            "MUNICIPIODESTINO_NORM"
        ]
        .str.contains(
            r"\bpasto\b",
            regex=True,
            na=False
        )

        &

        df[
            "DEPARTAMENTODESTINO_NORM"
        ]
        .str.contains(
            r"\bnarino\b",
            regex=True,
            na=False
        )
    )


    df_pasto = (

        df.loc[
            filtro_codigo
            |
            filtro_texto
        ]

        .copy()

        .reset_index(
            drop=True
        )
    )


    return df_pasto


# ------------------------------------------------------------
# 5. VALIDAR UN MES
# ------------------------------------------------------------

def validar_mes_pasto(df, periodo):

    errores = []


    if len(df) == 0:

        errores.append(
            "0 filas para Pasto"
        )


    if "PERIODO" not in df.columns:

        errores.append(
            "falta PERIODO"
        )

    else:

        periodos_en_df = (
            df["PERIODO"]
            .astype(str)
            .unique()
            .tolist()
        )

        if periodos_en_df != [
            str(periodo)
        ]:

            errores.append(
                f"periodo inconsistente: {periodos_en_df}"
            )


    faltantes = [
        c
        for c in COLUMNAS_RNDC
        if c not in df.columns
    ]


    if faltantes:

        errores.append(
            f"columnas faltantes: {faltantes}"
        )


    if errores:

        raise ValueError(
            f"{periodo}: "
            +
            " | ".join(
                errores
            )
        )


    return True


# ------------------------------------------------------------
# 6. HASH SHA-256
#
# Permitirá auditar exactamente los archivos descargados.
# ------------------------------------------------------------

def sha256_archivo(ruta):

    h = hashlib.sha256()


    with open(
        ruta,
        "rb"
    ) as f:

        for bloque in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b""
        ):

            h.update(
                bloque
            )


    return h.hexdigest()


# ------------------------------------------------------------
# 7. VALIDAR COBERTURA TEMPORAL
# ------------------------------------------------------------

def validar_cobertura(periodos_observados):

    observados = sorted(
        set(
            str(x)
            for x in periodos_observados
        )
    )


    requeridos = sorted(
        PERIODOS_REQUERIDOS
    )


    faltantes = sorted(
        set(requeridos)
        -
        set(observados)
    )


    extras = sorted(
        set(observados)
        -
        set(requeridos)
    )


    return {
        "completa":
            len(faltantes) == 0,

        "n_observados":
            len(observados),

        "n_requeridos":
            len(requeridos),

        "faltantes":
            faltantes,

        "extras":
            extras
    }


# ------------------------------------------------------------
# 8. AUTOPRUEBAS
# ------------------------------------------------------------

assert (
    normalizar_texto("NARIÑO")
    ==
    "narino"
)


assert (
    normalizar_texto("  San Juan de Pasto  ")
    ==
    "san juan de pasto"
)


prueba_cobertura = validar_cobertura(
    PERIODOS_REQUERIDOS
)


assert prueba_cobertura[
    "completa"
]


assert prueba_cobertura[
    "n_observados"
] == 85


# ------------------------------------------------------------
# 9. RESULTADO
# ------------------------------------------------------------

print(
    "=" * 90
)

print(
    "PASS — CELDA 2"
)

print(
    "=" * 90
)

print(
    "Columnas RNDC obligatorias:",
    len(COLUMNAS_RNDC)
)

print(
    "Variables numéricas:",
    len(COLUMNAS_NUMERICAS)
)

print(
    "Municipio objetivo:",
    MUNICIPIO_OBJETIVO,
    "| DANE:",
    CODIGO_PASTO
)

print(
    "Control temporal:",
    len(PERIODOS_REQUERIDOS),
    "periodos"
)

print(
    "Normalización y validadores cargados correctamente."
)

print(
    "=" * 90
)

PASS — CELDA 2
Columnas RNDC obligatorias: 13
Variables numéricas: 5
Municipio objetivo: PASTO | DANE: 52001
Control temporal: 85 periodos
Normalización y validadores cargados correctamente.


In [9]:
# ============================================================
# RNDC PASTO 2018–2024
# CELDA 5 — MOTOR RNDC ACTUAL + PRUEBA REAL 202001
# ============================================================
#
# OBJETIVO:
# Validar de extremo a extremo el flujo reproducible:
#
# RNDC oficial
#     ↓
# Excel mensual nacional
#     ↓
# validación de archivo
#     ↓
# lectura con Calamine
#     ↓
# selección de columnas necesarias
#     ↓
# filtro Pasto
#     ↓
# Parquet compacto
#     ↓
# auditoría SHA-256
#
# PRUEBA CONTROL:
#     202001
#
# NO usa Google Drive.
# ============================================================


# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

URL_RNDC_ACTUAL = (
    "https://rndc.mintransporte.gov.co/"
    "MenuPrincipal.aspx"
)


HEADERS_RNDC = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(X11; Linux x86_64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/130.0 Safari/537.36"
    ),
    "Accept-Language":
        "es-CO,es;q=0.9,en;q=0.8"
}


PERIODO_PRUEBA = "202001"


# ------------------------------------------------------------
# 2. NORMALIZAR NOMBRES DE COLUMNAS
# ------------------------------------------------------------

def clave_columna(nombre):

    texto = normalizar_texto(
        nombre
    )

    texto = re.sub(
        r"[^a-z0-9]",
        "",
        texto
    )

    return texto.upper()


MAPA_COLUMNAS_ESPERADAS = {

    clave_columna(c): c

    for c in COLUMNAS_RNDC
}


# ------------------------------------------------------------
# 3. DETECTAR CAPTCHA
# ------------------------------------------------------------

def detectar_captcha_actual(soup):

    texto = soup.get_text(
        " ",
        strip=True
    )

    patrones = [
        r"cu[aá]nto\s+es\s+(\d+)\s*\+\s*(\d+)",
        r"antes\s+de\s+consultar[^0-9]*(\d+)\s*\+\s*(\d+)",
        r"responder[^0-9]*(\d+)\s*\+\s*(\d+)",
        r"(\d+)\s*\+\s*(\d+)"
    ]


    for patron in patrones:

        m = re.search(
            patron,
            texto,
            flags=re.I
        )

        if m:

            n1 = int(
                m.group(1)
            )

            n2 = int(
                m.group(2)
            )

            return {
                "n1": n1,
                "n2": n2,
                "resultado": n1 + n2
            }


    return None


# ------------------------------------------------------------
# 4. BUSCAR CAMPOS DEL FORMULARIO
# ------------------------------------------------------------

def encontrar_campo_actual(
    soup,
    patrones
):

    for tag in soup.find_all(
        [
            "input",
            "button",
            "select"
        ]
    ):

        nombre = (
            tag.get("name")
            or
            ""
        )

        identificador = (
            tag.get("id")
            or
            ""
        )

        texto = (
            nombre
            +
            " "
            +
            identificador
        ).lower()


        for patron in patrones:

            if patron.lower() in texto:

                return tag


    return None


# ------------------------------------------------------------
# 5. DESCARGA RNDC ACTUAL
# ------------------------------------------------------------

def descargar_rndc_actual(
    periodo,
    sobrescribir=False
):

    ruta = (
        DIR_RAW
        /
        f"EstadisticasRNDC_{periodo}.xlsx"
    )


    # --------------------------------------------------------
    # REUTILIZAR ARCHIVO VÁLIDO
    # --------------------------------------------------------

    if (
        ruta.exists()
        and
        not sobrescribir
    ):

        with open(
            ruta,
            "rb"
        ) as f:

            firma = f.read(
                2
            )


        if firma == b"PK":

            print(
                periodo,
                "→ archivo local válido reutilizado."
            )

            return ruta


    # --------------------------------------------------------
    # SESIÓN
    # --------------------------------------------------------

    ses = requests.Session()

    ses.headers.update(
        HEADERS_RNDC
    )


    # --------------------------------------------------------
    # GET FORMULARIO
    # --------------------------------------------------------

    r_get = ses.get(
        URL_RNDC_ACTUAL,
        timeout=(10, 30)
    )

    r_get.raise_for_status()


    soup = BeautifulSoup(
        r_get.text,
        "html.parser"
    )


    # --------------------------------------------------------
    # CAMPOS
    # --------------------------------------------------------

    campo_periodo = encontrar_campo_actual(
        soup,
        [
            "anomesinicial",
            "anomes",
            "periodo"
        ]
    )


    campo_resultado = encontrar_campo_actual(
        soup,
        [
            "resultado"
        ]
    )


    campo_boton = encontrar_campo_actual(
        soup,
        [
            "btestadisticas",
            "estadisticas"
        ]
    )


    assert campo_periodo is not None, (
        "No se encontró el campo de periodo."
    )


    assert campo_resultado is not None, (
        "No se encontró el campo Resultado."
    )


    assert campo_boton is not None, (
        "No se encontró el botón de estadísticas."
    )


    nombre_periodo = campo_periodo.get(
        "name"
    )

    nombre_resultado = campo_resultado.get(
        "name"
    )

    nombre_boton = campo_boton.get(
        "name"
    )

    valor_boton = (
        campo_boton.get(
            "value"
        )
        or
        "del Mes"
    )


    # --------------------------------------------------------
    # CAPTCHA
    # --------------------------------------------------------

    captcha = detectar_captcha_actual(
        soup
    )


    assert captcha is not None, (
        "No fue posible detectar la validación aritmética."
    )


    # --------------------------------------------------------
    # PAYLOAD ASP.NET
    # --------------------------------------------------------

    payload = {}


    for inp in soup.find_all(
        "input",
        type="hidden"
    ):

        nombre = inp.get(
            "name"
        )

        if nombre:

            payload[
                nombre
            ] = inp.get(
                "value",
                ""
            )


    payload[
        nombre_periodo
    ] = str(
        periodo
    )


    payload[
        nombre_resultado
    ] = str(
        captcha["resultado"]
    )


    payload[
        nombre_boton
    ] = valor_boton


    if "__EVENTTARGET" in payload:

        payload[
            "__EVENTTARGET"
        ] = ""


    if "__EVENTARGUMENT" in payload:

        payload[
            "__EVENTARGUMENT"
        ] = ""


    formulario = soup.find(
        "form"
    )


    assert formulario is not None


    url_post = urljoin(
        r_get.url,
        formulario.get(
            "action",
            ""
        )
    )


    # --------------------------------------------------------
    # DESCARGA EN STREAM
    # --------------------------------------------------------

    print(
        periodo,
        "→ solicitando RNDC..."
    )


    inicio = time.time()


    r_post = ses.post(
        url_post,
        data=payload,
        headers={
            "Referer":
                r_get.url
        },
        timeout=(10, 300),
        allow_redirects=True,
        stream=True
    )


    r_post.raise_for_status()


    iterator = r_post.iter_content(
        chunk_size=
            1024 * 1024
    )


    primer_chunk = next(
        iterator,
        b""
    )


    # --------------------------------------------------------
    # VALIDACIÓN DE FIRMA XLSX
    # --------------------------------------------------------

    if not primer_chunk.startswith(
        b"PK"
    ):

        muestra = (
            primer_chunk[:200]
            .decode(
                "utf-8",
                errors="ignore"
            )
        )


        r_post.close()


        raise RuntimeError(
            f"{periodo}: RNDC no devolvió XLSX. "
            f"Inicio de respuesta: {muestra!r}"
        )


    # --------------------------------------------------------
    # GUARDAR
    # --------------------------------------------------------

    with open(
        ruta,
        "wb"
    ) as f:

        f.write(
            primer_chunk
        )


        for chunk in iterator:

            if chunk:

                f.write(
                    chunk
                )


    r_post.close()


    segundos = (
        time.time()
        -
        inicio
    )


    print(
        periodo,
        "→ descargado:",
        f"{ruta.stat().st_size / 1024**2:.2f} MB",
        "|",
        f"{segundos:.1f} s"
    )


    return ruta


# ------------------------------------------------------------
# 6. LEER SOLO COLUMNAS NECESARIAS
# ------------------------------------------------------------

def leer_rndc_columnas_necesarias(
    ruta
):

    libro = pd.ExcelFile(
        ruta,
        engine="calamine"
    )


    print(
        "Hojas:",
        libro.sheet_names
    )


    if (
        "Exportar Hoja de Trabajo"
        in
        libro.sheet_names
    ):

        hoja = (
            "Exportar Hoja de Trabajo"
        )

    else:

        hoja = libro.sheet_names[
            0
        ]


    # --------------------------------------------------------
    # LEER SOLO CABECERA
    # --------------------------------------------------------

    cabecera = pd.read_excel(
        libro,
        sheet_name=hoja,
        nrows=0
    )


    columnas_originales = (
        cabecera.columns
        .tolist()
    )


    mapa_original = {

        clave_columna(c): c

        for c
        in columnas_originales
    }


    faltantes = [

        original

        for clave, original
        in MAPA_COLUMNAS_ESPERADAS.items()

        if clave
        not in
        mapa_original
    ]


    if faltantes:

        print(
            "\nColumnas originales encontradas:"
        )

        print(
            columnas_originales
        )


        raise ValueError(
            "El archivo RNDC no contiene todas "
            f"las columnas necesarias: {faltantes}"
        )


    columnas_a_leer = [

        mapa_original[
            clave_columna(c)
        ]

        for c
        in COLUMNAS_RNDC
    ]


    # --------------------------------------------------------
    # LECTURA SELECTIVA
    # --------------------------------------------------------

    df = pd.read_excel(
        libro,
        sheet_name=hoja,
        usecols=columnas_a_leer
    )


    # --------------------------------------------------------
    # RENOMBRAR AL CONTRATO CANÓNICO
    # --------------------------------------------------------

    renombrar = {}


    for c in df.columns:

        clave = clave_columna(
            c
        )

        if clave in MAPA_COLUMNAS_ESPERADAS:

            renombrar[
                c
            ] = MAPA_COLUMNAS_ESPERADAS[
                clave
            ]


    df = df.rename(
        columns=renombrar
    )


    return df


# ------------------------------------------------------------
# 7. DESCARGAR PERIODO DE PRUEBA
# ------------------------------------------------------------

print(
    "=" * 100
)

print(
    "CELDA 5 — PRUEBA REAL RNDC"
)

print(
    "=" * 100
)


ruta_202001 = descargar_rndc_actual(
    PERIODO_PRUEBA
)


# ------------------------------------------------------------
# 8. HASH DEL ARCHIVO ORIGINAL
# ------------------------------------------------------------

hash_202001 = sha256_archivo(
    ruta_202001
)


print(
    "\nSHA-256:"
)

print(
    hash_202001
)


# ------------------------------------------------------------
# 9. LEER RNDC
# ------------------------------------------------------------

inicio_lectura = time.time()


df_202001_nacional = (
    leer_rndc_columnas_necesarias(
        ruta_202001
    )
)


print(
    "\nFilas nacionales:",
    f"{len(df_202001_nacional):,}"
)


print(
    "Tiempo lectura:",
    f"{time.time() - inicio_lectura:.1f} s"
)


print(
    "Columnas:"
)

print(
    df_202001_nacional.columns.tolist()
)


# ------------------------------------------------------------
# 10. FILTRAR PASTO
# ------------------------------------------------------------

df_202001_pasto = finalizar_base_pasto(
    df_202001_nacional,
    PERIODO_PRUEBA
)


validar_mes_pasto(
    df_202001_pasto,
    PERIODO_PRUEBA
)


print(
    "\nFilas con destino Pasto:",
    f"{len(df_202001_pasto):,}"
)


# ------------------------------------------------------------
# 11. GUARDAR PARQUET
# ------------------------------------------------------------

ruta_parquet_202001 = (

    DIR_CACHE
    /
    "RNDC_PASTO_202001.parquet"
)


df_202001_pasto.to_parquet(
    ruta_parquet_202001,
    index=False,
    compression="snappy"
)


assert (
    ruta_parquet_202001.exists()
)


assert (
    ruta_parquet_202001.stat().st_size
    >
    0
)


print(
    "Parquet:",
    ruta_parquet_202001
)


print(
    "Tamaño Parquet:",
    f"{ruta_parquet_202001.stat().st_size / 1024**2:.3f} MB"
)


# ------------------------------------------------------------
# 12. MANIFEST DE AUDITORÍA
# ------------------------------------------------------------

manifest_202001 = pd.DataFrame([
    {
        "periodo":
            PERIODO_PRUEBA,

        "fuente":
            URL_RNDC_ACTUAL,

        "archivo_original":
            ruta_202001.name,

        "sha256":
            hash_202001,

        "filas_nacionales":
            len(
                df_202001_nacional
            ),

        "filas_pasto":
            len(
                df_202001_pasto
            ),

        "parquet":
            ruta_parquet_202001.name
    }
])


ruta_manifest = (

    DIR_AUDIT
    /
    "manifest_prueba_202001.csv"
)


manifest_202001.to_csv(
    ruta_manifest,
    index=False,
    encoding="utf-8-sig"
)


display(
    manifest_202001
)


# ------------------------------------------------------------
# 13. LIBERAR MEMORIA
# ------------------------------------------------------------

del df_202001_nacional

gc.collect()


# ------------------------------------------------------------
# 14. BORRAR EXCEL PESADO
#
# Ya quedó:
# - hash
# - manifest
# - Parquet filtrado
# ------------------------------------------------------------

if ruta_202001.exists():

    ruta_202001.unlink()


assert (
    not ruta_202001.exists()
)


assert (
    ruta_parquet_202001.exists()
)


# ------------------------------------------------------------
# 15. RESULTADO
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 100
)

print(
    "PASS — CELDA 5"
)

print(
    "=" * 100
)


print(
    "202001 descargado desde RNDC oficial."
)

print(
    "Archivo nacional validado como XLSX."
)

print(
    "Columnas obligatorias validadas."
)

print(
    "Pasto filtrado correctamente."
)

print(
    "Parquet compacto generado."
)

print(
    "SHA-256 registrado."
)

print(
    "Excel nacional eliminado."
)

print(
    "No se utilizó Google Drive."
)

print(
    "=" * 100
)

CELDA 5 — PRUEBA REAL RNDC
202001 → solicitando RNDC...
202001 → descargado: 15.54 MB | 1.1 s

SHA-256:
ec1a0ea3c4697a622424472bc681bb3043318961e483170ca16c09ac18b47218
Hojas: ['Exportar Hoja de Trabajo', 'SQL']

Filas nacionales: 134,426
Tiempo lectura: 14.2 s
Columnas:
['CODMUNICIPIOORIGEN', 'MUNICIPIOORIGEN', 'DEPARTAMENTOORIGEN', 'CODMUNICIPIODESTINO', 'MUNICIPIODESTINO', 'DEPARTAMENTODESTINO', 'CODMERCANCIA', 'MERCANCIA', 'VIAJESTOTALES', 'KILOGRAMOS', 'VIAJESVALORCERO', 'KILOMETROS', 'VALORESPAGADOS']

Filas con destino Pasto: 1,156
Parquet: /content/RNDC_PASTO/cache/RNDC_PASTO_202001.parquet
Tamaño Parquet: 0.064 MB


,periodo,fuente,archivo_original,sha256,filas_nacionales,filas_pasto,parquet
0,202001,https://rndc.mintransporte.gov.co/MenuPrincipa...,EstadisticasRNDC_202001.xlsx,ec1a0ea3c4697a622424472bc681bb3043318961e48317...,134426,1156,RNDC_PASTO_202001.parquet



PASS — CELDA 5
202001 descargado desde RNDC oficial.
Archivo nacional validado como XLSX.
Columnas obligatorias validadas.
Pasto filtrado correctamente.
Parquet compacto generado.
SHA-256 registrado.
Excel nacional eliminado.
No se utilizó Google Drive.


In [20]:
# ============================================================
# CELDA 8A — CONTRATO FINAL RNDC PARA VARIABLE 2
# PRUEBA ESTRUCTURAL EN 202401
# ============================================================
#
# OBJETIVO:
# Validar el esquema definitivo de la Variable 2:
# conservar las 23 variables oficiales RNDC,
# incluyendo configuración vehicular.
#
# Solo procesa 202401.
# No modifica todavía el cache maestro 2020–2024.
# ============================================================


# ------------------------------------------------------------
# 1. CONTRATO RNDC COMPLETO
# ------------------------------------------------------------

COLUMNAS_RNDC_FINAL = [

    "MES",

    "COD_CONFIG_VEHICULO",
    "CONFIG_VEHICULO",

    "CODOPERACIONTRANSPORTE",
    "OPERACIONTRANSPORTE",

    "CODTIPOCONTENEDOR",
    "TIPOCONTENEDOR",

    "CODMUNICIPIOORIGEN",
    "MUNICIPIOORIGEN",
    "DEPARTAMENTOORIGEN",

    "CODMUNICIPIODESTINO",
    "MUNICIPIODESTINO",
    "DEPARTAMENTODESTINO",

    "CODMERCANCIA",
    "MERCANCIA",

    "NATURALEZACARGA",

    "VIAJESTOTALES",
    "KILOGRAMOS",
    "GALONES",
    "VIAJESLIQUIDOS",
    "VIAJESVALORCERO",

    "KILOMETROS",
    "VALORESPAGADOS"
]


COLUMNAS_NUMERICAS_FINAL = [

    "VIAJESTOTALES",
    "KILOGRAMOS",
    "GALONES",
    "VIAJESLIQUIDOS",
    "VIAJESVALORCERO",
    "KILOMETROS",
    "VALORESPAGADOS"
]


assert len(
    COLUMNAS_RNDC_FINAL
) == 23


# ------------------------------------------------------------
# 2. LECTOR FINAL
# ------------------------------------------------------------

def leer_rndc_final(ruta):

    libro = pd.ExcelFile(
        ruta,
        engine="calamine"
    )


    if (
        "Exportar Hoja de Trabajo"
        in libro.sheet_names
    ):

        hoja = (
            "Exportar Hoja de Trabajo"
        )

    else:

        hoja = libro.sheet_names[0]


    cabecera = pd.read_excel(
        libro,
        sheet_name=hoja,
        nrows=0
    )


    columnas_originales = [
        str(c).strip()
        for c in cabecera.columns
    ]


    mapa = {

        clave_columna(c):
            c

        for c in columnas_originales
    }


    faltantes = [

        c

        for c in COLUMNAS_RNDC_FINAL

        if clave_columna(c)
        not in mapa
    ]


    if faltantes:

        raise ValueError(
            "Faltan columnas RNDC: "
            f"{faltantes}"
        )


    columnas_reales = [

        mapa[
            clave_columna(c)
        ]

        for c in COLUMNAS_RNDC_FINAL
    ]


    df = pd.read_excel(
        libro,
        sheet_name=hoja,
        usecols=columnas_reales
    )


    renombrar = {}


    for c in df.columns:

        clave = clave_columna(c)


        for canonica in COLUMNAS_RNDC_FINAL:

            if (
                clave
                ==
                clave_columna(
                    canonica
                )
            ):

                renombrar[
                    c
                ] = canonica

                break


    return df.rename(
        columns=renombrar
    )


# ------------------------------------------------------------
# 3. FINALIZADOR FINAL PARA PASTO
# ------------------------------------------------------------

def finalizar_base_pasto_final(
    df,
    periodo
):

    df = df.copy()


    faltantes = [

        c

        for c in COLUMNAS_RNDC_FINAL

        if c not in df.columns
    ]


    if faltantes:

        raise ValueError(
            f"{periodo}: faltan "
            f"{faltantes}"
        )


    df[
        "PERIODO"
    ] = str(
        periodo
    )


    # --------------------------------------------------------
    # CÓDIGOS
    # --------------------------------------------------------

    df[
        "CODMUNICIPIOORIGEN_NUM"
    ] = normalizar_codigo_municipio(
        df[
            "CODMUNICIPIOORIGEN"
        ]
    )


    df[
        "CODMUNICIPIODESTINO_NUM"
    ] = normalizar_codigo_municipio(
        df[
            "CODMUNICIPIODESTINO"
        ]
    )


    # --------------------------------------------------------
    # NUMÉRICAS
    # --------------------------------------------------------

    for c in COLUMNAS_NUMERICAS_FINAL:

        df[c] = pd.to_numeric(
            df[c],
            errors="coerce"
        )


    # --------------------------------------------------------
    # TEXTO NORMALIZADO
    # --------------------------------------------------------

    for c in [

        "CONFIG_VEHICULO",
        "OPERACIONTRANSPORTE",

        "MUNICIPIOORIGEN",
        "DEPARTAMENTOORIGEN",

        "MUNICIPIODESTINO",
        "DEPARTAMENTODESTINO",

        "MERCANCIA",
        "NATURALEZACARGA"
    ]:

        df[
            f"{c}_NORM"
        ] = (
            df[c]
            .map(
                normalizar_texto
            )
        )


    # --------------------------------------------------------
    # DESTINO PASTO
    # --------------------------------------------------------

    filtro_codigo = (

        df[
            "CODMUNICIPIODESTINO_NUM"
        ]
        ==
        CODIGO_PASTO
    )


    filtro_texto = (

        df[
            "MUNICIPIODESTINO_NORM"
        ]
        .str.contains(
            r"\bpasto\b",
            regex=True,
            na=False
        )

        &

        df[
            "DEPARTAMENTODESTINO_NORM"
        ]
        .str.contains(
            r"\bnarino\b",
            regex=True,
            na=False
        )
    )


    return (

        df.loc[
            filtro_codigo
            |
            filtro_texto
        ]

        .copy()

        .reset_index(
            drop=True
        )
    )


# ------------------------------------------------------------
# 4. PRUEBA SOLO 202401
# ------------------------------------------------------------

PERIODO_TEST_FINAL = (
    "202401"
)


print(
    "=" * 100
)

print(
    "VARIABLE 2 — AUDITORÍA DEL CONTRATO FINAL"
)

print(
    "Periodo:",
    PERIODO_TEST_FINAL
)

print(
    "=" * 100
)


ruta_test = descargar_rndc_actual(
    PERIODO_TEST_FINAL,
    sobrescribir=True
)


df_test_nacional = leer_rndc_final(
    ruta_test
)


print(
    "\nFilas nacionales:",
    f"{len(df_test_nacional):,}"
)

print(
    "Columnas:",
    len(
        df_test_nacional.columns
    )
)


assert (
    set(
        COLUMNAS_RNDC_FINAL
    )
    <=
    set(
        df_test_nacional.columns
    )
)


# ------------------------------------------------------------
# 5. FILTRAR PASTO
# ------------------------------------------------------------

df_test_pasto = finalizar_base_pasto_final(
    df_test_nacional,
    PERIODO_TEST_FINAL
)


print(
    "\nFilas destino Pasto:",
    f"{len(df_test_pasto):,}"
)


assert (
    len(
        df_test_pasto
    )
    >
    0
)


# ------------------------------------------------------------
# 6. AUDITORÍA DE CONFIGURACIÓN
# ------------------------------------------------------------

audit_config = (

    df_test_pasto[
        [
            "COD_CONFIG_VEHICULO",
            "CONFIG_VEHICULO"
        ]
    ]

    .drop_duplicates()

    .sort_values(
        "COD_CONFIG_VEHICULO"
    )

    .reset_index(
        drop=True
    )
)


print(
    "\nConfiguraciones vehiculares:"
)

display(
    audit_config
)


# ------------------------------------------------------------
# 7. AUDITORÍA DE CELDAS
# ------------------------------------------------------------

n_origenes = (
    df_test_pasto[
        "CODMUNICIPIOORIGEN"
    ]
    .nunique()
)


n_config = (
    df_test_pasto[
        "COD_CONFIG_VEHICULO"
    ]
    .nunique()
)


n_mercancias = (
    df_test_pasto[
        "CODMERCANCIA"
    ]
    .nunique()
)


n_celdas = (

    df_test_pasto[
        [
            "CODMUNICIPIOORIGEN",
            "COD_CONFIG_VEHICULO",
            "CODMERCANCIA"
        ]
    ]

    .drop_duplicates()

    .shape[0]
)


print(
    "\nOrígenes:",
    n_origenes
)

print(
    "Configuraciones:",
    n_config
)

print(
    "Mercancías:",
    n_mercancias
)

print(
    "Celdas "
    "origen × configuración × mercancía:",
    n_celdas
)


# ------------------------------------------------------------
# 8. VARIABLES MONETARIAS
# ------------------------------------------------------------

viajes_con_valor = (

    df_test_pasto[
        "VIAJESTOTALES"
    ]

    -

    df_test_pasto[
        "VIAJESVALORCERO"
    ]
)


validos_precio = (

    viajes_con_valor.gt(0)

    &

    df_test_pasto[
        "VALORESPAGADOS"
    ].gt(0)
)


print(
    "\nRegistros con viajes con valor > 0:",
    int(
        validos_precio.sum()
    )
)

print(
    "Porcentaje:",
    round(
        100
        *
        validos_precio.mean(),
        2
    ),
    "%"
)


# ------------------------------------------------------------
# 9. NO GUARDAMOS TODAVÍA EN CACHE MAESTRO
# ------------------------------------------------------------

if ruta_test.exists():

    ruta_test.unlink()


print(
    "\n"
    + "=" * 100
)

print(
    "PASS — CELDA 8A"
)

print(
    "El esquema final RNDC contiene "
    "las 23 variables oficiales."
)

print(
    "La estructura necesaria para "
    "origen × configuración × mercancía "
    "está disponible."
)

print(
    "No se modificó todavía el cache maestro."
)

print(
    "=" * 100
)

VARIABLE 2 — AUDITORÍA DEL CONTRATO FINAL
Periodo: 202401
202401 → solicitando RNDC...
202401 → descargado: 23.27 MB | 1.5 s

Filas nacionales: 165,284
Columnas: 23

Filas destino Pasto: 1,221

Configuraciones vehiculares:


,COD_CONFIG_VEHICULO,CONFIG_VEHICULO
0,2,Camión Rígido de 2 ejes
1,2S2,Tractocamión de 2 ejes Semiremolque de 2 ejes
2,2S3,Tractocamión de 2 ejes Semiremolque de 3 ejes
3,3,Camión Rígido de 3 ejes
4,3S2,Tractocamión de 3 ejes Semiremolque de 2 ejes
5,3S3,Tractocamión de 3 ejes Semiremolque de 3 ejes
6,4,Camión Rígido de 4 ejes
7,CA,Camioneta de 2 ejes



Orígenes: 161
Configuraciones: 8
Mercancías: 354
Celdas origen × configuración × mercancía: 1205

Registros con viajes con valor > 0: 1128
Porcentaje: 92.38 %

PASS — CELDA 8A
El esquema final RNDC contiene las 23 variables oficiales.
La estructura necesaria para origen × configuración × mercancía está disponible.
No se modificó todavía el cache maestro.


In [21]:
# ============================================================
# CELDA 8B — BASE RNDC FINAL 2020–2024
# 23 VARIABLES OFICIALES + DESTINO PASTO
# ============================================================
#
# OBJETIVO:
# Regenerar los 60 meses RNDC (202001–202412)
# usando el contrato definitivo validado en la Celda 8A.
#
# CARACTERÍSTICAS:
# - 23 variables oficiales RNDC.
# - Destino Pasto.
# - Conserva configuración vehicular.
# - Descarga oficial reproducible.
# - Hasta 3 meses en paralelo.
# - Cache FINAL separado del cache viejo.
# - SHA-256 de originales y Parquet.
# - Reanudable.
# - El Excel nacional se elimina después de validar.
#
# NO usa Google Drive.
# ============================================================


from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import gc
import time
import hashlib
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. DIRECTORIOS FINALES
# ------------------------------------------------------------

DIR_CACHE_FINAL = (
    ROOT
    /
    "cache_final_23c"
)


DIR_AUDIT_FINAL = (
    ROOT
    /
    "audit_final_23c"
)


DIR_CACHE_FINAL.mkdir(
    parents=True,
    exist_ok=True
)


DIR_AUDIT_FINAL.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. PERIODOS
# ------------------------------------------------------------

PERIODOS_RNDC_FINAL = [

    f"{anio}{mes:02d}"

    for anio in range(
        2020,
        2025
    )

    for mes in range(
        1,
        13
    )
]


assert len(
    PERIODOS_RNDC_FINAL
) == 60


MAX_WORKERS_FINAL = 3


# ------------------------------------------------------------
# 3. MANIFEST FINAL
# ------------------------------------------------------------

RUTA_MANIFEST_FINAL = (
    DIR_AUDIT_FINAL
    /
    "manifest_rndc_final_2020_2024.csv"
)


COLUMNAS_MANIFEST_FINAL = [

    "periodo",

    "fuente",

    "archivo_original",
    "sha256_original",

    "filas_nacionales",
    "filas_pasto",

    "n_origenes",
    "n_configuraciones",
    "n_mercancias",
    "n_celdas",

    "registros_valor_positivo",
    "pct_registros_valor_positivo",

    "parquet",
    "sha256_parquet",
    "tamano_parquet_mb",

    "segundos_total",

    "estado"
]


if RUTA_MANIFEST_FINAL.exists():

    manifest_final = pd.read_csv(
        RUTA_MANIFEST_FINAL,
        dtype={
            "periodo": str
        }
    )

else:

    manifest_final = pd.DataFrame(
        columns=COLUMNAS_MANIFEST_FINAL
    )


# ------------------------------------------------------------
# 4. HASH
# ------------------------------------------------------------

def sha256_archivo_final(
    ruta,
    chunk_size=1024 * 1024
):

    h = hashlib.sha256()


    with open(
        ruta,
        "rb"
    ) as f:

        while True:

            bloque = f.read(
                chunk_size
            )

            if not bloque:
                break

            h.update(
                bloque
            )


    return h.hexdigest()


# ------------------------------------------------------------
# 5. VALIDADOR DEL PARQUET FINAL
# ------------------------------------------------------------

def validar_parquet_final(
    df,
    periodo
):

    requeridas = (

        set(
            COLUMNAS_RNDC_FINAL
        )

        |

        {
            "PERIODO"
        }
    )


    faltantes = sorted(
        requeridas
        -
        set(
            df.columns
        )
    )


    if faltantes:

        raise ValueError(
            f"{periodo}: faltan columnas "
            f"en el Parquet final: {faltantes}"
        )


    if df.empty:

        raise ValueError(
            f"{periodo}: no existen "
            "registros con destino Pasto."
        )


    periodos = (

        df[
            "PERIODO"
        ]
        .astype(str)
        .unique()
        .tolist()
    )


    if periodos != [
        str(
            periodo
        )
    ]:

        raise ValueError(
            f"{periodo}: PERIODO inconsistente: "
            f"{periodos}"
        )


    # --------------------------------------------------------
    # DESTINO
    # --------------------------------------------------------

    cod_destino = (
        pd.to_numeric(
            df[
                "CODMUNICIPIODESTINO_NUM"
            ],
            errors="coerce"
        )
    )


    texto_destino = (

        df[
            "MUNICIPIODESTINO_NORM"
        ]
        .astype(str)
    )


    dep_destino = (

        df[
            "DEPARTAMENTODESTINO_NORM"
        ]
        .astype(str)
    )


    valido_pasto = (

        cod_destino.eq(
            CODIGO_PASTO
        )

        |

        (
            texto_destino
            .str.contains(
                r"\bpasto\b",
                regex=True,
                na=False
            )

            &

            dep_destino
            .str.contains(
                r"\bnarino\b",
                regex=True,
                na=False
            )
        )
    )


    if not valido_pasto.all():

        raise ValueError(
            f"{periodo}: existen registros "
            "que no corresponden a Pasto."
        )


    # --------------------------------------------------------
    # CONFIGURACIÓN VEHICULAR
    # --------------------------------------------------------

    if (
        df[
            "COD_CONFIG_VEHICULO"
        ]
        .notna()
        .sum()
        ==
        0
    ):

        raise ValueError(
            f"{periodo}: configuración "
            "vehicular completamente vacía."
        )


    # --------------------------------------------------------
    # MERCANCÍA
    # --------------------------------------------------------

    if (
        df[
            "CODMERCANCIA"
        ]
        .notna()
        .sum()
        ==
        0
    ):

        raise ValueError(
            f"{periodo}: mercancía "
            "completamente vacía."
        )


    return True


# ------------------------------------------------------------
# 6. COMPROBAR SI UN MES YA ESTÁ TERMINADO
# ------------------------------------------------------------

def mes_final_completo(
    periodo
):

    ruta = (
        DIR_CACHE_FINAL
        /
        f"RNDC_PASTO_FINAL_{periodo}.parquet"
    )


    if not ruta.exists():

        return False


    if manifest_final.empty:

        return False


    fila = manifest_final.loc[
        (
            manifest_final[
                "periodo"
            ]
            .astype(str)
            ==
            str(
                periodo
            )
        )

        &

        (
            manifest_final[
                "estado"
            ]
            ==
            "OK"
        )
    ]


    if fila.empty:

        return False


    try:

        control = pd.read_parquet(
            ruta
        )


        validar_parquet_final(
            control,
            periodo
        )


        del control

        return True


    except Exception:

        return False


# ------------------------------------------------------------
# 7. DETERMINAR PENDIENTES
# ------------------------------------------------------------

periodos_completos_final = [

    p

    for p
    in PERIODOS_RNDC_FINAL

    if mes_final_completo(
        p
    )
]


periodos_pendientes_final = [

    p

    for p
    in PERIODOS_RNDC_FINAL

    if p
    not in periodos_completos_final
]


print(
    "=" * 100
)

print(
    "CELDA 8B — RNDC FINAL 2020–2024"
)

print(
    "=" * 100
)


print(
    "Meses requeridos:",
    len(
        PERIODOS_RNDC_FINAL
    )
)


print(
    "Meses finales ya completos:",
    len(
        periodos_completos_final
    )
)


print(
    "Meses pendientes:",
    len(
        periodos_pendientes_final
    )
)


print(
    "Trabajadores paralelos:",
    MAX_WORKERS_FINAL
)


# ------------------------------------------------------------
# 8. PROCESAR UN MES
# ------------------------------------------------------------

def procesar_mes_final(
    periodo
):

    ultimo_error = None


    for intento in range(
        1,
        4
    ):

        inicio = time.time()


        ruta_xlsx = None

        ruta_tmp = (
            DIR_CACHE_FINAL
            /
            f"RNDC_PASTO_FINAL_{periodo}.tmp.parquet"
        )

        ruta_final = (
            DIR_CACHE_FINAL
            /
            f"RNDC_PASTO_FINAL_{periodo}.parquet"
        )


        try:

            # ------------------------------------------------
            # DESCARGA OFICIAL
            # ------------------------------------------------

            ruta_xlsx = descargar_rndc_actual(
                periodo,
                sobrescribir=True
            )


            hash_original = (
                sha256_archivo_final(
                    ruta_xlsx
                )
            )


            # ------------------------------------------------
            # LECTURA 23 VARIABLES
            # ------------------------------------------------

            df_nacional = (
                leer_rndc_final(
                    ruta_xlsx
                )
            )


            n_nacional = len(
                df_nacional
            )


            # ------------------------------------------------
            # DESTINO PASTO
            # ------------------------------------------------

            df_pasto = (
                finalizar_base_pasto_final(
                    df_nacional,
                    periodo
                )
            )


            n_pasto = len(
                df_pasto
            )


            validar_parquet_final(
                df_pasto,
                periodo
            )


            # ------------------------------------------------
            # ESTADÍSTICAS ESTRUCTURALES
            # ------------------------------------------------

            n_origenes = (

                df_pasto[
                    "CODMUNICIPIOORIGEN"
                ]
                .nunique(
                    dropna=True
                )
            )


            n_configuraciones = (

                df_pasto[
                    "COD_CONFIG_VEHICULO"
                ]
                .nunique(
                    dropna=True
                )
            )


            n_mercancias = (

                df_pasto[
                    "CODMERCANCIA"
                ]
                .nunique(
                    dropna=True
                )
            )


            n_celdas = (

                df_pasto[
                    [
                        "CODMUNICIPIOORIGEN",
                        "COD_CONFIG_VEHICULO",
                        "CODMERCANCIA"
                    ]
                ]

                .drop_duplicates()

                .shape[0]
            )


            # ------------------------------------------------
            # COBERTURA MONETARIA
            # ------------------------------------------------

            viajes_con_valor = (

                pd.to_numeric(
                    df_pasto[
                        "VIAJESTOTALES"
                    ],
                    errors="coerce"
                )

                -

                pd.to_numeric(
                    df_pasto[
                        "VIAJESVALORCERO"
                    ],
                    errors="coerce"
                )
            )


            monetario_valido = (

                viajes_con_valor.gt(
                    0
                )

                &

                pd.to_numeric(
                    df_pasto[
                        "VALORESPAGADOS"
                    ],
                    errors="coerce"
                ).gt(
                    0
                )
            )


            n_monetario = int(
                monetario_valido.sum()
            )


            pct_monetario = round(

                100
                *
                monetario_valido.mean(),

                4
            )


            # ------------------------------------------------
            # GUARDAR PARQUET TEMPORAL
            # ------------------------------------------------

            if ruta_tmp.exists():

                ruta_tmp.unlink()


            df_pasto.to_parquet(

                ruta_tmp,

                index=False,

                compression="snappy"
            )


            # ------------------------------------------------
            # VALIDACIÓN POST-ESCRITURA
            # ------------------------------------------------

            control = pd.read_parquet(
                ruta_tmp
            )


            validar_parquet_final(
                control,
                periodo
            )


            if len(
                control
            ) != n_pasto:

                raise ValueError(
                    f"{periodo}: cambió el "
                    "número de filas al guardar."
                )


            del control


            # ------------------------------------------------
            # MOVER A FINAL
            # ------------------------------------------------

            os.replace(
                ruta_tmp,
                ruta_final
            )


            hash_parquet = (
                sha256_archivo_final(
                    ruta_final
                )
            )


            segundos = round(
                time.time()
                -
                inicio,
                2
            )


            fila_manifest = {

                "periodo":
                    periodo,

                "fuente":
                    URL_RNDC_ACTUAL,

                "archivo_original":
                    ruta_xlsx.name,

                "sha256_original":
                    hash_original,

                "filas_nacionales":
                    int(
                        n_nacional
                    ),

                "filas_pasto":
                    int(
                        n_pasto
                    ),

                "n_origenes":
                    int(
                        n_origenes
                    ),

                "n_configuraciones":
                    int(
                        n_configuraciones
                    ),

                "n_mercancias":
                    int(
                        n_mercancias
                    ),

                "n_celdas":
                    int(
                        n_celdas
                    ),

                "registros_valor_positivo":
                    n_monetario,

                "pct_registros_valor_positivo":
                    pct_monetario,

                "parquet":
                    ruta_final.name,

                "sha256_parquet":
                    hash_parquet,

                "tamano_parquet_mb":
                    round(
                        ruta_final.stat().st_size
                        /
                        1024**2,
                        6
                    ),

                "segundos_total":
                    segundos,

                "estado":
                    "OK"
            }


            # ------------------------------------------------
            # LIMPIAR RAM
            # ------------------------------------------------

            del df_nacional
            del df_pasto


            # ------------------------------------------------
            # BORRAR EXCEL NACIONAL
            # ------------------------------------------------

            if (
                ruta_xlsx is not None
                and
                ruta_xlsx.exists()
            ):

                ruta_xlsx.unlink()


            gc.collect()


            return {

                "ok":
                    True,

                "periodo":
                    periodo,

                "fila":
                    fila_manifest,

                "intento":
                    intento,

                "error":
                    None
            }


        except Exception as e:

            ultimo_error = (
                f"{type(e).__name__}: "
                f"{e}"
            )


            # ------------------------------------------------
            # LIMPIEZA DE INTENTO FALLIDO
            # ------------------------------------------------

            if (
                ruta_xlsx is not None
                and
                ruta_xlsx.exists()
            ):

                try:
                    ruta_xlsx.unlink()
                except Exception:
                    pass


            if ruta_tmp.exists():

                try:
                    ruta_tmp.unlink()
                except Exception:
                    pass


            gc.collect()


    return {

        "ok":
            False,

        "periodo":
            periodo,

        "fila":
            None,

        "intento":
            3,

        "error":
            ultimo_error
    }


# ------------------------------------------------------------
# 9. EJECUCIÓN PARALELA
# ------------------------------------------------------------

resultados_finales = []


if periodos_pendientes_final:

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS_FINAL
    ) as executor:

        futuros = {

            executor.submit(
                procesar_mes_final,
                periodo
            ):
            periodo

            for periodo
            in periodos_pendientes_final
        }


        total = len(
            periodos_pendientes_final
        )


        terminados = 0


        for futuro in as_completed(
            futuros
        ):

            resultado = futuro.result()


            resultados_finales.append(
                resultado
            )


            terminados += 1


            periodo = resultado[
                "periodo"
            ]


            if resultado[
                "ok"
            ]:

                fila = resultado[
                    "fila"
                ]


                # --------------------------------------------
                # ACTUALIZAR MANIFEST SOLO EN HILO PRINCIPAL
                # --------------------------------------------

                manifest_final = (

                    manifest_final.loc[
                        manifest_final[
                            "periodo"
                        ]
                        .astype(str)
                        !=
                        periodo
                    ]

                    .copy()
                )


                manifest_final = pd.concat(

                    [
                        manifest_final,

                        pd.DataFrame(
                            [
                                fila
                            ]
                        )
                    ],

                    ignore_index=True
                )


                manifest_final[
                    "periodo"
                ] = (

                    manifest_final[
                        "periodo"
                    ]
                    .astype(str)
                    .str.zfill(
                        6
                    )
                )


                manifest_final = (

                    manifest_final

                    .sort_values(
                        "periodo"
                    )

                    .reset_index(
                        drop=True
                    )
                )


                manifest_final.to_csv(

                    RUTA_MANIFEST_FINAL,

                    index=False,

                    encoding="utf-8-sig"
                )


                print(

                    f"[{terminados:02d}/{total:02d}] "
                    f"{periodo} → OK | "
                    f"Pasto={fila['filas_pasto']:,} | "
                    f"Config={fila['n_configuraciones']} | "
                    f"Merc={fila['n_mercancias']} | "
                    f"Monetario={fila['pct_registros_valor_positivo']:.1f}%"
                )


            else:

                print(

                    f"[{terminados:02d}/{total:02d}] "
                    f"{periodo} → ERROR | "
                    f"{resultado['error']}"
                )


            gc.collect()


else:

    print(
        "\nLos 60 meses finales "
        "ya estaban completos."
    )


# ------------------------------------------------------------
# 10. RECARGAR MANIFEST
# ------------------------------------------------------------

if RUTA_MANIFEST_FINAL.exists():

    manifest_final = pd.read_csv(

        RUTA_MANIFEST_FINAL,

        dtype={
            "periodo": str
        }
    )


else:

    manifest_final = pd.DataFrame(
        columns=COLUMNAS_MANIFEST_FINAL
    )


# ------------------------------------------------------------
# 11. AUDITORÍA FÍSICA
# ------------------------------------------------------------

periodos_manifest_ok = sorted(

    manifest_final.loc[
        manifest_final[
            "estado"
        ]
        ==
        "OK",
        "periodo"
    ]

    .astype(str)

    .tolist()
)


faltantes_manifest_final = sorted(

    set(
        PERIODOS_RNDC_FINAL
    )

    -

    set(
        periodos_manifest_ok
    )
)


faltantes_fisicos_final = []


for periodo in PERIODOS_RNDC_FINAL:

    ruta = (

        DIR_CACHE_FINAL
        /
        f"RNDC_PASTO_FINAL_{periodo}.parquet"
    )


    if not ruta.exists():

        faltantes_fisicos_final.append(
            periodo
        )


# ------------------------------------------------------------
# 12. RESUMEN ANUAL
# ------------------------------------------------------------

manifest_ok_final = (

    manifest_final.loc[
        manifest_final[
            "estado"
        ]
        ==
        "OK"
    ]

    .copy()
)


if not manifest_ok_final.empty:

    manifest_ok_final[
        "anio"
    ] = (

        manifest_ok_final[
            "periodo"
        ]

        .astype(str)

        .str[:4]

        .astype(int)
    )


    resumen_anual_final = (

        manifest_ok_final

        .groupby(
            "anio",
            as_index=False
        )

        .agg(

            meses=(
                "periodo",
                "count"
            ),

            filas_pasto=(
                "filas_pasto",
                "sum"
            ),

            origenes_promedio=(
                "n_origenes",
                "mean"
            ),

            configuraciones_promedio=(
                "n_configuraciones",
                "mean"
            ),

            mercancias_promedio=(
                "n_mercancias",
                "mean"
            ),

            cobertura_monetaria_media=(
                "pct_registros_valor_positivo",
                "mean"
            )
        )
    )


    print(
        "\n"
        + "=" * 100
    )

    print(
        "RESUMEN ANUAL — BASE FINAL"
    )

    print(
        "=" * 100
    )


    display(
        resumen_anual_final
    )


# ------------------------------------------------------------
# 13. MESES CON COBERTURA ATÍPICA
# ------------------------------------------------------------

if not manifest_ok_final.empty:

    q1 = (
        manifest_ok_final[
            "filas_pasto"
        ]
        .quantile(
            0.25
        )
    )


    q3 = (
        manifest_ok_final[
            "filas_pasto"
        ]
        .quantile(
            0.75
        )
    )


    iqr = (
        q3
        -
        q1
    )


    limite_inferior = max(
        0,
        q1
        -
        1.5
        *
        iqr
    )


    meses_baja_cobertura = (

        manifest_ok_final.loc[
            manifest_ok_final[
                "filas_pasto"
            ]
            <
            limite_inferior,
            [
                "periodo",
                "filas_pasto",
                "n_origenes",
                "n_configuraciones",
                "n_mercancias",
                "pct_registros_valor_positivo"
            ]
        ]

        .sort_values(
            "periodo"
        )
    )


    print(
        "\nMeses con número de filas "
        "inusualmente bajo:"
    )


    display(
        meses_baja_cobertura
    )


# ------------------------------------------------------------
# 14. RESULTADO FINAL
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 100
)

print(
    "AUDITORÍA FINAL — CELDA 8B"
)

print(
    "=" * 100
)


print(
    "Meses requeridos:",
    len(
        PERIODOS_RNDC_FINAL
    )
)


print(
    "Meses OK:",
    len(
        periodos_manifest_ok
    )
)


print(
    "Faltantes manifest:",
    faltantes_manifest_final
)


print(
    "Faltantes físicos:",
    faltantes_fisicos_final
)


if (

    len(
        periodos_manifest_ok
    )
    ==
    60

    and

    len(
        faltantes_manifest_final
    )
    ==
    0

    and

    len(
        faltantes_fisicos_final
    )
    ==
    0

):

    print(
        "\n"
        + "=" * 100
    )

    print(
        "PASS — CELDA 8B"
    )

    print(
        "60/60 meses RNDC finales "
        "procesados correctamente."
    )

    print(
        "Contrato de 23 variables "
        "preservado."
    )

    print(
        "Configuración vehicular "
        "disponible para el índice."
    )

    print(
        "Base lista para auditoría "
        "monetaria y construcción "
        "del índice regional."
    )

    print(
        "=" * 100
    )


else:

    print(
        "\n"
        + "=" * 100
    )

    print(
        "REVISIÓN REQUERIDA — CELDA 8B"
    )

    print(
        "Puedes ejecutar nuevamente "
        "esta misma celda."
    )

    print(
        "Solo intentará los periodos "
        "que no estén completos."
    )

    print(
        "=" * 100
    )

CELDA 8B — RNDC FINAL 2020–2024
Meses requeridos: 60
Meses finales ya completos: 0
Meses pendientes: 60
Trabajadores paralelos: 3
202001202002 → solicitando RNDC...
 → solicitando RNDC...
202003 → solicitando RNDC...
202001 → descargado: 15.54 MB | 1.1 s
202003 → descargado: 14.66 MB | 1.2 s
202002 → descargado: 15.82 MB | 1.2 s
[01/60] 202003 → OK | Pasto=1,210 | Config=10 | Merc=355 | Monetario=94.0%
202004 → solicitando RNDC...
[02/60] 202001 → OK | Pasto=1,156 | Config=11 | Merc=348 | Monetario=93.0%
[03/60] 202002 → OK | Pasto=1,260 | Config=10 | Merc=377 | Monetario=93.6%
202004 → descargado: 10.52 MB | 1.3 s
202005 → solicitando RNDC...
202006 → solicitando RNDC...
202005 → descargado: 13.10 MB | 1.0 s
202006 → descargado: 14.12 MB | 1.0 s
[04/60] 202004 → OK | Pasto=723 | Config=7 | Merc=251 | Monetario=93.6%
202007 → solicitando RNDC...
202007 → descargado: 15.74 MB | 1.2 s
[05/60] 202005 → OK | Pasto=1,071 | Config=12 | Merc=339 | Monetario=93.9%
202008 → solicitando RNDC...


,anio,meses,filas_pasto,origenes_promedio,configuraciones_promedio,mercancias_promedio,cobertura_monetaria_media
0,2020,12,14687,147.250000,11.083333,359.500000,93.070642
1,2021,12,15978,157.750000,12.916667,363.833333,92.204825
2,2022,12,16352,163.833333,11.083333,387.166667,92.406742
3,2023,12,15558,159.416667,9.583333,361.333333,93.927583
4,2024,12,16244,162.000000,9.666667,361.583333,92.606708



Meses con número de filas inusualmente bajo:


,periodo,filas_pasto,n_origenes,n_configuraciones,n_mercancias,pct_registros_valor_positivo
3,202004,723,109,7,251,93.6376
16,202105,180,61,8,88,86.6667
36,202301,929,131,10,282,94.5102
37,202302,1012,122,9,320,96.8379



AUDITORÍA FINAL — CELDA 8B
Meses requeridos: 60
Meses OK: 60
Faltantes manifest: []
Faltantes físicos: []

PASS — CELDA 8B
60/60 meses RNDC finales procesados correctamente.
Contrato de 23 variables preservado.
Configuración vehicular disponible para el índice.
Base lista para auditoría monetaria y construcción del índice regional.


In [22]:
# ============================================================
# CELDA 8C — CONSOLIDACIÓN RÁPIDA DE LA BASE RNDC FINAL
# ============================================================
#
# OBJETIVO:
# Consolidar los 60 Parquet ya validados de 2020–2024
# en UNA base maestra compacta.
#
# IMPORTANTE:
# - NO descarga RNDC.
# - NO lee Excel.
# - NO vuelve a ejecutar la Celda 8B.
# - NO elimina observaciones.
# - NO imputa.
# - NO winsoriza.
# - Conserva TODAS las columnas del cache final.
#
# Esta base será la fuente de trabajo de aquí en adelante.
# ============================================================


import pandas as pd
import numpy as np
import hashlib
import json
import gc
from pathlib import Path
from datetime import datetime


# ------------------------------------------------------------
# 1. RUTAS
# ------------------------------------------------------------

DIR_CACHE_FINAL = (
    ROOT
    /
    "cache_final_23c"
)

DIR_AUDIT_FINAL = (
    ROOT
    /
    "audit_final_23c"
)

RUTA_MANIFEST_FINAL = (
    DIR_AUDIT_FINAL
    /
    "manifest_rndc_final_2020_2024.csv"
)

RUTA_MASTER = (
    ROOT
    /
    "RNDC_PASTO_MASTER_2020_2024.parquet"
)

RUTA_METADATA = (
    ROOT
    /
    "RNDC_PASTO_MASTER_2020_2024_metadata.json"
)


# ------------------------------------------------------------
# 2. PERIODOS ESPERADOS
# ------------------------------------------------------------

PERIODOS_ESPERADOS = [

    f"{anio}{mes:02d}"

    for anio in range(2020, 2025)

    for mes in range(1, 13)
]


assert len(PERIODOS_ESPERADOS) == 60


# ------------------------------------------------------------
# 3. LOCALIZAR LOS 60 PARQUET
# ------------------------------------------------------------

archivos = sorted(
    DIR_CACHE_FINAL.glob(
        "RNDC_PASTO_FINAL_*.parquet"
    )
)


periodos_archivos = [

    p.stem.split("_")[-1]

    for p in archivos
]


print("=" * 100)
print("CELDA 8C — CONSOLIDACIÓN RNDC FINAL")
print("=" * 100)

print(
    "Parquet encontrados:",
    len(archivos)
)


assert len(archivos) == 60, (
    f"Se esperaban 60 archivos y "
    f"se encontraron {len(archivos)}."
)


assert (
    set(periodos_archivos)
    ==
    set(PERIODOS_ESPERADOS)
), (
    "Los periodos físicos no coinciden "
    "con 202001–202412."
)


# ------------------------------------------------------------
# 4. MANIFEST ORIGINAL
# ------------------------------------------------------------

assert RUTA_MANIFEST_FINAL.exists(), (
    "No se encontró el manifest final "
    "generado por la Celda 8B."
)


manifest = pd.read_csv(
    RUTA_MANIFEST_FINAL,
    dtype={
        "periodo": str
    }
)


manifest = (
    manifest
    .loc[
        manifest["estado"].eq("OK")
    ]
    .copy()
)


assert len(manifest) == 60


filas_esperadas = int(
    manifest[
        "filas_pasto"
    ].sum()
)


print(
    "Filas esperadas según manifest:",
    f"{filas_esperadas:,}"
)


# Resultado observado de 8B:
# 78,819 registros hacia Pasto.
assert filas_esperadas == 78819, (
    "El total del manifest cambió "
    "respecto de la ejecución validada."
)


# ------------------------------------------------------------
# 5. LEER LOS 60 PARQUET
# ------------------------------------------------------------

partes = []

esquema_referencia = None


for i, ruta in enumerate(
    archivos,
    start=1
):

    periodo = (
        ruta.stem
        .split("_")[-1]
    )


    df_mes = pd.read_parquet(
        ruta
    )


    # --------------------------------------------------------
    # Validación básica
    # --------------------------------------------------------

    assert not df_mes.empty, (
        f"{periodo}: Parquet vacío."
    )


    assert (
        df_mes["PERIODO"]
        .astype(str)
        .eq(periodo)
        .all()
    ), (
        f"{periodo}: PERIODO inconsistente."
    )


    # --------------------------------------------------------
    # Validar esquema idéntico
    # --------------------------------------------------------

    esquema_actual = tuple(
        df_mes.columns
    )


    if esquema_referencia is None:

        esquema_referencia = (
            esquema_actual
        )

    else:

        assert (
            esquema_actual
            ==
            esquema_referencia
        ), (
            f"{periodo}: esquema distinto "
            "al resto de meses."
        )


    # --------------------------------------------------------
    # Comparar filas con manifest
    # --------------------------------------------------------

    esperado_mes = int(

        manifest.loc[
            manifest[
                "periodo"
            ].eq(periodo),
            "filas_pasto"
        ].iloc[0]
    )


    assert (
        len(df_mes)
        ==
        esperado_mes
    ), (
        f"{periodo}: filas físicas "
        f"{len(df_mes):,} != "
        f"manifest {esperado_mes:,}"
    )


    partes.append(
        df_mes
    )


print(
    "60/60 archivos leídos "
    "y validados."
)


# ------------------------------------------------------------
# 6. CONSOLIDAR
# ------------------------------------------------------------

rndc_master = pd.concat(
    partes,
    ignore_index=True,
    copy=False
)


del partes

gc.collect()


print(
    "\nFilas consolidadas:",
    f"{len(rndc_master):,}"
)

print(
    "Columnas conservadas:",
    len(rndc_master.columns)
)


assert (
    len(rndc_master)
    ==
    filas_esperadas
)


# ------------------------------------------------------------
# 7. VALIDACIÓN TEMPORAL
# ------------------------------------------------------------

periodos_master = sorted(

    rndc_master[
        "PERIODO"
    ]
    .astype(str)
    .unique()
    .tolist()
)


assert (
    periodos_master
    ==
    PERIODOS_ESPERADOS
)


conteo_master = (

    rndc_master
    .groupby(
        "PERIODO"
    )
    .size()
    .rename(
        "FILAS_MASTER"
    )
    .reset_index()
)


conteo_control = (

    manifest[
        [
            "periodo",
            "filas_pasto"
        ]
    ]
    .rename(
        columns={
            "periodo":
                "PERIODO",

            "filas_pasto":
                "FILAS_MANIFEST"
        }
    )
)


control = conteo_master.merge(
    conteo_control,
    on="PERIODO",
    how="outer"
)


control[
    "OK"
] = (

    control[
        "FILAS_MASTER"
    ]
    ==
    control[
        "FILAS_MANIFEST"
    ]
)


assert control[
    "OK"
].all()


# ------------------------------------------------------------
# 8. NO ELIMINAR DUPLICADOS
#
# Los duplicados exactos se cuentan únicamente
# como auditoría. No se eliminan automáticamente.
# ------------------------------------------------------------

n_duplicados_exactos = int(
    rndc_master
    .duplicated()
    .sum()
)


print(
    "\nDuplicados exactos detectados:",
    f"{n_duplicados_exactos:,}"
)

print(
    "No se eliminaron."
)


# ------------------------------------------------------------
# 9. ORDEN DETERMINISTA
# ------------------------------------------------------------

columnas_orden = [

    c

    for c in [

        "PERIODO",
        "CODMUNICIPIOORIGEN",
        "COD_CONFIG_VEHICULO",
        "CODMERCANCIA"

    ]

    if c in rndc_master.columns
]


rndc_master = (

    rndc_master

    .sort_values(
        columnas_orden,
        kind="stable"
    )

    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 10. GUARDAR MASTER
#
# ZSTD ofrece muy buena compresión
# sin perder absolutamente ningún dato.
# ------------------------------------------------------------

rndc_master.to_parquet(
    RUTA_MASTER,
    index=False,
    compression="zstd"
)


# ------------------------------------------------------------
# 11. HASH SHA-256
# ------------------------------------------------------------

def sha256_archivo(ruta):

    h = hashlib.sha256()

    with open(
        ruta,
        "rb"
    ) as f:

        for bloque in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b""
        ):

            h.update(
                bloque
            )


    return h.hexdigest()


sha_master = sha256_archivo(
    RUTA_MASTER
)


tamano_master_mb = (

    RUTA_MASTER.stat().st_size
    /
    1024**2
)


print(
    "\nArchivo MASTER:",
    RUTA_MASTER
)

print(
    "Tamaño:",
    round(
        tamano_master_mb,
        3
    ),
    "MB"
)

print(
    "SHA-256:"
)

print(
    sha_master
)


# ------------------------------------------------------------
# 12. VALIDACIÓN DESPUÉS DE ESCRIBIR
# ------------------------------------------------------------

control_master = pd.read_parquet(
    RUTA_MASTER
)


assert (
    len(control_master)
    ==
    len(rndc_master)
)


assert (
    list(control_master.columns)
    ==
    list(rndc_master.columns)
)


assert (

    sorted(
        control_master[
            "PERIODO"
        ]
        .astype(str)
        .unique()
        .tolist()
    )

    ==
    PERIODOS_ESPERADOS
)


# ------------------------------------------------------------
# 13. METADATA REPRODUCIBLE
# ------------------------------------------------------------

metadata = {

    "archivo":
        RUTA_MASTER.name,

    "periodo_inicial":
        "202001",

    "periodo_final":
        "202412",

    "n_meses":
        60,

    "n_filas":
        int(
            len(
                rndc_master
            )
        ),

    "n_columnas":
        int(
            len(
                rndc_master.columns
            )
        ),

    "sha256":
        sha_master,

    "fuente_original":
        URL_RNDC_ACTUAL,

    "destino":
        "Pasto, Nariño",

    "codigo_dane_destino":
        int(
            CODIGO_PASTO
        ),

    "transformacion":
        (
            "Filtro destino Pasto sobre "
            "archivos oficiales RNDC mensuales; "
            "sin imputación, sin winsorización "
            "y sin eliminación de observaciones."
        ),

    "duplicados_exactos_detectados":
        n_duplicados_exactos,

    "fecha_consolidacion_utc":
        datetime.utcnow()
        .isoformat(
            timespec="seconds"
        )
        +
        "Z"
}


with open(
    RUTA_METADATA,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 14. MESES DE COBERTURA BAJA
# ------------------------------------------------------------

q1 = manifest[
    "filas_pasto"
].quantile(
    0.25
)

q3 = manifest[
    "filas_pasto"
].quantile(
    0.75
)

iqr = q3 - q1

limite_bajo = max(
    0,
    q1
    -
    1.5
    *
    iqr
)


alertas_cobertura = (

    manifest.loc[
        manifest[
            "filas_pasto"
        ]
        <
        limite_bajo,
        [
            "periodo",
            "filas_pasto",
            "n_origenes",
            "n_configuraciones",
            "n_mercancias",
            "pct_registros_valor_positivo"
        ]
    ]

    .sort_values(
        "periodo"
    )

    .reset_index(
        drop=True
    )
)


print(
    "\nMeses marcados para auditoría:"
)

display(
    alertas_cobertura
)


# ------------------------------------------------------------
# 15. RESULTADO
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 100
)

print(
    "PASS — CELDA 8C"
)

print(
    "60 meses consolidados en una sola "
    "base maestra compacta."
)

print(
    f"Filas preservadas: "
    f"{len(rndc_master):,}"
)

print(
    "Ninguna observación fue imputada, "
    "winsorizada o eliminada."
)

print(
    "Desde este punto NO es necesario "
    "volver a ejecutar la Celda 8B."
)

print(
    "Toda la construcción del índice "
    "trabajará sobre RUTA_MASTER."
)

print(
    "=" * 100
)

CELDA 8C — CONSOLIDACIÓN RNDC FINAL
Parquet encontrados: 60
Filas esperadas según manifest: 78,819
60/60 archivos leídos y validados.

Filas consolidadas: 78,819
Columnas conservadas: 34

Duplicados exactos detectados: 1
No se eliminaron.

Archivo MASTER: /content/RNDC_PASTO/RNDC_PASTO_MASTER_2020_2024.parquet
Tamaño: 0.954 MB
SHA-256:
b6db3d2caef389ade80117f36c2db58dd3d4fe25a27425c796eb8dd1dadb6096

Meses marcados para auditoría:


/tmp/ipykernel_15690/47344296.py:638: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow()


,periodo,filas_pasto,n_origenes,n_configuraciones,n_mercancias,pct_registros_valor_positivo
0,202004,723,109,7,251,93.6376
1,202105,180,61,8,88,86.6667
2,202301,929,131,10,282,94.5102
3,202302,1012,122,9,320,96.8379



PASS — CELDA 8C
60 meses consolidados en una sola base maestra compacta.
Filas preservadas: 78,819
Ninguna observación fue imputada, winsorizada o eliminada.
Desde este punto NO es necesario volver a ejecutar la Celda 8B.
Toda la construcción del índice trabajará sobre RUTA_MASTER.


In [23]:
# ============================================================
# CELDA 8D — AUDITORÍA DE MESES ATÍPICOS Y DUPLICADO
# ============================================================
#
# OBJETIVO:
# Revisar 202004, 202105, 202301 y 202302
# antes de construir el índice regional.
#
# NO elimina observaciones.
# NO imputa.
# NO winsoriza.
# NO descarga nada.
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. CARGAR MASTER
# ------------------------------------------------------------

rndc = pd.read_parquet(
    RUTA_MASTER
)

rndc["PERIODO"] = (
    rndc["PERIODO"]
    .astype(str)
    .str.zfill(6)
)


print("=" * 100)
print("CELDA 8D — AUDITORÍA RNDC")
print("=" * 100)

print(
    "Filas:",
    f"{len(rndc):,}"
)

print(
    "Meses:",
    rndc["PERIODO"].nunique()
)


# ------------------------------------------------------------
# 2. VARIABLES NUMÉRICAS
# ------------------------------------------------------------

for c in [
    "VIAJESTOTALES",
    "VIAJESVALORCERO",
    "KILOGRAMOS",
    "KILOMETROS",
    "VALORESPAGADOS"
]:

    rndc[c] = pd.to_numeric(
        rndc[c],
        errors="coerce"
    )


rndc["VIAJES_CON_VALOR"] = (
    rndc["VIAJESTOTALES"]
    -
    rndc["VIAJESVALORCERO"]
)


# ------------------------------------------------------------
# 3. RESUMEN MENSUAL COMPLETO
# ------------------------------------------------------------

resumen_mensual = (

    rndc

    .groupby(
        "PERIODO",
        as_index=False
    )

    .agg(

        FILAS=(
            "PERIODO",
            "size"
        ),

        ORIGENES=(
            "CODMUNICIPIOORIGEN",
            "nunique"
        ),

        CONFIGURACIONES=(
            "COD_CONFIG_VEHICULO",
            "nunique"
        ),

        MERCANCIAS=(
            "CODMERCANCIA",
            "nunique"
        ),

        VIAJES=(
            "VIAJESTOTALES",
            "sum"
        ),

        VIAJES_CON_VALOR=(
            "VIAJES_CON_VALOR",
            "sum"
        ),

        KILOGRAMOS=(
            "KILOGRAMOS",
            "sum"
        ),

        VALOR_PAGADO=(
            "VALORESPAGADOS",
            "sum"
        )
    )
)


resumen_mensual[
    "PCT_VIAJES_CON_VALOR"
] = (

    100
    *
    resumen_mensual[
        "VIAJES_CON_VALOR"
    ]
    /
    resumen_mensual[
        "VIAJES"
    ]
)


# ------------------------------------------------------------
# 4. MEDIANA MÓVIL LOCAL
#
# Comparamos cada mes con los dos anteriores
# y los dos posteriores.
#
# SOLO DIAGNÓSTICO.
# No se usa para modificar datos.
# ------------------------------------------------------------

variables_control = [
    "FILAS",
    "ORIGENES",
    "MERCANCIAS",
    "VIAJES",
    "KILOGRAMOS",
    "VALOR_PAGADO"
]


for c in variables_control:

    resumen_mensual[
        f"MEDIANA_LOCAL_{c}"
    ] = (

        resumen_mensual[c]

        .rolling(
            window=5,
            center=True,
            min_periods=3
        )

        .median()
    )


    resumen_mensual[
        f"RATIO_LOCAL_{c}"
    ] = (

        resumen_mensual[c]
        /
        resumen_mensual[
            f"MEDIANA_LOCAL_{c}"
        ]
    )


# ------------------------------------------------------------
# 5. MESES OBJETIVO
# ------------------------------------------------------------

MESES_AUDITAR = [
    "202004",
    "202105",
    "202301",
    "202302"
]


columnas_salida = [

    "PERIODO",

    "FILAS",
    "RATIO_LOCAL_FILAS",

    "ORIGENES",
    "RATIO_LOCAL_ORIGENES",

    "CONFIGURACIONES",

    "MERCANCIAS",
    "RATIO_LOCAL_MERCANCIAS",

    "VIAJES",
    "RATIO_LOCAL_VIAJES",

    "KILOGRAMOS",
    "RATIO_LOCAL_KILOGRAMOS",

    "VALOR_PAGADO",
    "RATIO_LOCAL_VALOR_PAGADO",

    "PCT_VIAJES_CON_VALOR"
]


audit_meses = (

    resumen_mensual.loc[
        resumen_mensual[
            "PERIODO"
        ].isin(
            MESES_AUDITAR
        ),
        columnas_salida
    ]

    .copy()
)


print(
    "\nMESES ATÍPICOS — COMPARACIÓN LOCAL"
)

display(
    audit_meses
)


# ------------------------------------------------------------
# 6. VENTANA ALREDEDOR DE CADA MES
# ------------------------------------------------------------

for periodo in MESES_AUDITAR:

    idx = (

        resumen_mensual.index[
            resumen_mensual[
                "PERIODO"
            ].eq(periodo)
        ]
    )


    if len(idx) == 0:
        continue


    i = int(
        idx[0]
    )


    inicio = max(
        0,
        i - 2
    )


    fin = min(
        len(resumen_mensual),
        i + 3
    )


    print(
        "\n"
        + "=" * 100
    )

    print(
        "VENTANA:",
        periodo
    )

    print(
        "=" * 100
    )


    display(

        resumen_mensual.iloc[
            inicio:fin
        ][
            [
                "PERIODO",
                "FILAS",
                "ORIGENES",
                "CONFIGURACIONES",
                "MERCANCIAS",
                "VIAJES",
                "KILOGRAMOS",
                "VALOR_PAGADO",
                "PCT_VIAJES_CON_VALOR"
            ]
        ]
    )


# ------------------------------------------------------------
# 7. AUDITORÍA DEL DUPLICADO EXACTO
# ------------------------------------------------------------

mask_dup = rndc.duplicated(
    keep=False
)


duplicados = (

    rndc.loc[
        mask_dup
    ]

    .copy()

    .sort_values(
        [
            "PERIODO",
            "CODMUNICIPIOORIGEN",
            "COD_CONFIG_VEHICULO",
            "CODMERCANCIA"
        ]
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "AUDITORÍA DEL DUPLICADO EXACTO"
)

print(
    "=" * 100
)


print(
    "Filas involucradas:",
    len(
        duplicados
    )
)


if not duplicados.empty:

    columnas_dup = [

        "PERIODO",

        "CODMUNICIPIOORIGEN",
        "MUNICIPIOORIGEN",

        "COD_CONFIG_VEHICULO",
        "CONFIG_VEHICULO",

        "CODMERCANCIA",
        "MERCANCIA",

        "VIAJESTOTALES",
        "VIAJESVALORCERO",

        "KILOGRAMOS",
        "KILOMETROS",
        "VALORESPAGADOS"
    ]


    display(
        duplicados[
            columnas_dup
        ]
    )


# ------------------------------------------------------------
# 8. CELDAS HOMOGÉNEAS POR MES
# ------------------------------------------------------------

celdas_mes = (

    rndc

    .groupby(
        "PERIODO"
    )

    .apply(
        lambda x:
            x[
                [
                    "CODMUNICIPIOORIGEN",
                    "COD_CONFIG_VEHICULO",
                    "CODMERCANCIA"
                ]
            ]
            .drop_duplicates()
            .shape[0],

        include_groups=False
    )

    .rename(
        "CELDAS"
    )

    .reset_index()
)


print(
    "\nNúmero de celdas homogéneas:"
)


display(

    celdas_mes.loc[
        celdas_mes[
            "PERIODO"
        ].isin(
            MESES_AUDITAR
        )
    ]
)


# ------------------------------------------------------------
# 9. REGLA DE DECISIÓN
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 100
)

print(
    "FIN — CELDA 8D"
)

print(
    "Esta auditoría NO modifica los datos."
)

print(
    "Los meses atípicos y el duplicado "
    "se mantienen hasta interpretar "
    "la evidencia observada."
)

print(
    "=" * 100
)

CELDA 8D — AUDITORÍA RNDC
Filas: 78,819
Meses: 60

MESES ATÍPICOS — COMPARACIÓN LOCAL


,PERIODO,FILAS,RATIO_LOCAL_FILAS,ORIGENES,RATIO_LOCAL_ORIGENES,CONFIGURACIONES,MERCANCIAS,RATIO_LOCAL_MERCANCIAS,VIAJES,RATIO_LOCAL_VIAJES,KILOGRAMOS,RATIO_LOCAL_KILOGRAMOS,VALOR_PAGADO,RATIO_LOCAL_VALOR_PAGADO,PCT_VIAJES_CON_VALOR
3,202004,723,0.597521,109,0.784173,7,251,0.719198,2819,0.573434,40518050,0.591057,5857736748,0.601771,89.819085
16,202105,180,0.124740,61,0.371951,8,88,0.229765,370,0.060996,3100616,0.039418,659940506,0.054936,85.675676
36,202301,929,0.692250,131,0.834395,10,282,0.740157,4326,0.739993,34305772,0.474968,12035518319,0.906613,94.128525
37,202302,1012,0.794349,122,0.777070,9,320,0.922190,5444,1.000000,37605841,0.587384,13856169928,1.000000,94.544453



VENTANA: 202004


,PERIODO,FILAS,ORIGENES,CONFIGURACIONES,MERCANCIAS,VIAJES,KILOGRAMOS,VALOR_PAGADO,PCT_VIAJES_CON_VALOR
1,202002,1260,151,10,377,5208,70667498,10783029926,90.284178
2,202003,1210,139,10,355,4916,68551892,10347074915,90.296989
3,202004,723,109,7,251,2819,40518050,5857736748,89.819085
4,202005,1071,136,12,339,4160,57476071,8434997146,90.865385
5,202006,1210,158,12,349,5114,68587954,9734166388,90.535784



VENTANA: 202105


,PERIODO,FILAS,ORIGENES,CONFIGURACIONES,MERCANCIAS,VIAJES,KILOGRAMOS,VALOR_PAGADO,PCT_VIAJES_CON_VALOR
14,202103,1443,171,12,386,6090,78659053,12012891257,89.770115
15,202104,1286,158,13,364,5349,73381501,10905537124,89.549448
16,202105,180,61,8,88,370,3100616,659940506,85.675676
17,202106,1483,164,12,400,6066,81843401,16219281737,88.460270
18,202107,1465,177,14,383,6337,89293059,14577649858,90.121509


... (5 outputs left out)

,PERIODO,CODMUNICIPIOORIGEN,MUNICIPIOORIGEN,COD_CONFIG_VEHICULO,CONFIG_VEHICULO,CODMERCANCIA,MERCANCIA,VIAJESTOTALES,VIAJESVALORCERO,KILOGRAMOS,KILOMETROS,VALORESPAGADOS
65771,202403,52001000,PASTO NARINO,2,Camión Rígido de 2 ejes,007321,ESTUFAS; CALDERAS CON HOGAR; COCINAS (INCLUI...,1,0,150,87,387000
65772,202403,52001000,PASTO NARINO,2,Camión Rígido de 2 ejes,007321,ESTUFAS; CALDERAS CON HOGAR; COCINAS (INCLUI...,1,0,150,87,387000



Número de celdas homogéneas:


,PERIODO,CELDAS
3,202004,693
16,202105,170
36,202301,915
37,202302,989



FIN — CELDA 8D
Esta auditoría NO modifica los datos.
Los meses atípicos y el duplicado se mantienen hasta interpretar la evidencia observada.


In [24]:
# ============================================================
# CELDA 8E — PANEL ELEMENTAL RNDC
# ORIGEN × CONFIGURACIÓN × MERCANCÍA
# ============================================================
#
# OBJETIVO:
# 1. Cargar el MASTER compacto validado.
# 2. Construir celdas homogéneas mensuales:
#       origen × configuración × mercancía
# 3. Calcular:
#       VIAJES_CON_VALOR
#       PRECIO_VIAJE
#       LOG_PRECIO
#       PRECIO_VIAJE_KM
# 4. Auditar validez monetaria básica.
# 5. Guardar un panel elemental compacto.
#
# IMPORTANTE:
# - NO descarga nada.
# - NO imputa.
# - NO winsoriza.
# - NO elimina extremos bajos.
# - NO elimina extremos altos.
# - NO aplica todavía la regla jerárquica.
# - NO modifica el MASTER.
# ============================================================

import pandas as pd
import numpy as np
import hashlib
import gc
from pathlib import Path


# ------------------------------------------------------------
# 1. RUTAS
# ------------------------------------------------------------

assert "RUTA_MASTER" in globals(), (
    "Falta RUTA_MASTER. Ejecuta primero la Celda 8C."
)

assert Path(RUTA_MASTER).exists(), (
    f"No existe el MASTER: {RUTA_MASTER}"
)


RUTA_PANEL_B = (
    ROOT
    /
    "RNDC_PASTO_PANEL_ELEMENTAL_2020_2024.parquet"
)


# ------------------------------------------------------------
# 2. CARGAR MASTER
# ------------------------------------------------------------

rndc = pd.read_parquet(
    RUTA_MASTER
)


print("=" * 105)
print("CELDA 8E — PANEL ELEMENTAL RNDC")
print("=" * 105)

print(
    "Filas MASTER:",
    f"{len(rndc):,}"
)

print(
    "Columnas MASTER:",
    len(rndc.columns)
)


assert len(rndc) == 78819

assert rndc["PERIODO"].astype(str).nunique() == 60


# ------------------------------------------------------------
# 3. NORMALIZACIÓN DE CÓDIGOS
#
# Solo estandariza representación.
# NO cambia el significado de los códigos.
# ------------------------------------------------------------

def normalizar_codigo_serie(
    serie,
    ancho=None
):

    s = (
        serie
        .astype("string")
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )

    s = s.mask(
        s.isin([
            "",
            "nan",
            "None",
            "<NA>"
        ]),
        pd.NA
    )


    if ancho is not None:

        mask_num = (
            s
            .fillna("")
            .str.fullmatch(
                r"\d+"
            )
        )

        s.loc[
            mask_num
        ] = (
            s.loc[
                mask_num
            ]
            .str.zfill(
                ancho
            )
        )


    return s


rndc["PERIODO"] = (
    rndc["PERIODO"]
    .astype(str)
    .str.zfill(6)
)


rndc["ORIGEN_N"] = normalizar_codigo_serie(
    rndc["CODMUNICIPIOORIGEN"],
    ancho=5
)


rndc["CONFIG_N"] = normalizar_codigo_serie(
    rndc["COD_CONFIG_VEHICULO"]
)


rndc["MERCANCIA_N"] = normalizar_codigo_serie(
    rndc["CODMERCANCIA"],
    ancho=6
)


# ------------------------------------------------------------
# 4. NUMÉRICAS
# ------------------------------------------------------------

COLUMNAS_NUM = [

    "VIAJESTOTALES",
    "VIAJESVALORCERO",
    "KILOGRAMOS",
    "KILOMETROS",
    "VALORESPAGADOS"
]


for c in COLUMNAS_NUM:

    rndc[c] = pd.to_numeric(
        rndc[c],
        errors="coerce"
    )


# ------------------------------------------------------------
# 5. AUDITORÍA DE LLAVES
# ------------------------------------------------------------

rndc[
    "LLAVE_COMPLETA"
] = (

    rndc["ORIGEN_N"].notna()

    &

    rndc["CONFIG_N"].notna()

    &

    rndc["MERCANCIA_N"].notna()
)


print(
    "\nRegistros con llave completa:",
    f"{rndc['LLAVE_COMPLETA'].sum():,}",
    "/",
    f"{len(rndc):,}",
    "(",
    round(
        100
        *
        rndc["LLAVE_COMPLETA"].mean(),
        3
    ),
    "% )"
)


# ------------------------------------------------------------
# IMPORTANTE:
# Las observaciones sin llave NO se borran del MASTER.
# Simplemente no pueden formar una celda homogénea B.
# ------------------------------------------------------------

x = (
    rndc.loc[
        rndc["LLAVE_COMPLETA"]
    ]
    .copy()
)


# ------------------------------------------------------------
# 6. CONTAR OPERACIONES DENTRO DE LA CELDA
#
# Es solo diagnóstico de composición.
# No forma parte de la llave principal.
# ------------------------------------------------------------

if "CODOPERACIONTRANSPORTE" in x.columns:

    x[
        "OPERACION_N"
    ] = normalizar_codigo_serie(
        x[
            "CODOPERACIONTRANSPORTE"
        ]
    )

else:

    x[
        "OPERACION_N"
    ] = pd.NA


# ------------------------------------------------------------
# 7. AGREGAR CELDA B
#
# B =
# periodo
# × origen
# × configuración
# × mercancía
# ------------------------------------------------------------

LLAVE_B = [

    "PERIODO",
    "ORIGEN_N",
    "CONFIG_N",
    "MERCANCIA_N"
]


panel_B = (

    x

    .groupby(
        LLAVE_B,
        as_index=False,
        sort=False,
        observed=True
    )

    .agg(

        MUNICIPIOORIGEN=(
            "MUNICIPIOORIGEN",
            "first"
        ),

        DEPARTAMENTOORIGEN=(
            "DEPARTAMENTOORIGEN",
            "first"
        ),

        CONFIG_VEHICULO=(
            "CONFIG_VEHICULO",
            "first"
        ),

        MERCANCIA=(
            "MERCANCIA",
            "first"
        ),

        N_FILAS_RNDC=(
            "PERIODO",
            "size"
        ),

        N_OPERACIONES_TRANSPORTE=(
            "OPERACION_N",
            "nunique"
        ),

        VIAJESTOTALES=(
            "VIAJESTOTALES",
            "sum"
        ),

        VIAJESVALORCERO=(
            "VIAJESVALORCERO",
            "sum"
        ),

        KILOGRAMOS=(
            "KILOGRAMOS",
            "sum"
        ),

        VALORESPAGADOS=(
            "VALORESPAGADOS",
            "sum"
        ),

        KM_MEDIANA=(
            "KILOMETROS",
            "median"
        )
    )
)


print(
    "\nCeldas-mes construidas:",
    f"{len(panel_B):,}"
)


# ------------------------------------------------------------
# 8. VARIABLES DERIVADAS
# ------------------------------------------------------------

panel_B[
    "VIAJES_CON_VALOR"
] = (

    panel_B[
        "VIAJESTOTALES"
    ]

    -

    panel_B[
        "VIAJESVALORCERO"
    ]
)


panel_B[
    "TONELADAS"
] = (

    panel_B[
        "KILOGRAMOS"
    ]

    /
    1000
)


# ------------------------------------------------------------
# 9. PRECIO ELEMENTAL PRINCIPAL
#
# Precio del flete por viaje con valor monetario.
#
# NO usamos los viajes con valor cero en el denominador.
# ------------------------------------------------------------

panel_B[
    "PRECIO_VIAJE"
] = np.where(

    (
        panel_B[
            "VIAJES_CON_VALOR"
        ].gt(0)
    )

    &

    (
        panel_B[
            "VALORESPAGADOS"
        ].gt(0)
    ),

    panel_B[
        "VALORESPAGADOS"
    ]

    /

    panel_B[
        "VIAJES_CON_VALOR"
    ],

    np.nan
)


# ------------------------------------------------------------
# 10. VALIDEZ MONETARIA BÁSICA
#
# Esto NO es todavía depuración robusta.
# Solo determina si matemáticamente existe
# un precio elemental positivo y finito.
# ------------------------------------------------------------

panel_B[
    "VALIDO_BASICO"
] = (

    panel_B[
        "VIAJES_CON_VALOR"
    ].gt(0)

    &

    panel_B[
        "VALORESPAGADOS"
    ].gt(0)

    &

    panel_B[
        "PRECIO_VIAJE"
    ].gt(0)

    &

    np.isfinite(
        panel_B[
            "PRECIO_VIAJE"
        ]
    )
)


# ------------------------------------------------------------
# 11. LOG PRECIO
#
# Se utiliza en la regla robusta jerárquica.
# ------------------------------------------------------------

panel_B[
    "LOG_PRECIO"
] = np.where(

    panel_B[
        "VALIDO_BASICO"
    ],

    np.log(
        panel_B[
            "PRECIO_VIAJE"
        ]
    ),

    np.nan
)


# ------------------------------------------------------------
# 12. PRECIO POR VIAJE/KM
#
# IMPORTANTE:
# SOLO será benchmark del NIVEL 2
# entre rutas de diferente distancia.
#
# NO reemplaza PRECIO_VIAJE en el índice.
# ------------------------------------------------------------

panel_B[
    "PRECIO_VIAJE_KM"
] = np.where(

    panel_B[
        "VALIDO_BASICO"
    ]

    &

    panel_B[
        "KM_MEDIANA"
    ].gt(0)

    &

    np.isfinite(
        panel_B[
            "KM_MEDIANA"
        ]
    ),

    panel_B[
        "PRECIO_VIAJE"
    ]

    /

    panel_B[
        "KM_MEDIANA"
    ],

    np.nan
)


panel_B[
    "LOG_PRECIO_VIAJE_KM"
] = np.where(

    panel_B[
        "PRECIO_VIAJE_KM"
    ].gt(0),

    np.log(
        panel_B[
            "PRECIO_VIAJE_KM"
        ]
    ),

    np.nan
)


# ------------------------------------------------------------
# 13. IDENTIFICADOR DE CELDA
# ------------------------------------------------------------

panel_B[
    "CELDA_ID"
] = (

    panel_B[
        "ORIGEN_N"
    ].astype(str)

    +
    "|"

    +
    panel_B[
        "CONFIG_N"
    ].astype(str)

    +
    "|"

    +
    panel_B[
        "MERCANCIA_N"
    ].astype(str)
)


# ------------------------------------------------------------
# 14. ORDEN TEMPORAL
# ------------------------------------------------------------

panel_B[
    "FECHA"
] = pd.to_datetime(
    panel_B[
        "PERIODO"
    ],
    format="%Y%m"
)


panel_B[
    "MES_NUM"
] = (

    panel_B[
        "FECHA"
    ].dt.year
    *
    12

    +

    panel_B[
        "FECHA"
    ].dt.month
)


panel_B = (

    panel_B

    .sort_values(
        [
            "FECHA",
            "ORIGEN_N",
            "CONFIG_N",
            "MERCANCIA_N"
        ],
        kind="stable"
    )

    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 15. AUDITORÍA DE INCONSISTENCIAS BÁSICAS
# ------------------------------------------------------------

audit_basico = pd.DataFrame({

    "CONTROL": [

        "Celdas-mes",
        "Celdas válidas básicas",
        "Celdas inválidas básicas",

        "VIAJES_CON_VALOR < 0",
        "VIAJES_CON_VALOR = 0",

        "VALORESPAGADOS <= 0",

        "KM mediana <= 0",

        "Celdas con >1 operación transporte",
        "Celdas provenientes de >1 fila RNDC"
    ],

    "N": [

        len(
            panel_B
        ),

        int(
            panel_B[
                "VALIDO_BASICO"
            ].sum()
        ),

        int(
            (
                ~panel_B[
                    "VALIDO_BASICO"
                ]
            ).sum()
        ),

        int(
            panel_B[
                "VIAJES_CON_VALOR"
            ].lt(0)
            .sum()
        ),

        int(
            panel_B[
                "VIAJES_CON_VALOR"
            ].eq(0)
            .sum()
        ),

        int(
            panel_B[
                "VALORESPAGADOS"
            ].le(0)
            .sum()
        ),

        int(
            panel_B[
                "KM_MEDIANA"
            ].le(0)
            .sum()
        ),

        int(
            panel_B[
                "N_OPERACIONES_TRANSPORTE"
            ].gt(1)
            .sum()
        ),

        int(
            panel_B[
                "N_FILAS_RNDC"
            ].gt(1)
            .sum()
        )
    ]
})


print(
    "\n"
    + "=" * 105
)

print(
    "AUDITORÍA BÁSICA DEL PANEL"
)

print(
    "=" * 105
)


display(
    audit_basico
)


# ------------------------------------------------------------
# 16. COBERTURA MENSUAL DEL PRECIO ELEMENTAL
# ------------------------------------------------------------

cobertura_mensual_B = (

    panel_B

    .groupby(
        "PERIODO",
        as_index=False
    )

    .agg(

        CELDAS=(
            "CELDA_ID",
            "size"
        ),

        CELDAS_VALIDAS=(
            "VALIDO_BASICO",
            "sum"
        ),

        ORIGENES=(
            "ORIGEN_N",
            "nunique"
        ),

        CONFIGURACIONES=(
            "CONFIG_N",
            "nunique"
        ),

        MERCANCIAS=(
            "MERCANCIA_N",
            "nunique"
        ),

        VIAJES=(
            "VIAJESTOTALES",
            "sum"
        ),

        VIAJES_CON_VALOR=(
            "VIAJES_CON_VALOR",
            "sum"
        ),

        VALOR_PAGADO=(
            "VALORESPAGADOS",
            "sum"
        )
    )
)


cobertura_mensual_B[
    "PCT_CELDAS_VALIDAS"
] = (

    100
    *
    cobertura_mensual_B[
        "CELDAS_VALIDAS"
    ]

    /
    cobertura_mensual_B[
        "CELDAS"
    ]
)


print(
    "\nCobertura mensual — meses con menor % de celdas válidas:"
)


display(

    cobertura_mensual_B

    .sort_values(
        "PCT_CELDAS_VALIDAS"
    )

    .head(12)
)


# ------------------------------------------------------------
# 17. FOCO EN MESES QUE YA ESTABAN BAJO AUDITORÍA
# ------------------------------------------------------------

MESES_FOCO = [
    "202004",
    "202105",
    "202301",
    "202302"
]


print(
    "\nMeses de auditoría previa:"
)


display(

    cobertura_mensual_B.loc[
        cobertura_mensual_B[
            "PERIODO"
        ].isin(
            MESES_FOCO
        )
    ]
)


# ------------------------------------------------------------
# 18. DISTRIBUCIÓN DEL PRECIO ELEMENTAL
#
# SOLO diagnóstico.
# No se usa para cortar observaciones.
# ------------------------------------------------------------

precios_validos = (

    panel_B.loc[
        panel_B[
            "VALIDO_BASICO"
        ],
        "PRECIO_VIAJE"
    ]
)


cuantiles_precio = (

    precios_validos

    .quantile(
        [
            0.001,
            0.005,
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )

    .rename(
        "PRECIO_VIAJE"
    )

    .reset_index()

    .rename(
        columns={
            "index":
                "CUANTIL"
        }
    )
)


print(
    "\nDistribución del precio elemental:"
)


display(
    cuantiles_precio
)


# ------------------------------------------------------------
# 19. OBSERVACIONES EXTREMAS
#
# Solo se muestran.
# NO se eliminan.
# ------------------------------------------------------------

columnas_extremos = [

    "PERIODO",

    "ORIGEN_N",
    "MUNICIPIOORIGEN",

    "CONFIG_N",
    "CONFIG_VEHICULO",

    "MERCANCIA_N",
    "MERCANCIA",

    "VIAJES_CON_VALOR",
    "VALORESPAGADOS",

    "KM_MEDIANA",
    "PRECIO_VIAJE"
]


print(
    "\n12 precios elementales positivos más bajos:"
)


display(

    panel_B.loc[
        panel_B[
            "VALIDO_BASICO"
        ],
        columnas_extremos
    ]

    .nsmallest(
        12,
        "PRECIO_VIAJE"
    )
)


print(
    "\n12 precios elementales positivos más altos:"
)


display(

    panel_B.loc[
        panel_B[
            "VALIDO_BASICO"
        ],
        columnas_extremos
    ]

    .nlargest(
        12,
        "PRECIO_VIAJE"
    )
)


# ------------------------------------------------------------
# 20. CONTINUIDAD TEMPORAL DE LAS CELDAS
#
# Fundamental para saber cuánta historia propia
# tendrá disponible la regla jerárquica.
# ------------------------------------------------------------

continuidad = (

    panel_B

    .groupby(
        "CELDA_ID",
        as_index=False
    )

    .agg(

        N_MESES=(
            "PERIODO",
            "nunique"
        ),

        PRIMER_MES=(
            "PERIODO",
            "min"
        ),

        ULTIMO_MES=(
            "PERIODO",
            "max"
        )
    )
)


print(
    "\nContinuidad temporal de las celdas:"
)


display(

    continuidad[
        "N_MESES"
    ]

    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )

    .to_frame(
        "N_MESES"
    )
)


print(
    "\nCeldas con al menos 4 meses observados:",
    f"{(continuidad['N_MESES'] >= 4).sum():,}",
    "/",
    f"{len(continuidad):,}",
    "(",
    round(
        100
        *
        (
            continuidad[
                "N_MESES"
            ]
            >=
            4
        ).mean(),
        2
    ),
    "% )"
)


# ------------------------------------------------------------
# 21. GUARDAR PANEL ELEMENTAL
# ------------------------------------------------------------

panel_B.to_parquet(

    RUTA_PANEL_B,

    index=False,

    compression="zstd"
)


# ------------------------------------------------------------
# 22. HASH DEL PANEL
# ------------------------------------------------------------

def sha256_panel(
    ruta
):

    h = hashlib.sha256()


    with open(
        ruta,
        "rb"
    ) as f:

        for bloque in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b""
        ):

            h.update(
                bloque
            )


    return h.hexdigest()


SHA_PANEL_B = sha256_panel(
    RUTA_PANEL_B
)


tamano_panel_mb = (

    RUTA_PANEL_B.stat().st_size

    /

    1024**2
)


# ------------------------------------------------------------
# 23. VALIDACIÓN POST-ESCRITURA
# ------------------------------------------------------------

control_panel = pd.read_parquet(
    RUTA_PANEL_B
)


assert len(
    control_panel
) == len(
    panel_B
)


assert (
    control_panel[
        "PERIODO"
    ]
    .astype(str)
    .nunique()
    ==
    60
)


assert (
    control_panel[
        "CELDA_ID"
    ]
    .notna()
    .all()
)


# ------------------------------------------------------------
# 24. RESUMEN FINAL
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 105
)

print(
    "PASS — CELDA 8E"
)

print(
    "=" * 105
)


print(
    "MASTER original:",
    f"{len(rndc):,}",
    "registros"
)


print(
    "Panel elemental:",
    f"{len(panel_B):,}",
    "celdas-mes"
)


print(
    "Meses:",
    panel_B[
        "PERIODO"
    ].nunique()
)


print(
    "Celdas monetariamente válidas:",
    f"{panel_B['VALIDO_BASICO'].sum():,}",
    "(",
    round(
        100
        *
        panel_B[
            "VALIDO_BASICO"
        ].mean(),
        2
    ),
    "% )"
)


print(
    "Archivo:",
    RUTA_PANEL_B
)


print(
    "Tamaño:",
    round(
        tamano_panel_mb,
        3
    ),
    "MB"
)


print(
    "SHA-256:"
)

print(
    SHA_PANEL_B
)


print(
    """
No se eliminaron extremos.
No se imputaron valores.
No se winsorizó.
No se utilizó información futura.

El panel queda preparado para:

NIVEL 1:
historia propia trailing + pares contemporáneos.

NIVEL 2:
pares contemporáneos + misma mercancía/configuración
en otras rutas ajustando por kilómetros.
"""
)


del control_panel
gc.collect()

CELDA 8E — PANEL ELEMENTAL RNDC
Filas MASTER: 78,819
Columnas MASTER: 34

Registros con llave completa: 78,817 / 78,819 ( 99.997 % )

Celdas-mes construidas: 76,218

AUDITORÍA BÁSICA DEL PANEL


,CONTROL,N
0,Celdas-mes,76218
1,Celdas válidas básicas,70928
2,Celdas inválidas básicas,5290
3,VIAJES_CON_VALOR < 0,0
4,VIAJES_CON_VALOR = 0,4973
5,VALORESPAGADOS <= 0,5290
6,KM mediana <= 0,2802
7,Celdas con >1 operación transporte,1581
8,Celdas provenientes de >1 fila RNDC,2207



Cobertura mensual — meses con menor % de celdas válidas:


,PERIODO,CELDAS,CELDAS_VALIDAS,ORIGENES,CONFIGURACIONES,MERCANCIAS,VIAJES,VIAJES_CON_VALOR,VALOR_PAGADO,PCT_CELDAS_VALIDAS
16,202105,170,147,61,8,88,370,317,659940506,86.470588
10,202011,1292,1175,152,11,374,5949,5307,12029056231,90.944272
55,202408,1371,1258,163,9,371,6243,5469,13772116192,91.757841
24,202201,1226,1126,159,14,381,4797,4258,10126546028,91.843393
30,202207,1317,1211,160,9,375,5866,5212,12484891626,91.951405
56,202409,1333,1227,152,10,362,5835,5066,14278083090,92.048012
50,202403,1215,1119,159,9,342,5174,4518,10925929199,92.098765
23,202112,1367,1259,163,15,376,5728,5115,12151344679,92.099488
12,202101,1180,1087,150,13,370,5103,4521,9883647873,92.118644
9,202010,1345,1239,162,11,382,6142,5478,12276645052,92.118959



Meses de auditoría previa:


,PERIODO,CELDAS,CELDAS_VALIDAS,ORIGENES,CONFIGURACIONES,MERCANCIAS,VIAJES,VIAJES_CON_VALOR,VALOR_PAGADO,PCT_CELDAS_VALIDAS
3,202004,693,649,109,7,251,2819,2532,5857736748,93.650794
16,202105,170,147,61,8,88,370,317,659940506,86.470588
36,202301,915,864,131,10,280,4326,4072,12035518319,94.426230
37,202302,989,959,122,9,319,5444,5147,13856169928,96.966633


... (6 outputs left out)


Continuidad temporal de las celdas:


,N_MESES
count,13681.000000
mean,5.571084
std,10.004681
min,1.000000
25%,1.000000
50%,2.000000
75%,5.000000
90%,15.000000
95%,27.000000
99%,54.000000



Celdas con al menos 4 meses observados: 4,156 / 13,681 ( 30.38 % )

PASS — CELDA 8E
MASTER original: 78,819 registros
Panel elemental: 76,218 celdas-mes
Meses: 60
Celdas monetariamente válidas: 70,928 ( 93.06 % )
Archivo: /content/RNDC_PASTO/RNDC_PASTO_PANEL_ELEMENTAL_2020_2024.parquet
Tamaño: 2.234 MB
SHA-256:
d48424fc1545a9de8455ec70d3182d5f78937602b3b78c4ae62f13acf1ef516d

No se eliminaron extremos.
No se imputaron valores.
No se winsorizó.
No se utilizó información futura.

El panel queda preparado para:

NIVEL 1:
historia propia trailing + pares contemporáneos.

NIVEL 2:
pares contemporáneos + misma mercancía/configuración
en otras rutas ajustando por kilómetros.



0

In [25]:
# ============================================================
# CELDA 8F — REGLA JERÁRQUICA SIN INFORMACIÓN FUTURA
# ============================================================
#
# NIVEL 1
# ------------------------------------------------------------
# Si existe historia propia suficiente:
#
#   precio bajo frente a t-12,...,t-1
#   Y
#   precio bajo frente a otras mercancías
#   del mismo origen × configuración en t.
#
# NIVEL 2
# ------------------------------------------------------------
# Si NO existe historia propia suficiente:
#
#   precio bajo frente a otras mercancías
#   del mismo origen × configuración en t
#   Y
#   precio/km bajo para la misma mercancía
#   y configuración en otras rutas hacia Pasto.
#
# EXTREMOS ALTOS:
#   se auditan, NO se eliminan.
#
# NO imputación.
# NO winsorización.
# NO look-ahead.
# ============================================================

import numpy as np
import pandas as pd
import hashlib
from pathlib import Path


# ------------------------------------------------------------
# 1. CARGAR PANEL ELEMENTAL
# ------------------------------------------------------------

assert "RUTA_PANEL_B" in globals()

assert Path(
    RUTA_PANEL_B
).exists()


panel_J = pd.read_parquet(
    RUTA_PANEL_B
)


panel_J[
    "FECHA"
] = pd.to_datetime(
    panel_J[
        "PERIODO"
    ].astype(str),
    format="%Y%m"
)


panel_J[
    "MES_NUM"
] = (
    panel_J[
        "FECHA"
    ].dt.year
    *
    12

    +

    panel_J[
        "FECHA"
    ].dt.month
)


panel_J = (

    panel_J

    .sort_values(
        [
            "MES_NUM",
            "ORIGEN_N",
            "CONFIG_N",
            "MERCANCIA_N"
        ],
        kind="stable"
    )

    .reset_index(
        drop=True
    )
)


print("=" * 105)
print("CELDA 8F — REGLA JERÁRQUICA")
print("=" * 105)

print(
    "Celdas-mes:",
    f"{len(panel_J):,}"
)


# ------------------------------------------------------------
# 2. PARÁMETROS CONGELADOS
# ------------------------------------------------------------

VENTANA_MESES = 12

Z_CRITICO = 3.5

N_TEMP_MIN = 4

N_PEER_MIN = 5

N_RUTAS_MIN = 3


print(
    "\nParámetros:"
)

print(
    "Ventana temporal:",
    VENTANA_MESES,
    "meses previos"
)

print(
    "Z crítico:",
    Z_CRITICO
)

print(
    "Historia mínima:",
    N_TEMP_MIN
)

print(
    "Pares misma ruta mínimos:",
    N_PEER_MIN
)

print(
    "Otras rutas mínimas:",
    N_RUTAS_MIN
)


# ------------------------------------------------------------
# 3. FUNCIÓN ROBUSTA
# ------------------------------------------------------------

def stats_robustos(
    valores
):

    v = np.asarray(
        valores,
        dtype=float
    )


    v = v[
        np.isfinite(v)
    ]


    n = v.size


    if n == 0:

        return (
            0,
            np.nan,
            np.nan
        )


    med = np.median(
        v
    )


    mad = np.median(
        np.abs(
            v - med
        )
    )


    escala_mad = (
        1.4826
        *
        mad
    )


    if (
        np.isfinite(
            escala_mad
        )

        and

        escala_mad > 0
    ):

        escala = (
            escala_mad
        )


    else:

        q25, q75 = np.quantile(
            v,
            [
                0.25,
                0.75
            ]
        )


        iqr = (
            q75
            -
            q25
        )


        escala_iqr = (
            iqr
            /
            1.349
        )


        escala = (

            escala_iqr

            if (
                np.isfinite(
                    escala_iqr
                )

                and

                escala_iqr > 0
            )

            else np.nan
        )


    return (
        int(n),
        float(med),
        float(escala)
    )


# ============================================================
# 4. HISTORIA PROPIA TRAILING
#
# EXACTAMENTE:
#      t-12,...,t-1
#
# Nunca t.
# Nunca futuro.
# ============================================================

temp_n = np.zeros(
    len(panel_J),
    dtype=np.int16
)

temp_med = np.full(
    len(panel_J),
    np.nan
)

temp_scale = np.full(
    len(panel_J),
    np.nan
)


for _, grupo in panel_J.groupby(
    "CELDA_ID",
    sort=False
):

    idx = grupo.index.to_numpy()

    meses = grupo[
        "MES_NUM"
    ].to_numpy(
        dtype=int
    )

    logs = grupo[
        "LOG_PRECIO"
    ].to_numpy(
        dtype=float
    )

    validos = grupo[
        "VALIDO_BASICO"
    ].to_numpy(
        dtype=bool
    )


    for j in range(
        len(grupo)
    ):

        actual = meses[j]


        mask = (

            (meses < actual)

            &

            (
                meses
                >=
                actual
                -
                VENTANA_MESES
            )

            &

            validos

            &

            np.isfinite(
                logs
            )
        )


        n, med, escala = (
            stats_robustos(
                logs[
                    mask
                ]
            )
        )


        pos = idx[j]


        temp_n[
            pos
        ] = n

        temp_med[
            pos
        ] = med

        temp_scale[
            pos
        ] = escala


panel_J[
    "TEMP_N"
] = temp_n


panel_J[
    "TEMP_MEDIANA_LOG"
] = temp_med


panel_J[
    "TEMP_ESCALA"
] = temp_scale


panel_J[
    "Z_TEMP"
] = (

    panel_J[
        "LOG_PRECIO"
    ]

    -

    panel_J[
        "TEMP_MEDIANA_LOG"
    ]

) / panel_J[
    "TEMP_ESCALA"
]


# ============================================================
# 5. PARES CONTEMPORÁNEOS
#
# Otras mercancías del mismo:
#
# periodo × origen × configuración
#
# Leave-one-out exacto.
# ============================================================

peer_n = np.zeros(
    len(panel_J),
    dtype=np.int16
)

peer_med = np.full(
    len(panel_J),
    np.nan
)

peer_scale = np.full(
    len(panel_J),
    np.nan
)


LLAVE_PEER = [

    "PERIODO",
    "ORIGEN_N",
    "CONFIG_N"
]


for _, grupo in panel_J.groupby(
    LLAVE_PEER,
    sort=False,
    observed=True
):

    idx = grupo.index.to_numpy()

    logs = grupo[
        "LOG_PRECIO"
    ].to_numpy(
        dtype=float
    )

    validos = (

        grupo[
            "VALIDO_BASICO"
        ].to_numpy(
            dtype=bool
        )

        &

        np.isfinite(
            logs
        )
    )


    for j in range(
        len(grupo)
    ):

        mask = (
            validos.copy()
        )


        mask[j] = False


        n, med, escala = (
            stats_robustos(
                logs[
                    mask
                ]
            )
        )


        pos = idx[j]


        peer_n[
            pos
        ] = n

        peer_med[
            pos
        ] = med

        peer_scale[
            pos
        ] = escala


panel_J[
    "PEER_N"
] = peer_n


panel_J[
    "PEER_MEDIANA_LOG"
] = peer_med


panel_J[
    "PEER_ESCALA"
] = peer_scale


panel_J[
    "Z_PEER"
] = (

    panel_J[
        "LOG_PRECIO"
    ]

    -

    panel_J[
        "PEER_MEDIANA_LOG"
    ]

) / panel_J[
    "PEER_ESCALA"
]


# ============================================================
# 6. MISMA MERCANCÍA + CONFIGURACIÓN
# EN OTRAS RUTAS HACIA PASTO
#
# Variable:
#
#   log(precio por viaje / km)
#
# Leave-one-origin-out.
# ============================================================

ruta_n = np.zeros(
    len(panel_J),
    dtype=np.int16
)

ruta_med = np.full(
    len(panel_J),
    np.nan
)

ruta_scale = np.full(
    len(panel_J),
    np.nan
)


LLAVE_RUTAS = [

    "PERIODO",
    "MERCANCIA_N",
    "CONFIG_N"
]


for _, grupo in panel_J.groupby(
    LLAVE_RUTAS,
    sort=False,
    observed=True
):

    idx = grupo.index.to_numpy()

    origenes = (
        grupo[
            "ORIGEN_N"
        ]
        .astype(str)
        .to_numpy()
    )


    logs_km = grupo[
        "LOG_PRECIO_VIAJE_KM"
    ].to_numpy(
        dtype=float
    )


    validos = (

        grupo[
            "VALIDO_BASICO"
        ].to_numpy(
            dtype=bool
        )

        &

        np.isfinite(
            logs_km
        )
    )


    # --------------------------------------------------------
    # Una referencia por origen.
    # En el panel B normalmente existe una sola,
    # pero mantenemos la lógica exacta.
    # --------------------------------------------------------

    referencias = {}


    for origen in np.unique(
        origenes[
            validos
        ]
    ):

        valores = logs_km[
            validos
            &
            (
                origenes
                ==
                origen
            )
        ]


        referencias[
            origen
        ] = np.median(
            valores
        )


    for j in range(
        len(grupo)
    ):

        origen_actual = (
            origenes[j]
        )


        otros = [

            valor

            for origen, valor
            in referencias.items()

            if origen
            !=
            origen_actual
        ]


        n, med, escala = (
            stats_robustos(
                otros
            )
        )


        pos = idx[j]


        ruta_n[
            pos
        ] = n

        ruta_med[
            pos
        ] = med

        ruta_scale[
            pos
        ] = escala


panel_J[
    "RUTA_N"
] = ruta_n


panel_J[
    "RUTA_MEDIANA_LOG_KM"
] = ruta_med


panel_J[
    "RUTA_ESCALA"
] = ruta_scale


panel_J[
    "Z_RUTA"
] = (

    panel_J[
        "LOG_PRECIO_VIAJE_KM"
    ]

    -

    panel_J[
        "RUTA_MEDIANA_LOG_KM"
    ]

) / panel_J[
    "RUTA_ESCALA"
]


# ============================================================
# 7. REFERENCIAS UTILIZABLES
# ============================================================

panel_J[
    "HISTORIA_UTILIZABLE"
] = (

    panel_J[
        "TEMP_N"
    ].ge(
        N_TEMP_MIN
    )

    &

    np.isfinite(
        panel_J[
            "Z_TEMP"
        ]
    )
)


panel_J[
    "PEER_UTILIZABLE"
] = (

    panel_J[
        "PEER_N"
    ].ge(
        N_PEER_MIN
    )

    &

    np.isfinite(
        panel_J[
            "Z_PEER"
        ]
    )
)


panel_J[
    "RUTA_UTILIZABLE"
] = (

    panel_J[
        "RUTA_N"
    ].ge(
        N_RUTAS_MIN
    )

    &

    np.isfinite(
        panel_J[
            "Z_RUTA"
        ]
    )
)


# ============================================================
# 8. NIVEL 1
#
# Historia propia suficiente
# +
# pares contemporáneos misma ruta.
# ============================================================

panel_J[
    "FLAG_BAJO_N1"
] = (

    panel_J[
        "VALIDO_BASICO"
    ]

    &

    panel_J[
        "HISTORIA_UTILIZABLE"
    ]

    &

    panel_J[
        "PEER_UTILIZABLE"
    ]

    &

    panel_J[
        "Z_TEMP"
    ].lt(
        -Z_CRITICO
    )

    &

    panel_J[
        "Z_PEER"
    ].lt(
        -Z_CRITICO
    )
)


# ============================================================
# 9. NIVEL 2
#
# Historia propia NO utilizable
# +
# pares misma ruta
# +
# misma mercancía/configuración en otras rutas,
# ajustando distancia.
# ============================================================

panel_J[
    "FLAG_BAJO_N2"
] = (

    panel_J[
        "VALIDO_BASICO"
    ]

    &

    ~panel_J[
        "HISTORIA_UTILIZABLE"
    ]

    &

    panel_J[
        "PEER_UTILIZABLE"
    ]

    &

    panel_J[
        "RUTA_UTILIZABLE"
    ]

    &

    panel_J[
        "Z_PEER"
    ].lt(
        -Z_CRITICO
    )

    &

    panel_J[
        "Z_RUTA"
    ].lt(
        -Z_CRITICO
    )
)


# ============================================================
# 10. REGLA FINAL
# ============================================================

panel_J[
    "FLAG_BAJO_JERARQUICO"
] = (

    panel_J[
        "FLAG_BAJO_N1"
    ]

    |

    panel_J[
        "FLAG_BAJO_N2"
    ]
)


panel_J[
    "ELEGIBLE_JERARQUICO"
] = (

    panel_J[
        "VALIDO_BASICO"
    ]

    &

    ~panel_J[
        "FLAG_BAJO_JERARQUICO"
    ]
)


# ============================================================
# 11. EXTREMOS ALTOS
#
# SOLO AUDITORÍA.
# NO CAMBIAN ELEGIBILIDAD.
# ============================================================

panel_J[
    "FLAG_ALTO_N1"
] = (

    panel_J[
        "VALIDO_BASICO"
    ]

    &

    panel_J[
        "HISTORIA_UTILIZABLE"
    ]

    &

    panel_J[
        "PEER_UTILIZABLE"
    ]

    &

    panel_J[
        "Z_TEMP"
    ].gt(
        Z_CRITICO
    )

    &

    panel_J[
        "Z_PEER"
    ].gt(
        Z_CRITICO
    )
)


panel_J[
    "FLAG_ALTO_N2"
] = (

    panel_J[
        "VALIDO_BASICO"
    ]

    &

    ~panel_J[
        "HISTORIA_UTILIZABLE"
    ]

    &

    panel_J[
        "PEER_UTILIZABLE"
    ]

    &

    panel_J[
        "RUTA_UTILIZABLE"
    ]

    &

    panel_J[
        "Z_PEER"
    ].gt(
        Z_CRITICO
    )

    &

    panel_J[
        "Z_RUTA"
    ].gt(
        Z_CRITICO
    )
)


panel_J[
    "FLAG_ALTO_AUDITAR"
] = (

    panel_J[
        "FLAG_ALTO_N1"
    ]

    |

    panel_J[
        "FLAG_ALTO_N2"
    ]
)


# ============================================================
# 12. AUDITORÍA GENERAL
# ============================================================

resumen_regla = pd.DataFrame({

    "CONTROL": [

        "Celdas-mes totales",

        "Válidas básicas",

        "Historia propia utilizable",

        "Peer misma ruta utilizable",

        "Benchmark otras rutas utilizable",

        "Bajas Nivel 1",

        "Bajas Nivel 2",

        "Bajas jerárquicas totales",

        "Altas marcadas para auditoría",

        "Elegibles finales"
    ],

    "N": [

        len(
            panel_J
        ),

        int(
            panel_J[
                "VALIDO_BASICO"
            ].sum()
        ),

        int(
            panel_J[
                "HISTORIA_UTILIZABLE"
            ].sum()
        ),

        int(
            panel_J[
                "PEER_UTILIZABLE"
            ].sum()
        ),

        int(
            panel_J[
                "RUTA_UTILIZABLE"
            ].sum()
        ),

        int(
            panel_J[
                "FLAG_BAJO_N1"
            ].sum()
        ),

        int(
            panel_J[
                "FLAG_BAJO_N2"
            ].sum()
        ),

        int(
            panel_J[
                "FLAG_BAJO_JERARQUICO"
            ].sum()
        ),

        int(
            panel_J[
                "FLAG_ALTO_AUDITAR"
            ].sum()
        ),

        int(
            panel_J[
                "ELEGIBLE_JERARQUICO"
            ].sum()
        )
    ]
})


resumen_regla[
    "PCT_TOTAL"
] = (

    100
    *
    resumen_regla[
        "N"
    ]

    /
    len(
        panel_J
    )
)


print(
    "\nRESUMEN REGLA JERÁRQUICA"
)

display(
    resumen_regla
)


# ============================================================
# 13. EXCLUIDAS
# ============================================================

columnas_audit = [

    "PERIODO",

    "ORIGEN_N",
    "MUNICIPIOORIGEN",

    "CONFIG_N",
    "CONFIG_VEHICULO",

    "MERCANCIA_N",
    "MERCANCIA",

    "PRECIO_VIAJE",
    "KM_MEDIANA",

    "TEMP_N",
    "Z_TEMP",

    "PEER_N",
    "Z_PEER",

    "RUTA_N",
    "Z_RUTA",

    "FLAG_BAJO_N1",
    "FLAG_BAJO_N2"
]


excluidas_J = (

    panel_J.loc[
        panel_J[
            "FLAG_BAJO_JERARQUICO"
        ],
        columnas_audit
    ]

    .sort_values(
        [
            "PERIODO",
            "PRECIO_VIAJE"
        ]
    )

    .reset_index(
        drop=True
    )
)


print(
    "\nOBSERVACIONES BAJAS EXCLUIDAS:"
)


display(
    excluidas_J.head(
        30
    )
)


# ============================================================
# 14. ¿LOS $2.000 FUERON DETECTADOS?
# ============================================================

audit_2000 = (

    panel_J.loc[
        panel_J[
            "PRECIO_VIAJE"
        ].eq(
            2000
        ),
        [
            "PERIODO",
            "ORIGEN_N",
            "CONFIG_N",
            "MERCANCIA_N",
            "PRECIO_VIAJE",

            "TEMP_N",
            "Z_TEMP",

            "PEER_N",
            "Z_PEER",

            "RUTA_N",
            "Z_RUTA",

            "FLAG_BAJO_N1",
            "FLAG_BAJO_N2",

            "ELEGIBLE_JERARQUICO"
        ]
    ]
)


print(
    "\nAUDITORÍA ESPECÍFICA — PRECIO = $2.000"
)

display(
    audit_2000
)


# ============================================================
# 15. FRECUENCIA DE EXCLUSIONES POR MES
# ============================================================

frecuencia_mes = (

    panel_J

    .groupby(
        "PERIODO",
        as_index=False
    )

    .agg(

        CELDAS=(
            "PERIODO",
            "size"
        ),

        VALIDAS=(
            "VALIDO_BASICO",
            "sum"
        ),

        EXCLUIDAS_BAJAS=(
            "FLAG_BAJO_JERARQUICO",
            "sum"
        ),

        ALTAS_AUDITADAS=(
            "FLAG_ALTO_AUDITAR",
            "sum"
        )
    )
)


frecuencia_mes[
    "PCT_EXCLUIDAS_SOBRE_VALIDAS"
] = (

    100
    *
    frecuencia_mes[
        "EXCLUIDAS_BAJAS"
    ]

    /

    frecuencia_mes[
        "VALIDAS"
    ]
)


print(
    "\nMeses con mayor proporción "
    "de exclusiones:"
)


display(

    frecuencia_mes

    .sort_values(
        "PCT_EXCLUIDAS_SOBRE_VALIDAS",
        ascending=False
    )

    .head(
        15
    )
)


# ============================================================
# 16. GUARDAR PANEL JERÁRQUICO
# ============================================================

RUTA_PANEL_J = (

    ROOT
    /
    "RNDC_PASTO_PANEL_JERARQUICO_2020_2024.parquet"
)


panel_J.to_parquet(

    RUTA_PANEL_J,

    index=False,

    compression="zstd"
)


# ============================================================
# 17. HASH
# ============================================================

def sha256_archivo_8f(
    ruta
):

    h = hashlib.sha256()


    with open(
        ruta,
        "rb"
    ) as f:

        for bloque in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b""
        ):

            h.update(
                bloque
            )


    return h.hexdigest()


SHA_PANEL_J = (
    sha256_archivo_8f(
        RUTA_PANEL_J
    )
)


print(
    "\n"
    + "=" * 105
)

print(
    "PASS — CELDA 8F"
)

print(
    "=" * 105
)


print(
    "Bajas excluidas:",
    int(
        panel_J[
            "FLAG_BAJO_JERARQUICO"
        ].sum()
    )
)


print(
    "Extremos altos conservados:",
    int(
        panel_J[
            "FLAG_ALTO_AUDITAR"
        ].sum()
    )
)


print(
    "Elegibles finales:",
    int(
        panel_J[
            "ELEGIBLE_JERARQUICO"
        ].sum()
    )
)


print(
    "Panel:",
    RUTA_PANEL_J
)


print(
    "SHA-256:",
    SHA_PANEL_J
)


print(
    f"""
Regla jerárquica aplicada.

Z crítico: {Z_CRITICO}
Ventana: {VENTANA_MESES} meses previos.
Historia mínima: {N_TEMP_MIN}.
Pares misma ruta mínimos: {N_PEER_MIN}.
Otras rutas mínimas: {N_RUTAS_MIN}.

Los extremos altos NO fueron eliminados.
No se utilizaron meses futuros.
No hubo imputación.
No hubo winsorización.

La siguiente etapa puede construir
Törnqvist principal + Fisher de sensibilidad.
"""
)

CELDA 8F — REGLA JERÁRQUICA
Celdas-mes: 76,218

Parámetros:
Ventana temporal: 12 meses previos
Z crítico: 3.5
Historia mínima: 4
Pares misma ruta mínimos: 5
Otras rutas mínimas: 3

RESUMEN REGLA JERÁRQUICA


,CONTROL,N,PCT_TOTAL
0,Celdas-mes totales,76218,100.000000
1,Válidas básicas,70928,93.059382
2,Historia propia utilizable,33576,44.052586
3,Peer misma ruta utilizable,42921,56.313469
4,Benchmark otras rutas utilizable,27809,36.486132
5,Bajas Nivel 1,272,0.356871
6,Bajas Nivel 2,191,0.250597
7,Bajas jerárquicas totales,463,0.607468
8,Altas marcadas para auditoría,171,0.224356
9,Elegibles finales,70465,92.451914



OBSERVACIONES BAJAS EXCLUIDAS:


,PERIODO,ORIGEN_N,MUNICIPIOORIGEN,CONFIG_N,CONFIG_VEHICULO,MERCANCIA_N,MERCANCIA,PRECIO_VIAJE,KM_MEDIANA,TEMP_N,Z_TEMP,PEER_N,Z_PEER,RUTA_N,Z_RUTA,FLAG_BAJO_N1,FLAG_BAJO_N2
0,202001,11001000,BOGOTA BOGOTA D. C.,2,Camión Rígido de 2 ejes,001704,ARTICULOS DE CONFITERIA SIN CACAO (INCLUIDO EL...,1.500000e+04,819.0,0,NaN,63,-21.594874,5,-27.007244,False,True
1,202001,76001000,CALI VALLE DEL CAUCA,2,Camión Rígido de 2 ejes,003923,ARTICULOS PARA EL TRANSPORTE O ENVASADO; DE P...,1.500000e+05,378.0,0,NaN,70,-6.480390,4,-9.437298,False,True
2,202001,76892000,YUMBO VALLE DEL CAUCA,3S3,Tractocamión de 3 ejes Semiremolque de 3 ejes,002203,CERVEZA DE MALTA,2.741823e+05,393.0,0,NaN,23,-11.070469,4,-14.648681,False,True
3,202001,68001000,BUCARAMANGA SANTANDER,2,Camión Rígido de 2 ejes,003101,ABONOS DE ORIGEN ANIMAL O VEGETAL; INCLUSO ME...,1.100000e+06,1136.0,0,NaN,7,-6.674587,9,-3.642606,False,True
4,202002,11001000,BOGOTA BOGOTA D. C.,2,Camión Rígido de 2 ejes,001704,ARTICULOS DE CONFITERIA SIN CACAO (INCLUIDO EL...,1.500000e+04,819.0,1,NaN,74,-15.399912,5,-208.607157,False,True
5,202002,76001000,CALI VALLE DEL CAUCA,3S3,Tractocamión de 3 ejes Semiremolque de 3 ejes,006901,LADRILLOS; PLACAS; BALDOSAS Y OTRAS PIEZAS C...,1.000000e+05,378.0,0,NaN,10,-26.665527,3,-142.981761,False,True
6,202002,25286000,FUNZA CUNDINAMARCA,2,Camión Rígido de 2 ejes,009990,CONTENEDOR VACIO,1.200000e+05,823.0,0,NaN,10,-20.568307,11,-5.018900,False,True
7,202002,11001000,BOGOTA BOGOTA D. C.,2,Camión Rígido de 2 ejes,007216,PERFILES DE HIERRO O DE ACERO SIN ALEAR,3.500000e+05,819.0,1,NaN,74,-5.217858,4,-28.023089,False,True
8,202002,76001000,CALI VALLE DEL CAUCA,2,Camión Rígido de 2 ejes,008702,VEHICULOS AUTOMOVILES PARA EL TRANSPORTE DE DI...,3.550000e+05,378.0,1,NaN,67,-3.622983,3,-8.713760,False,True
9,202002,11001000,BOGOTA BOGOTA D. C.,2,Camión Rígido de 2 ejes,003105,ABONOS MINERALES O QUIMICOS; CON DOS O TRES D...,4.000000e+05,819.0,1,NaN,74,-4.786215,6,-5.579708,False,True



AUDITORÍA ESPECÍFICA — PRECIO = $2.000


,PERIODO,ORIGEN_N,CONFIG_N,MERCANCIA_N,PRECIO_VIAJE,TEMP_N,Z_TEMP,PEER_N,Z_PEER,RUTA_N,Z_RUTA,FLAG_BAJO_N1,FLAG_BAJO_N2,ELEGIBLE_JERARQUICO
174,202001,19001000,3S2,008430,2000.0,0,NaN,0,NaN,0,NaN,False,False,True
5412,202006,19001000,3S2,008430,2000.0,1,NaN,0,NaN,1,NaN,False,False,True
5415,202006,19001000,3S3,008430,2000.0,0,NaN,1,NaN,0,NaN,False,False,True
6107,202006,76130000,3S2,008430,2000.0,0,NaN,0,NaN,1,NaN,False,False,True
6614,202007,17380000,3S2,008430,2000.0,0,NaN,0,NaN,0,NaN,False,False,True
6627,202007,19075000,3S2,008430,2000.0,0,NaN,0,NaN,1,NaN,False,False,True
6628,202007,19075000,3S3,008430,2000.0,0,NaN,0,NaN,0,NaN,False,False,True
7926,202008,19824000,3S3,008430,2000.0,0,NaN,0,NaN,0,NaN,False,False,True
8368,202008,73168000,3S4,008430,2000.0,0,NaN,0,NaN,0,NaN,False,False,True
9170,202009,19824000,3S2,008430,2000.0,0,NaN,0,NaN,2,-144.619603,False,False,True



Meses con mayor proporción de exclusiones:


,PERIODO,CELDAS,VALIDAS,EXCLUIDAS_BAJAS,ALTAS_AUDITADAS,PCT_EXCLUIDAS_SOBRE_VALIDAS
14,202103,1386,1305,15,2,1.149425
5,202006,1169,1090,11,5,1.009174
54,202407,1306,1205,12,4,0.995851
7,202008,1221,1133,11,2,0.970874
9,202010,1345,1239,12,1,0.968523
49,202402,1353,1252,12,3,0.958466
4,202005,1028,968,9,3,0.929752
39,202304,1254,1189,11,11,0.925147
11,202012,1290,1202,11,2,0.915141
45,202310,1428,1327,12,4,0.904295



PASS — CELDA 8F
Bajas excluidas: 463
Extremos altos conservados: 171
Elegibles finales: 70465
Panel: /content/RNDC_PASTO/RNDC_PASTO_PANEL_JERARQUICO_2020_2024.parquet
SHA-256: c285c6e3c1f9de3eb10b38ef2ab5fe9236617d04a4d404a9a7362e19ec09d9b0

Regla jerárquica aplicada.

Z crítico: 3.5
Ventana: 12 meses previos.
Historia mínima: 4.
Pares misma ruta mínimos: 5.
Otras rutas mínimas: 3.

Los extremos altos NO fueron eliminados.
No se utilizaron meses futuros.
No hubo imputación.
No hubo winsorización.

La siguiente etapa puede construir
Törnqvist principal + Fisher de sensibilidad.



In [26]:
# ============================================================
# CELDA 8G — ÍNDICE REGIONAL DE FLETES HACIA PASTO
# TÖRNQVIST PRINCIPAL + FISHER + SENSIBILIDADES
# ============================================================
#
# PERIODO:
#   2020-01 a 2024-12
#
# BASE:
#   2020-01 = 100
#
# ESLABONES:
#   59 comparaciones mensuales
#   202001->202002 ... 202411->202412
#
# DEFINICIÓN ELEMENTAL:
#   origen × configuración × mercancía
#
# PRECIO:
#   VALORESPAGADOS / VIAJES_CON_VALOR
#
# PRINCIPAL:
#   Regla jerárquica de Celda 8F
#   + Törnqvist encadenado
#
# ROBUSTEZ:
#   1. Fisher sobre EXACTAMENTE las mismas celdas.
#   2. Serie cruda: VALIDO_BASICO.
#   3. Jerárquico excluyendo únicamente PRECIO=2000
#      como sensibilidad, NO como regla principal.
#
# NO descarga nada.
# NO imputa.
# NO winsoriza.
# NO elimina extremos altos.
# ============================================================

import pandas as pd
import numpy as np
import hashlib
from pathlib import Path


# ------------------------------------------------------------
# 1. CARGAR PANEL JERÁRQUICO
# ------------------------------------------------------------

assert "RUTA_PANEL_J" in globals(), (
    "Debe ejecutarse primero la Celda 8F."
)

assert Path(RUTA_PANEL_J).exists()


panel_idx = pd.read_parquet(
    RUTA_PANEL_J
)


panel_idx[
    "PERIODO"
] = (
    panel_idx[
        "PERIODO"
    ]
    .astype(str)
    .str.zfill(6)
)


print("=" * 110)
print("CELDA 8G — ÍNDICE REGIONAL DE FLETES")
print("=" * 110)

print(
    "Celdas-mes:",
    f"{len(panel_idx):,}"
)

print(
    "Meses:",
    panel_idx["PERIODO"].nunique()
)


# ------------------------------------------------------------
# 2. VALIDACIONES
# ------------------------------------------------------------

REQUERIDAS_8G = [

    "PERIODO",

    "ORIGEN_N",
    "CONFIG_N",
    "MERCANCIA_N",

    "PRECIO_VIAJE",

    "VALORESPAGADOS",
    "VIAJES_CON_VALOR",
    "KILOGRAMOS",

    "VALIDO_BASICO",
    "ELEGIBLE_JERARQUICO"
]


faltantes = [

    c

    for c in REQUERIDAS_8G

    if c not in panel_idx.columns
]


assert not faltantes, (
    f"Faltan columnas: {faltantes}"
)


# ------------------------------------------------------------
# 3. PERIODOS Y ESLABONES
# ------------------------------------------------------------

PERIODOS_INDICE = sorted(

    panel_idx[
        "PERIODO"
    ]
    .unique()
    .tolist()
)


assert len(PERIODOS_INDICE) == 60

assert PERIODOS_INDICE[0] == "202001"
assert PERIODOS_INDICE[-1] == "202412"


COMPARACIONES = list(
    zip(
        PERIODOS_INDICE[:-1],
        PERIODOS_INDICE[1:]
    )
)


assert len(COMPARACIONES) == 59


# ------------------------------------------------------------
# 4. MODOS DE ANÁLISIS
# ------------------------------------------------------------

panel_idx[
    "ELEGIBLE_SENS_2000"
] = (

    panel_idx[
        "ELEGIBLE_JERARQUICO"
    ]

    &

    ~panel_idx[
        "PRECIO_VIAJE"
    ].eq(
        2000
    )
)


MODOS = {

    "CRUDO":
        "VALIDO_BASICO",

    "JERARQUICO":
        "ELEGIBLE_JERARQUICO",

    "SENS_SIN_2000":
        "ELEGIBLE_SENS_2000"
}


# ------------------------------------------------------------
# 5. PREPARAR MESES UNA SOLA VEZ
#
# Evita filtrar el DataFrame completo cientos de veces.
# ------------------------------------------------------------

LLAVES = [

    "ORIGEN_N",
    "CONFIG_N",
    "MERCANCIA_N"
]


COLUMNAS_INDICE = (

    LLAVES

    +

    [
        "PRECIO_VIAJE",
        "VALORESPAGADOS",
        "VIAJES_CON_VALOR",
        "KILOGRAMOS",

        "VALIDO_BASICO",
        "ELEGIBLE_JERARQUICO",
        "ELEGIBLE_SENS_2000"
    ]
)


MESES = {

    periodo:

        panel_idx.loc[
            panel_idx[
                "PERIODO"
            ].eq(periodo),
            COLUMNAS_INDICE
        ]
        .copy()

    for periodo
    in PERIODOS_INDICE
}


# ============================================================
# 6. FUNCIÓN DE UN ESLABÓN
# ============================================================

def calcular_eslabon_8G(
    periodo_0,
    periodo_1,
    modo
):

    columna_elegible = (
        MODOS[
            modo
        ]
    )


    base0 = MESES[
        periodo_0
    ]


    base1 = MESES[
        periodo_1
    ]


    # --------------------------------------------------------
    # Universo válido básico
    # para medir cobertura.
    #
    # No usamos el universo depurado como denominador porque
    # queremos saber cuánto de la información monetaria
    # originalmente utilizable queda representado.
    # --------------------------------------------------------

    universo0 = (

        base0.loc[
            base0[
                "VALIDO_BASICO"
            ]
        ]
    )


    universo1 = (

        base1.loc[
            base1[
                "VALIDO_BASICO"
            ]
        ]
    )


    # --------------------------------------------------------
    # Celdas elegibles según modo
    # --------------------------------------------------------

    t0 = (

        base0.loc[
            base0[
                columna_elegible
            ],
            LLAVES
            +
            [
                "PRECIO_VIAJE",
                "VALORESPAGADOS",
                "VIAJES_CON_VALOR",
                "KILOGRAMOS"
            ]
        ]

        .rename(
            columns={

                "PRECIO_VIAJE":
                    "P0",

                "VALORESPAGADOS":
                    "V0",

                "VIAJES_CON_VALOR":
                    "Q0",

                "KILOGRAMOS":
                    "KG0"
            }
        )
    )


    t1 = (

        base1.loc[
            base1[
                columna_elegible
            ],
            LLAVES
            +
            [
                "PRECIO_VIAJE",
                "VALORESPAGADOS",
                "VIAJES_CON_VALOR",
                "KILOGRAMOS"
            ]
        ]

        .rename(
            columns={

                "PRECIO_VIAJE":
                    "P1",

                "VALORESPAGADOS":
                    "V1",

                "VIAJES_CON_VALOR":
                    "Q1",

                "KILOGRAMOS":
                    "KG1"
            }
        )
    )


    # --------------------------------------------------------
    # Matching exacto
    # --------------------------------------------------------

    comp = t0.merge(

        t1,

        on=LLAVES,

        how="inner",

        validate="one_to_one"
    )


    # --------------------------------------------------------
    # Condiciones matemáticas
    # --------------------------------------------------------

    comp = comp.loc[

        comp[
            "P0"
        ].gt(0)

        &

        comp[
            "P1"
        ].gt(0)

        &

        comp[
            "V0"
        ].gt(0)

        &

        comp[
            "V1"
        ].gt(0)

        &

        comp[
            "Q0"
        ].gt(0)

        &

        comp[
            "Q1"
        ].gt(0)

    ].copy()


    assert len(comp) > 0, (
        f"Sin celdas comparables: "
        f"{periodo_0}->{periodo_1}, {modo}"
    )


    # ========================================================
    # 7. TÖRNQVIST
    # ========================================================

    valor0 = (
        comp[
            "V0"
        ].sum()
    )


    valor1 = (
        comp[
            "V1"
        ].sum()
    )


    assert valor0 > 0
    assert valor1 > 0


    comp[
        "S0"
    ] = (

        comp[
            "V0"
        ]

        /

        valor0
    )


    comp[
        "S1"
    ] = (

        comp[
            "V1"
        ]

        /

        valor1
    )


    comp[
        "W_TORNQVIST"
    ] = (

        comp[
            "S0"
        ]

        +

        comp[
            "S1"
        ]

    ) / 2


    assert np.isclose(

        comp[
            "W_TORNQVIST"
        ].sum(),

        1.0,

        atol=1e-10
    )


    comp[
        "LOG_RELATIVO"
    ] = np.log(

        comp[
            "P1"
        ]

        /

        comp[
            "P0"
        ]
    )


    assert np.isfinite(
        comp[
            "LOG_RELATIVO"
        ]
    ).all()


    delta_log_T = (

        comp[
            "W_TORNQVIST"
        ]

        *

        comp[
            "LOG_RELATIVO"
        ]

    ).sum()


    relativo_T = float(
        np.exp(
            delta_log_T
        )
    )


    var_T = (

        100
        *
        (
            relativo_T
            -
            1
        )
    )


    # ========================================================
    # 8. FISHER
    #
    # Q = VIAJES_CON_VALOR.
    #
    # Se calcula sobre EXACTAMENTE las mismas
    # celdas utilizadas por Törnqvist.
    # ========================================================

    denominador_L = (

        comp[
            "P0"
        ]

        *

        comp[
            "Q0"
        ]

    ).sum()


    numerador_L = (

        comp[
            "P1"
        ]

        *

        comp[
            "Q0"
        ]

    ).sum()


    denominador_P = (

        comp[
            "P0"
        ]

        *

        comp[
            "Q1"
        ]

    ).sum()


    numerador_P = (

        comp[
            "P1"
        ]

        *

        comp[
            "Q1"
        ]

    ).sum()


    assert denominador_L > 0
    assert denominador_P > 0


    relativo_L = (
        numerador_L
        /
        denominador_L
    )


    relativo_P = (
        numerador_P
        /
        denominador_P
    )


    relativo_F = float(

        np.sqrt(
            relativo_L
            *
            relativo_P
        )
    )


    var_F = (

        100
        *
        (
            relativo_F
            -
            1
        )
    )


    # ========================================================
    # 9. COBERTURA EN AMBOS MESES
    # ========================================================

    def cobertura(
        numerador,
        denominador
    ):

        return (

            100
            *
            numerador
            /
            denominador

            if (
                np.isfinite(
                    denominador
                )

                and

                denominador > 0
            )

            else np.nan
        )


    # --------------------------------------------------------
    # Totales universo válido básico
    # --------------------------------------------------------

    total_valor0 = (
        universo0[
            "VALORESPAGADOS"
        ].sum()
    )

    total_valor1 = (
        universo1[
            "VALORESPAGADOS"
        ].sum()
    )


    total_viajes0 = (
        universo0[
            "VIAJES_CON_VALOR"
        ].sum()
    )

    total_viajes1 = (
        universo1[
            "VIAJES_CON_VALOR"
        ].sum()
    )


    total_kg0 = (
        universo0[
            "KILOGRAMOS"
        ].sum()
    )

    total_kg1 = (
        universo1[
            "KILOGRAMOS"
        ].sum()
    )


    cobertura_valor0 = cobertura(
        comp["V0"].sum(),
        total_valor0
    )

    cobertura_valor1 = cobertura(
        comp["V1"].sum(),
        total_valor1
    )


    cobertura_viajes0 = cobertura(
        comp["Q0"].sum(),
        total_viajes0
    )

    cobertura_viajes1 = cobertura(
        comp["Q1"].sum(),
        total_viajes1
    )


    cobertura_kg0 = cobertura(
        comp["KG0"].sum(),
        total_kg0
    )

    cobertura_kg1 = cobertura(
        comp["KG1"].sum(),
        total_kg1
    )


    # ========================================================
    # 10. DIAGNÓSTICOS DEL MATCH
    # ========================================================

    relativos = (

        comp[
            "P1"
        ]

        /

        comp[
            "P0"
        ]
    )


    # contribución absoluta más grande al Törnqvist
    comp[
        "CONTRIB_LOG"
    ] = (

        comp[
            "W_TORNQVIST"
        ]

        *

        comp[
            "LOG_RELATIVO"
        ]
    )


    contrib_abs_total = (

        comp[
            "CONTRIB_LOG"
        ]
        .abs()
        .sum()
    )


    max_contrib_share = (

        100
        *
        comp[
            "CONTRIB_LOG"
        ]
        .abs()
        .max()

        /
        contrib_abs_total

        if contrib_abs_total > 0

        else np.nan
    )


    return {

        "modo":
            modo,

        "periodo_0":
            periodo_0,

        "periodo_1":
            periodo_1,

        "comparacion":
            (
                f"{periodo_0}"
                f"->{periodo_1}"
            ),

        "celdas_comparables":
            int(
                len(comp)
            ),

        "relativo_T":
            relativo_T,

        "var_T_pct":
            float(
                var_T
            ),

        "relativo_L":
            float(
                relativo_L
            ),

        "relativo_P":
            float(
                relativo_P
            ),

        "relativo_F":
            relativo_F,

        "var_F_pct":
            float(
                var_F
            ),

        "dif_TF_pp":
            float(
                var_T
                -
                var_F
            ),

        "cob_valor_0_pct":
            cobertura_valor0,

        "cob_valor_1_pct":
            cobertura_valor1,

        "cob_viajes_0_pct":
            cobertura_viajes0,

        "cob_viajes_1_pct":
            cobertura_viajes1,

        "cob_kg_0_pct":
            cobertura_kg0,

        "cob_kg_1_pct":
            cobertura_kg1,

        "relativo_mediana":
            float(
                relativos.median()
            ),

        "relativo_p05":
            float(
                relativos.quantile(
                    0.05
                )
            ),

        "relativo_p95":
            float(
                relativos.quantile(
                    0.95
                )
            ),

        "max_contrib_abs_pct":
            float(
                max_contrib_share
            )
    }


# ============================================================
# 11. CALCULAR LOS 59 ESLABONES
# ============================================================

resultados_8G = []


for modo in MODOS:

    for p0, p1 in COMPARACIONES:

        resultados_8G.append(

            calcular_eslabon_8G(

                p0,
                p1,
                modo
            )
        )


df_8G = pd.DataFrame(
    resultados_8G
)


# ------------------------------------------------------------
# Controles
# ------------------------------------------------------------

for modo in MODOS:

    assert (

        df_8G.loc[
            df_8G[
                "modo"
            ].eq(modo)
        ].shape[0]

        ==
        59
    )


assert np.isfinite(
    df_8G[
        "var_T_pct"
    ]
).all()


assert np.isfinite(
    df_8G[
        "var_F_pct"
    ]
).all()


# ============================================================
# 12. SEPARAR MODOS
# ============================================================

principal = (

    df_8G.loc[
        df_8G[
            "modo"
        ].eq(
            "JERARQUICO"
        )
    ]

    .sort_values(
        "periodo_1"
    )

    .reset_index(
        drop=True
    )
)


crudo = (

    df_8G.loc[
        df_8G[
            "modo"
        ].eq(
            "CRUDO"
        )
    ]

    .sort_values(
        "periodo_1"
    )

    .reset_index(
        drop=True
    )
)


sin_2000 = (

    df_8G.loc[
        df_8G[
            "modo"
        ].eq(
            "SENS_SIN_2000"
        )
    ]

    .sort_values(
        "periodo_1"
    )

    .reset_index(
        drop=True
    )
)


# ============================================================
# 13. ROBUSTEZ TÖRNQVIST VS FISHER
# ============================================================

corr_TF = (

    principal[
        "var_T_pct"
    ]

    .corr(
        principal[
            "var_F_pct"
        ]
    )
)


mae_TF = (

    (
        principal[
            "var_T_pct"
        ]

        -

        principal[
            "var_F_pct"
        ]
    )

    .abs()
    .mean()
)


max_TF = (

    (
        principal[
            "var_T_pct"
        ]

        -

        principal[
            "var_F_pct"
        ]
    )

    .abs()
    .max()
)


# ============================================================
# 14. ROBUSTEZ PRINCIPAL VS CRUDO
# ============================================================

corr_crudo = (

    principal[
        "var_T_pct"
    ]

    .corr(
        crudo[
            "var_T_pct"
        ]
    )
)


mae_crudo = (

    (
        principal[
            "var_T_pct"
        ]

        -

        crudo[
            "var_T_pct"
        ]
    )

    .abs()
    .mean()
)


max_crudo = (

    (
        principal[
            "var_T_pct"
        ]

        -

        crudo[
            "var_T_pct"
        ]
    )

    .abs()
    .max()
)


# ============================================================
# 15. SENSIBILIDAD PRECIO = $2.000
# ============================================================

corr_2000 = (

    principal[
        "var_T_pct"
    ]

    .corr(
        sin_2000[
            "var_T_pct"
        ]
    )
)


mae_2000 = (

    (
        principal[
            "var_T_pct"
        ]

        -

        sin_2000[
            "var_T_pct"
        ]
    )

    .abs()
    .mean()
)


max_2000 = (

    (
        principal[
            "var_T_pct"
        ]

        -

        sin_2000[
            "var_T_pct"
        ]
    )

    .abs()
    .max()
)


resumen_robustez = pd.DataFrame({

    "COMPARACION": [

        "Törnqvist vs Fisher",

        "Törnqvist jerárquico vs crudo",

        "Törnqvist principal vs sin $2.000"
    ],

    "CORRELACION": [

        corr_TF,
        corr_crudo,
        corr_2000
    ],

    "MAE_PP": [

        mae_TF,
        mae_crudo,
        mae_2000
    ],

    "MAX_DIF_PP": [

        max_TF,
        max_crudo,
        max_2000
    ]
})


print(
    "\n"
    + "=" * 110
)

print(
    "ROBUSTEZ DEL ÍNDICE"
)

print(
    "=" * 110
)


display(
    resumen_robustez
)


# ============================================================
# 16. COBERTURA DEL ÍNDICE PRINCIPAL
# ============================================================

resumen_cobertura = pd.DataFrame({

    "METRICA": [

        "Valor t-1",
        "Valor t",

        "Viajes t-1",
        "Viajes t",

        "Kg t-1",
        "Kg t"
    ],

    "MEDIA": [

        principal[
            "cob_valor_0_pct"
        ].mean(),

        principal[
            "cob_valor_1_pct"
        ].mean(),

        principal[
            "cob_viajes_0_pct"
        ].mean(),

        principal[
            "cob_viajes_1_pct"
        ].mean(),

        principal[
            "cob_kg_0_pct"
        ].mean(),

        principal[
            "cob_kg_1_pct"
        ].mean()
    ],

    "MINIMO": [

        principal[
            "cob_valor_0_pct"
        ].min(),

        principal[
            "cob_valor_1_pct"
        ].min(),

        principal[
            "cob_viajes_0_pct"
        ].min(),

        principal[
            "cob_viajes_1_pct"
        ].min(),

        principal[
            "cob_kg_0_pct"
        ].min(),

        principal[
            "cob_kg_1_pct"
        ].min()
    ],

    "MAXIMO": [

        principal[
            "cob_valor_0_pct"
        ].max(),

        principal[
            "cob_valor_1_pct"
        ].max(),

        principal[
            "cob_viajes_0_pct"
        ].max(),

        principal[
            "cob_viajes_1_pct"
        ].max(),

        principal[
            "cob_kg_0_pct"
        ].max(),

        principal[
            "cob_kg_1_pct"
        ].max()
    ]
})


print(
    "\nCOBERTURA — ÍNDICE PRINCIPAL"
)


display(
    resumen_cobertura
)


# ============================================================
# 17. ESLABONES CON MENOR COBERTURA
# ============================================================

principal[
    "COB_MIN"
] = (

    principal[
        [
            "cob_valor_0_pct",
            "cob_valor_1_pct",
            "cob_viajes_0_pct",
            "cob_viajes_1_pct",
            "cob_kg_0_pct",
            "cob_kg_1_pct"
        ]
    ]

    .min(
        axis=1
    )
)


print(
    "\n10 eslabones con menor cobertura:"
)


display(

    principal[
        [
            "comparacion",
            "celdas_comparables",
            "COB_MIN",

            "cob_valor_1_pct",
            "cob_viajes_1_pct",
            "cob_kg_1_pct",

            "var_T_pct",
            "var_F_pct",

            "max_contrib_abs_pct"
        ]
    ]

    .sort_values(
        "COB_MIN"
    )

    .head(
        10
    )
)


# ============================================================
# 18. FOCO EN MAYO 2021
# ============================================================

print(
    "\n"
    + "=" * 110
)

print(
    "FOCO — MAYO 2021"
)

print(
    "=" * 110
)


display(

    principal.loc[
        principal[
            "comparacion"
        ].isin(
            [
                "202104->202105",
                "202105->202106"
            ]
        ),
        [
            "comparacion",

            "celdas_comparables",

            "cob_valor_0_pct",
            "cob_valor_1_pct",

            "cob_viajes_0_pct",
            "cob_viajes_1_pct",

            "cob_kg_0_pct",
            "cob_kg_1_pct",

            "var_T_pct",
            "var_F_pct",

            "max_contrib_abs_pct"
        ]
    ]
)


# ============================================================
# 19. ENCADENAR ÍNDICES
#
# Enero 2020 = 100.
#
# La primera VARIACIÓN observable es febrero 2020
# porque no disponemos de diciembre 2019 mensual
# en la fuente reproducible actual.
# ============================================================

indice_T = 100.0
indice_F = 100.0


serie_indice = [

    {

        "PERIODO":
            "202001",

        "INDICE_FLETE_TORNQVIST":
            100.0,

        "INDICE_FLETE_FISHER":
            100.0,

        "VAR_FLETE_TORNQVIST_PCT":
            np.nan,

        "VAR_FLETE_FISHER_PCT":
            np.nan,

        "CELDAS_COMPARABLES":
            np.nan,

        "COB_VALOR_T_PCT":
            np.nan,

        "COB_VIAJES_T_PCT":
            np.nan,

        "COB_KG_T_PCT":
            np.nan
    }
]


for _, fila in principal.iterrows():

    indice_T *= float(
        fila[
            "relativo_T"
        ]
    )


    indice_F *= float(
        fila[
            "relativo_F"
        ]
    )


    serie_indice.append({

        "PERIODO":
            fila[
                "periodo_1"
            ],

        "INDICE_FLETE_TORNQVIST":
            indice_T,

        "INDICE_FLETE_FISHER":
            indice_F,

        "VAR_FLETE_TORNQVIST_PCT":
            fila[
                "var_T_pct"
            ],

        "VAR_FLETE_FISHER_PCT":
            fila[
                "var_F_pct"
            ],

        "CELDAS_COMPARABLES":
            fila[
                "celdas_comparables"
            ],

        "COB_VALOR_T_PCT":
            fila[
                "cob_valor_1_pct"
            ],

        "COB_VIAJES_T_PCT":
            fila[
                "cob_viajes_1_pct"
            ],

        "COB_KG_T_PCT":
            fila[
                "cob_kg_1_pct"
            ]
    })


variable2 = pd.DataFrame(
    serie_indice
)


assert len(variable2) == 60

assert (
    variable2[
        "VAR_FLETE_TORNQVIST_PCT"
    ]
    .notna()
    .sum()
    ==
    59
)


# ============================================================
# 20. INCORPORAR SENSIBILIDADES MENSUALES
# ============================================================

sens_crudo = (

    crudo[
        [
            "periodo_1",
            "var_T_pct"
        ]
    ]

    .rename(
        columns={

            "periodo_1":
                "PERIODO",

            "var_T_pct":
                "VAR_FLETE_CRUDO_PCT"
        }
    )
)


sens_2000 = (

    sin_2000[
        [
            "periodo_1",
            "var_T_pct"
        ]
    ]

    .rename(
        columns={

            "periodo_1":
                "PERIODO",

            "var_T_pct":
                "VAR_FLETE_SIN_2000_PCT"
        }
    )
)


variable2 = (

    variable2

    .merge(
        sens_crudo,
        on="PERIODO",
        how="left"
    )

    .merge(
        sens_2000,
        on="PERIODO",
        how="left"
    )
)


# ============================================================
# 21. GUARDAR VARIABLE 2
# ============================================================

RUTA_VARIABLE2_PARQUET = (

    ROOT
    /
    "VARIABLE2_FLETE_REGIONAL_PASTO_2020_2024.parquet"
)


RUTA_VARIABLE2_CSV = (

    ROOT
    /
    "VARIABLE2_FLETE_REGIONAL_PASTO_2020_2024.csv"
)


variable2.to_parquet(

    RUTA_VARIABLE2_PARQUET,

    index=False,

    compression="zstd"
)


variable2.to_csv(

    RUTA_VARIABLE2_CSV,

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# 22. HASH
# ============================================================

def sha256_8G(
    ruta
):

    h = hashlib.sha256()


    with open(
        ruta,
        "rb"
    ) as f:

        for bloque in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b""
        ):

            h.update(
                bloque
            )


    return h.hexdigest()


SHA_VARIABLE2 = sha256_8G(
    RUTA_VARIABLE2_PARQUET
)


# ============================================================
# 23. MOSTRAR VARIABLE FINAL
# ============================================================

print(
    "\n"
    + "=" * 110
)

print(
    "VARIABLE 2 — SERIE MENSUAL"
)

print(
    "=" * 110
)


display(
    variable2
)


# ============================================================
# 24. CONTROLES DUROS
# ============================================================

assert variable2[
    "PERIODO"
].nunique() == 60


assert variable2[
    "PERIODO"
].min() == "202001"


assert variable2[
    "PERIODO"
].max() == "202412"


assert np.isfinite(

    variable2.loc[
        variable2[
            "PERIODO"
        ].ne(
            "202001"
        ),
        "VAR_FLETE_TORNQVIST_PCT"
    ]

).all()


assert principal[
    "cob_valor_1_pct"
].between(
    0,
    100.000001
).all()


assert principal[
    "cob_viajes_1_pct"
].between(
    0,
    100.000001
).all()


assert principal[
    "cob_kg_1_pct"
].between(
    0,
    100.000001
).all()


# ============================================================
# 25. RESULTADO
# ============================================================

print(
    "\n"
    + "=" * 110
)

print(
    "PASS — CELDA 8G"
)

print(
    "=" * 110
)


print(
    "Meses del índice:",
    len(
        variable2
    )
)


print(
    "Variaciones mensuales observadas:",
    int(
        variable2[
            "VAR_FLETE_TORNQVIST_PCT"
        ].notna().sum()
    )
)


print(
    "\nRobustez Törnqvist vs Fisher:"
)

print(
    "Correlación:",
    round(
        corr_TF,
        6
    )
)

print(
    "MAE:",
    round(
        mae_TF,
        6
    ),
    "pp"
)

print(
    "Máxima diferencia:",
    round(
        max_TF,
        6
    ),
    "pp"
)


print(
    "\nSensibilidad regla jerárquica vs crudo:"
)

print(
    "Correlación:",
    round(
        corr_crudo,
        6
    )
)

print(
    "MAE:",
    round(
        mae_crudo,
        6
    ),
    "pp"
)


print(
    "\nSensibilidad exclusión de $2.000:"
)

print(
    "Correlación:",
    round(
        corr_2000,
        6
    )
)

print(
    "MAE:",
    round(
        mae_2000,
        6
    ),
    "pp"
)


print(
    "\nArchivo principal:"
)

print(
    RUTA_VARIABLE2_PARQUET
)


print(
    "SHA-256:"
)

print(
    SHA_VARIABLE2
)


print(
    """
ÍNDICE PRINCIPAL:
Törnqvist encadenado.

CELDA:
origen × configuración × mercancía.

PRECIO ELEMENTAL:
valor pagado por viaje con valor.

DEPURACIÓN:
regla jerárquica sin información futura.

ROBUSTEZ:
Fisher sobre las mismas celdas.
Serie cruda.
Sensibilidad sin precios exactamente iguales a $2.000.

Extremos altos conservados.
Sin imputación.
Sin winsorización.
"""
)

CELDA 8G — ÍNDICE REGIONAL DE FLETES
Celdas-mes: 76,218
Meses: 60

ROBUSTEZ DEL ÍNDICE


,COMPARACION,CORRELACION,MAE_PP,MAX_DIF_PP
0,Törnqvist vs Fisher,0.999096,0.174071,1.547264
1,Törnqvist jerárquico vs crudo,0.991811,0.205234,6.812639
2,Törnqvist principal vs sin $2.000,1.000000,0.000141,0.008317



COBERTURA — ÍNDICE PRINCIPAL


,METRICA,MEDIA,MINIMO,MAXIMO
0,Valor t-1,79.755161,19.700214,84.670101
1,Valor t,79.299396,17.898999,84.821793
2,Viajes t-1,82.161570,29.014638,86.374086
3,Viajes t,81.869884,29.559505,86.945546
4,Kg t-1,79.488242,15.137718,87.063182
5,Kg t,79.198643,16.041304,87.684367



10 eslabones con menor cobertura:


,comparacion,celdas_comparables,COB_MIN,cob_valor_1_pct,cob_viajes_1_pct,cob_kg_1_pct,var_T_pct,var_F_pct,max_contrib_abs_pct
15,202104->202105,85,15.137718,73.495726,75.238095,68.056836,20.901337,19.354073,13.932901
16,202105->202106,94,16.041304,17.898999,29.559505,16.041304,-0.578763,0.505267,14.519416
37,202302->202303,534,50.597614,58.813596,71.298905,50.597614,-8.629420,-8.849210,6.136765
35,202212->202301,415,60.822584,62.166606,65.518950,62.974368,43.024373,41.946329,6.045822
2,202003->202004,368,65.317347,81.730415,84.459733,84.530527,-2.540699,-1.841138,15.614524
3,202004->202005,370,66.936639,66.936639,72.099973,67.911346,0.373169,0.322672,7.559159
41,202306->202307,618,67.413438,81.417059,78.877588,67.413438,-1.480630,-1.400057,5.853317
55,202408->202409,682,72.071188,80.776420,81.755151,72.071188,4.782095,4.785880,5.966685
12,202101->202102,598,72.108241,78.780261,82.139901,81.232749,-0.188122,-0.190078,4.911460
40,202305->202306,628,72.359876,81.579664,82.960447,81.054019,-4.325603,-4.414964,5.126676



FOCO — MAYO 2021


,comparacion,celdas_comparables,cob_valor_0_pct,cob_valor_1_pct,cob_viajes_0_pct,cob_viajes_1_pct,cob_kg_0_pct,cob_kg_1_pct,var_T_pct,var_F_pct,max_contrib_abs_pct
15,202104->202105,85,19.700214,73.495726,29.014638,75.238095,15.137718,68.056836,20.901337,19.354073,13.932901
16,202105->202106,94,75.955536,17.898999,79.682540,29.559505,74.024884,16.041304,-0.578763,0.505267,14.519416



VARIABLE 2 — SERIE MENSUAL


,PERIODO,INDICE_FLETE_TORNQVIST,INDICE_FLETE_FISHER,VAR_FLETE_TORNQVIST_PCT,VAR_FLETE_FISHER_PCT,CELDAS_COMPARABLES,COB_VALOR_T_PCT,COB_VIAJES_T_PCT,COB_KG_T_PCT,VAR_FLETE_CRUDO_PCT,VAR_FLETE_SIN_2000_PCT
0,202001,100.000000,100.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,202002,99.863536,99.919486,-0.136464,-0.080514,558.0,79.961606,82.335069,82.027423,-0.191602,-0.136464
2,202003,100.693334,100.782750,0.830932,0.863960,587.0,80.868181,83.429362,80.424104,0.806579,0.830932
3,202004,98.135020,98.927200,-2.540699,-1.841138,368.0,81.730415,84.459733,84.530527,-2.601311,-2.540699
4,202005,98.501230,99.246411,0.373169,0.322672,370.0,66.936639,72.099973,67.911346,0.242291,0.373169
5,202006,98.162783,98.793261,-0.343597,-0.456590,509.0,78.975368,80.554314,79.176567,-0.347183,-0.343597
6,202007,99.423690,99.749862,1.284507,0.968286,621.0,80.586895,79.698807,74.070545,1.186920,1.284507
7,202008,98.799370,99.673087,-0.627939,-0.076968,625.0,82.775834,85.829273,86.076949,-0.794509,-0.627939
8,202009,99.188398,99.205229,0.393755,-0.469392,608.0,81.460678,83.800267,84.217432,0.400520,0.385438
9,202010,99.777599,99.648321,0.594022,0.446642,646.0,81.974229,84.640884,82.549053,0.528028,0.594022



PASS — CELDA 8G
Meses del índice: 60
Variaciones mensuales observadas: 59

Robustez Törnqvist vs Fisher:
Correlación: 0.999096
MAE: 0.174071 pp
Máxima diferencia: 1.547264 pp

Sensibilidad regla jerárquica vs crudo:
Correlación: 0.991811
MAE: 0.205234 pp

Sensibilidad exclusión de $2.000:
Correlación: 1.0
MAE: 0.000141 pp

Archivo principal:
/content/RNDC_PASTO/VARIABLE2_FLETE_REGIONAL_PASTO_2020_2024.parquet
SHA-256:
4336398949cbc5c74a95b7bf982f241a5b4e5564e44b6d67313536c0a4fa0e19

ÍNDICE PRINCIPAL:
Törnqvist encadenado.

CELDA:
origen × configuración × mercancía.

PRECIO ELEMENTAL:
valor pagado por viaje con valor.

DEPURACIÓN:
regla jerárquica sin información futura.

ROBUSTEZ:
Fisher sobre las mismas celdas.
Serie cruda.
Sensibilidad sin precios exactamente iguales a $2.000.

Extremos altos conservados.
Sin imputación.
Sin winsorización.



In [28]:
# ============================================================
# CELDA 8H — AUDITORÍA FINAL DE ESLABONES CRÍTICOS
# + SENSIBILIDAD ORIGEN × CONFIGURACIÓN
# ============================================================
#
# OBJETIVOS
# 1. Auditar:
#       202104 -> 202105
#       202105 -> 202106
#       202212 -> 202301
#       202408 -> 202409
#
# 2. Identificar qué celdas explican los movimientos.
#
# 3. Ver exactamente qué observaciones producen la diferencia
#    entre índice crudo y jerárquico.
#
# 4. Construir una sensibilidad más agregada:
#       origen × configuración
#
#    Esta NO reemplaza el índice principal.
#
# NO descarga nada.
# NO modifica panel_J.
# NO imputa.
# NO winsoriza.
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. CARGAR PANEL JERÁRQUICO
# ------------------------------------------------------------

panel_H = pd.read_parquet(
    RUTA_PANEL_J
)

panel_H["PERIODO"] = (
    panel_H["PERIODO"]
    .astype(str)
    .str.zfill(6)
)


ESLABONES_CRITICOS = [
    ("202104", "202105"),
    ("202105", "202106"),
    ("202212", "202301"),
    ("202408", "202409")
]


LLAVE_B = [
    "ORIGEN_N",
    "CONFIG_N",
    "MERCANCIA_N"
]


print("=" * 110)
print("CELDA 8H — AUDITORÍA FINAL")
print("=" * 110)


# ============================================================
# 2. FUNCIÓN PARA RECONSTRUIR UN MATCH
# ============================================================

def match_eslabon_H(
    p0,
    p1,
    elegibilidad="ELEGIBLE_JERARQUICO"
):

    a = (
        panel_H.loc[
            panel_H["PERIODO"].eq(p0)
            &
            panel_H[elegibilidad],
            LLAVE_B
            +
            [
                "MUNICIPIOORIGEN",
                "CONFIG_VEHICULO",
                "MERCANCIA",
                "PRECIO_VIAJE",
                "VALORESPAGADOS",
                "VIAJES_CON_VALOR",
                "KILOGRAMOS",
                "FLAG_BAJO_JERARQUICO"
            ]
        ]
        .copy()
    )

    b = (
        panel_H.loc[
            panel_H["PERIODO"].eq(p1)
            &
            panel_H[elegibilidad],
            LLAVE_B
            +
            [
                "MUNICIPIOORIGEN",
                "CONFIG_VEHICULO",
                "MERCANCIA",
                "PRECIO_VIAJE",
                "VALORESPAGADOS",
                "VIAJES_CON_VALOR",
                "KILOGRAMOS",
                "FLAG_BAJO_JERARQUICO"
            ]
        ]
        .copy()
    )


    a = a.rename(
        columns={
            "PRECIO_VIAJE": "P0",
            "VALORESPAGADOS": "V0",
            "VIAJES_CON_VALOR": "Q0",
            "KILOGRAMOS": "KG0"
        }
    )


    b = b.rename(
        columns={
            "MUNICIPIOORIGEN": "MUNICIPIOORIGEN_1",
            "CONFIG_VEHICULO": "CONFIG_VEHICULO_1",
            "MERCANCIA": "MERCANCIA_1",
            "PRECIO_VIAJE": "P1",
            "VALORESPAGADOS": "V1",
            "VIAJES_CON_VALOR": "Q1",
            "KILOGRAMOS": "KG1"
        }
    )


    m = a.merge(
        b,
        on=LLAVE_B,
        how="inner",
        validate="one_to_one"
    )


    m = m.loc[
        m["P0"].gt(0)
        &
        m["P1"].gt(0)
        &
        m["V0"].gt(0)
        &
        m["V1"].gt(0)
    ].copy()


    # --------------------------------------------------------
    # Törnqvist
    # --------------------------------------------------------

    m["S0"] = (
        m["V0"]
        /
        m["V0"].sum()
    )


    m["S1"] = (
        m["V1"]
        /
        m["V1"].sum()
    )


    m["W"] = (
        m["S0"]
        +
        m["S1"]
    ) / 2


    m["RELATIVO"] = (
        m["P1"]
        /
        m["P0"]
    )


    m["LOG_REL"] = np.log(
        m["RELATIVO"]
    )


    m["CONTRIB_LOG"] = (
        m["W"]
        *
        m["LOG_REL"]
    )


    m["CONTRIB_ABS"] = (
        m["CONTRIB_LOG"]
        .abs()
    )


    total_abs = (
        m["CONTRIB_ABS"]
        .sum()
    )


    m["PARTICIPACION_CONTRIB_ABS"] = np.where(
        total_abs > 0,
        100
        *
        m["CONTRIB_ABS"]
        /
        total_abs,
        np.nan
    )


    return m


# ============================================================
# 3. AUDITAR LOS CUATRO ESLABONES
# ============================================================

for p0, p1 in ESLABONES_CRITICOS:

    m = match_eslabon_H(
        p0,
        p1,
        "ELEGIBLE_JERARQUICO"
    )


    variacion = (
        100
        *
        (
            np.exp(
                m["CONTRIB_LOG"].sum()
            )
            -
            1
        )
    )


    print(
        "\n"
        + "=" * 110
    )

    print(
        f"ESLABÓN {p0} -> {p1}"
    )

    print(
        "=" * 110
    )


    print(
        "Celdas comparables:",
        len(m)
    )


    print(
        "Variación Törnqvist:",
        round(
            variacion,
            6
        ),
        "%"
    )


    print(
        "Máxima participación "
        "en contribución absoluta:",
        round(
            m[
                "PARTICIPACION_CONTRIB_ABS"
            ].max(),
            3
        ),
        "%"
    )


    print(
        "\n15 principales contribuciones:"
    )


    # --------------------------------------------------------
    # CORRECCIÓN:
    # Primero se ordena el DataFrame completo por CONTRIB_ABS
    # y después se seleccionan las columnas a mostrar.
    # --------------------------------------------------------

    tabla_top = (

        m

        .sort_values(
            "CONTRIB_ABS",
            ascending=False
        )

        .head(15)

        [
            [
                "ORIGEN_N",
                "MUNICIPIOORIGEN",

                "CONFIG_N",
                "CONFIG_VEHICULO",

                "MERCANCIA_N",
                "MERCANCIA",

                "P0",
                "P1",
                "RELATIVO",

                "S0",
                "S1",

                "CONTRIB_LOG",
                "CONTRIB_ABS",
                "PARTICIPACION_CONTRIB_ABS"
            ]
        ]
    )


    display(
        tabla_top
    )


# ============================================================
# 4. SEPTIEMBRE 2024:
# ¿QUÉ EXCLUYÓ LA REGLA?
# ============================================================

for periodo in [
    "202408",
    "202409"
]:

    excl = (

        panel_H.loc[
            panel_H["PERIODO"].eq(periodo)
            &
            panel_H["VALIDO_BASICO"]
            &
            ~panel_H["ELEGIBLE_JERARQUICO"],
            [
                "PERIODO",

                "ORIGEN_N",
                "MUNICIPIOORIGEN",

                "CONFIG_N",
                "CONFIG_VEHICULO",

                "MERCANCIA_N",
                "MERCANCIA",

                "PRECIO_VIAJE",

                "TEMP_N",
                "Z_TEMP",

                "PEER_N",
                "Z_PEER",

                "RUTA_N",
                "Z_RUTA",

                "FLAG_BAJO_N1",
                "FLAG_BAJO_N2"
            ]
        ]

        .sort_values(
            "PRECIO_VIAJE"
        )
    )


    print(
        "\n"
        + "=" * 110
    )

    print(
        "EXCLUIDAS POR REGLA —",
        periodo
    )

    print(
        "=" * 110
    )


    print(
        "N:",
        len(excl)
    )


    display(
        excl.head(30)
    )


# ============================================================
# 5. SENSIBILIDAD MÁS AGREGADA
#
# CELDA A:
#   ORIGEN × CONFIGURACIÓN
#
# IMPORTANTE:
# Se construye usando únicamente las celdas B
# elegibles jerárquicamente.
#
# Así NO reintroducimos las observaciones descartadas.
# ============================================================

elegibles = (

    panel_H.loc[
        panel_H[
            "ELEGIBLE_JERARQUICO"
        ]
    ]
    .copy()
)


panel_A = (

    elegibles

    .groupby(
        [
            "PERIODO",
            "ORIGEN_N",
            "CONFIG_N"
        ],
        as_index=False,
        observed=True
    )

    .agg(

        VALORESPAGADOS=(
            "VALORESPAGADOS",
            "sum"
        ),

        VIAJES_CON_VALOR=(
            "VIAJES_CON_VALOR",
            "sum"
        ),

        KILOGRAMOS=(
            "KILOGRAMOS",
            "sum"
        ),

        N_MERCANCIAS=(
            "MERCANCIA_N",
            "nunique"
        )
    )
)


panel_A[
    "PRECIO_A"
] = np.where(

    panel_A[
        "VIAJES_CON_VALOR"
    ].gt(0),

    panel_A[
        "VALORESPAGADOS"
    ]
    /
    panel_A[
        "VIAJES_CON_VALOR"
    ],

    np.nan
)


panel_A = panel_A.loc[
    panel_A[
        "PRECIO_A"
    ].gt(0)
].copy()


# ============================================================
# 6. TÖRNQVIST A
# ============================================================

periodos_A = sorted(
    panel_A[
        "PERIODO"
    ].unique()
)


resultados_A = []


for p0, p1 in zip(
    periodos_A[:-1],
    periodos_A[1:]
):

    a = (
        panel_A.loc[
            panel_A[
                "PERIODO"
            ].eq(p0)
        ]
        .copy()
    )


    b = (
        panel_A.loc[
            panel_A[
                "PERIODO"
            ].eq(p1)
        ]
        .copy()
    )


    comp = a.merge(
        b,
        on=[
            "ORIGEN_N",
            "CONFIG_N"
        ],
        how="inner",
        suffixes=(
            "_0",
            "_1"
        ),
        validate="one_to_one"
    )


    comp = comp.loc[
        comp[
            "PRECIO_A_0"
        ].gt(0)
        &
        comp[
            "PRECIO_A_1"
        ].gt(0)
    ].copy()


    s0 = (
        comp[
            "VALORESPAGADOS_0"
        ]
        /
        comp[
            "VALORESPAGADOS_0"
        ].sum()
    )


    s1 = (
        comp[
            "VALORESPAGADOS_1"
        ]
        /
        comp[
            "VALORESPAGADOS_1"
        ].sum()
    )


    w = (
        s0
        +
        s1
    ) / 2


    logrel = np.log(
        comp[
            "PRECIO_A_1"
        ]
        /
        comp[
            "PRECIO_A_0"
        ]
    )


    rel = np.exp(
        (
            w
            *
            logrel
        ).sum()
    )


    resultados_A.append({

        "PERIODO":
            p1,

        "VAR_FLETE_A_PCT":
            100
            *
            (
                rel
                -
                1
            ),

        "CELDAS_A_COMPARABLES":
            len(
                comp
            )
    })


sens_A = pd.DataFrame(
    resultados_A
)


# ============================================================
# 7. COMPARAR B PRINCIPAL VS A AGREGADO
# ============================================================

comparacion_A_B = (

    variable2[
        [
            "PERIODO",
            "VAR_FLETE_TORNQVIST_PCT"
        ]
    ]

    .merge(
        sens_A,
        on="PERIODO",
        how="inner"
    )
)


corr_AB = (

    comparacion_A_B[
        "VAR_FLETE_TORNQVIST_PCT"
    ]

    .corr(
        comparacion_A_B[
            "VAR_FLETE_A_PCT"
        ]
    )
)


mae_AB = (

    (
        comparacion_A_B[
            "VAR_FLETE_TORNQVIST_PCT"
        ]

        -

        comparacion_A_B[
            "VAR_FLETE_A_PCT"
        ]
    )

    .abs()
    .mean()
)


print(
    "\n"
    + "=" * 110
)

print(
    "SENSIBILIDAD DE COMPOSICIÓN"
)

print(
    "=" * 110
)


print(
    "Correlación B vs A:",
    round(
        corr_AB,
        6
    )
)


print(
    "MAE:",
    round(
        mae_AB,
        6
    ),
    "pp"
)


print(
    "\nEslabones críticos:"
)


display(

    comparacion_A_B.loc[
        comparacion_A_B[
            "PERIODO"
        ].isin(
            [
                "202105",
                "202106",
                "202301",
                "202409"
            ]
        )
    ]
)


# ============================================================
# 8. CONCENTRACIÓN DEL PRINCIPAL
# ============================================================

resumen_criticos = []


for p0, p1 in ESLABONES_CRITICOS:

    m = match_eslabon_H(
        p0,
        p1
    )


    hhi_pesos = (

        m[
            "W"
        ]
        .pow(2)
        .sum()
    )


    top5_peso = (

        100
        *
        m[
            "W"
        ]
        .nlargest(5)
        .sum()
    )


    top5_contrib = (

        100
        *
        m[
            "CONTRIB_ABS"
        ]
        .nlargest(5)
        .sum()
        /
        m[
            "CONTRIB_ABS"
        ].sum()
    )


    resumen_criticos.append({

        "ESLABON":
            f"{p0}->{p1}",

        "N_CELDAS":
            len(m),

        "HHI_PESOS":
            hhi_pesos,

        "TOP5_PESO_PCT":
            top5_peso,

        "TOP5_CONTRIB_ABS_PCT":
            top5_contrib
    })


resumen_criticos = pd.DataFrame(
    resumen_criticos
)


print(
    "\nConcentración de eslabones críticos:"
)


display(
    resumen_criticos
)


print(
    "\n"
    + "=" * 110
)

print(
    "PASS — CELDA 8H"
)

print(
    "=" * 110
)

print(
    """
La celda NO modifica la Variable 2.

Solo audita:
- mayo de 2021,
- enero de 2023,
- septiembre de 2024,
- concentración,
- sensibilidad de composición.

Si estas pruebas son coherentes,
podemos congelar formalmente
el índice regional de fletes.
"""
)

CELDA 8H — AUDITORÍA FINAL

ESLABÓN 202104 -> 202105
Celdas comparables: 85
Variación Törnqvist: 20.901337 %
Máxima participación en contribución absoluta: 13.933 %

15 principales contribuciones:


,ORIGEN_N,MUNICIPIOORIGEN,CONFIG_N,CONFIG_VEHICULO,MERCANCIA_N,MERCANCIA,P0,P1,RELATIVO,S0,S1,CONTRIB_LOG,CONTRIB_ABS,PARTICIPACION_CONTRIB_ABS
73,76892000,YUMBO VALLE DEL CAUCA,2,Camión Rígido de 2 ejes,004819,CAJAS; SACOS; BOLSAS; CUCURUCHOS Y DEMAS EN...,1.083713e+06,2.653319e+06,2.448360,0.019168,0.049234,0.030624,0.030624,13.932901
16,25286000,FUNZA CUNDINAMARCA,2,Camión Rígido de 2 ejes,003003,MEDICAMENTOS (CON EXCLUSION DE LOS PRODUCTOS D...,1.775000e+06,3.500000e+06,1.971831,0.003305,0.072161,0.025619,0.025619,11.655751
76,76892000,YUMBO VALLE DEL CAUCA,2,Camión Rígido de 2 ejes,009980,PRODUCTOS VARIOS,1.077096e+06,1.675000e+06,1.555108,0.037601,0.013814,0.011351,0.011351,5.164236
47,76001000,CALI VALLE DEL CAUCA,2,Camión Rígido de 2 ejes,001104,GRANOS DE CEREALES TRABAJADOS DE OTRA FORMA (P...,1.051429e+06,2.000000e+06,1.902174,0.003426,0.028864,0.010381,0.010381,4.723069
54,76001000,CALI VALLE DEL CAUCA,2,Camión Rígido de 2 ejes,007323,ARTICULOS DE USO DOMESTICO Y SUS PARTES; DE F...,1.065556e+06,2.333333e+06,2.189781,0.008928,0.014432,0.009155,0.009155,4.165034
26,5001000,MEDELLIN ANTIOQUIA,2,Camión Rígido de 2 ejes,001905,PRODUCTOS DE PANADERIA; PASTELERIA O GALLETER...,1.808824e+06,2.900000e+06,1.603252,0.028626,0.005979,0.008167,0.008167,3.715827
50,76001000,CALI VALLE DEL CAUCA,2,Camión Rígido de 2 ejes,003003,MEDICAMENTOS (CON EXCLUSION DE LOS PRODUCTOS D...,1.013333e+06,1.738286e+06,1.715414,0.002830,0.025087,0.007533,0.007533,3.427151
35,5318000,GUARNE ANTIOQUIA,2,Camión Rígido de 2 ejes,003304,PREPARACIONES DE BELLEZA; DE MAQUILLAJE Y PAR...,1.700000e+06,5.000000e+06,2.941176,0.000791,0.010309,0.005987,0.005987,2.724032
14,19698000,SANTANDER DE QUILICHAO CAUCA,2,Camión Rígido de 2 ejes,001704,ARTICULOS DE CONFITERIA SIN CACAO (INCLUIDO EL...,1.100000e+06,2.200000e+06,2.000000,0.003584,0.013607,0.005958,0.005958,2.710722
13,19573000,PUERTO TEJADA CAUCA,2,Camión Rígido de 2 ejes,003808,INSECTICIDAS; RATICIDAS; FUNGICIDAS; HERBIC...,2.400000e+06,5.850000e+06,2.437500,0.001117,0.012061,0.005871,0.005871,2.670966



ESLABÓN 202105 -> 202106
Celdas comparables: 94
Variación Törnqvist: -0.578763 %
Máxima participación en contribución absoluta: 14.519 %

15 principales contribuciones:


,ORIGEN_N,MUNICIPIOORIGEN,CONFIG_N,CONFIG_VEHICULO,MERCANCIA_N,MERCANCIA,P0,P1,RELATIVO,S0,S1,CONTRIB_LOG,CONTRIB_ABS,PARTICIPACION_CONTRIB_ABS
15,25286000,FUNZA CUNDINAMARCA,2,Camión Rígido de 2 ejes,003003,MEDICAMENTOS (CON EXCLUSION DE LOS PRODUCTOS D...,3.500000e+06,1.580000e+06,0.451429,0.069824,0.002721,-0.028849,0.028849,14.519416
58,76001000,CALI VALLE DEL CAUCA,2,Camión Rígido de 2 ejes,009880,MISCELANEOS CONTENIDOS EN PAQUETES ( PAQUETEO ),1.292954e+06,1.528057e+06,1.181834,0.056747,0.092112,0.012435,0.012435,6.258303
16,25286000,FUNZA CUNDINAMARCA,3S3,Tractocamión de 3 ejes Semiremolque de 3 ejes,002202,"AGUA, INCLUIDAS EL AGUA MINERAL Y LA GASEADA, ...",4.500000e+06,6.547500e+06,1.455000,0.008977,0.045107,0.010141,0.010141,5.103875
8,13001000,CARTAGENA BOLIVAR,2S2,Tractocamión de 2 ejes Semiremolque de 2 ejes,008703,COCHES DE TURISMO Y DEMAS VEHICULOS AUTOMOVILE...,4.298000e+06,7.130807e+06,1.659099,0.008574,0.027019,0.009010,0.009010,4.534672
57,76001000,CALI VALLE DEL CAUCA,2,Camión Rígido de 2 ejes,007323,ARTICULOS DE USO DOMESTICO Y SUS PARTES; DE F...,2.333333e+06,1.371429e+06,0.587755,0.013965,0.013227,-0.007226,0.007226,3.636551
81,76892000,YUMBO VALLE DEL CAUCA,2,Camión Rígido de 2 ejes,009880,MISCELANEOS CONTENIDOS EN PAQUETES ( PAQUETEO ),1.134000e+06,1.426478e+06,1.257917,0.015836,0.046188,0.007116,0.007116,3.581403
4,11001000,BOGOTA BOGOTA D. C.,2,Camión Rígido de 2 ejes,009980,PRODUCTOS VARIOS,1.772250e+06,2.003159e+06,1.130291,0.028285,0.084871,0.006929,0.006929,3.487506
49,76001000,CALI VALLE DEL CAUCA,2,Camión Rígido de 2 ejes,001104,GRANOS DE CEREALES TRABAJADOS DE OTRA FORMA (P...,2.000000e+06,1.310000e+06,0.655000,0.027930,0.003610,-0.006672,0.006672,3.358208
12,19573000,PUERTO TEJADA CAUCA,2,Camión Rígido de 2 ejes,003808,INSECTICIDAS; RATICIDAS; FUNGICIDAS; HERBIC...,5.850000e+06,2.687500e+06,0.459402,0.011671,0.003703,-0.005979,0.005979,3.009173
78,76892000,YUMBO VALLE DEL CAUCA,2,Camión Rígido de 2 ejes,004819,CAJAS; SACOS; BOLSAS; CUCURUCHOS Y DEMAS EN...,2.653319e+06,2.292547e+06,0.864030,0.047640,0.033957,-0.005963,0.005963,3.000914



ESLABÓN 202212 -> 202301
Celdas comparables: 415
Variación Törnqvist: 43.024373 %
Máxima participación en contribución absoluta: 6.046 %

15 principales contribuciones:


,ORIGEN_N,MUNICIPIOORIGEN,CONFIG_N,CONFIG_VEHICULO,MERCANCIA_N,MERCANCIA,P0,P1,RELATIVO,S0,S1,CONTRIB_LOG,CONTRIB_ABS,PARTICIPACION_CONTRIB_ABS
384,76892000,YUMBO VALLE DEL CAUCA,2,Camión Rígido de 2 ejes,009980,PRODUCTOS VARIOS,1.173636e+06,3.330725e+06,2.837953,0.006533,0.039174,0.023838,0.023838,6.045822
260,76001000,CALI VALLE DEL CAUCA,2,Camión Rígido de 2 ejes,009980,PRODUCTOS VARIOS,1.271219e+06,2.192649e+06,1.724839,0.035703,0.036632,0.019716,0.019716,5.000374
349,76892000,YUMBO VALLE DEL CAUCA,2,Camión Rígido de 2 ejes,001001,TRIGO Y MORCAJO O TRANQUILLON,1.600000e+06,3.978571e+06,2.486607,0.000405,0.029778,0.013747,0.013747,3.486496
321,76520000,PALMIRA VALLE DEL CAUCA,2,Camión Rígido de 2 ejes,003401,JABON; PRODUCTOS TENSOACTIVOS USADOS COMO JABO...,1.900000e+06,3.449330e+06,1.815437,0.000240,0.039647,0.011893,0.011893,3.016280
380,76892000,YUMBO VALLE DEL CAUCA,2,Camión Rígido de 2 ejes,008703,COCHES DE TURISMO Y DEMAS VEHICULOS AUTOMOVILE...,1.435268e+06,3.696254e+06,2.575306,0.004539,0.020255,0.011727,0.011727,2.974241
316,76520000,PALMIRA VALLE DEL CAUCA,2,Camión Rígido de 2 ejes,001101,HARINA DE TRIGO O MORCAJO,1.650000e+06,3.851042e+06,2.333965,0.000835,0.024706,0.010824,0.010824,2.745105
62,19142000,CALOTO CAUCA,2,Camión Rígido de 2 ejes,000407,HUEVOS DE AVES CON CASCARA(CASCARON) FRESCOS; ...,1.145222e+06,2.894029e+06,2.527046,0.006520,0.013151,0.009118,0.009118,2.312482
329,76520000,PALMIRA VALLE DEL CAUCA,2,Camión Rígido de 2 ejes,009980,PRODUCTOS VARIOS,1.242857e+06,3.321337e+06,2.672340,0.001101,0.015981,0.008395,0.008395,2.129144
259,76001000,CALI VALLE DEL CAUCA,2,Camión Rígido de 2 ejes,009880,MISCELANEOS CONTENIDOS EN PAQUETES ( PAQUETEO ),1.359850e+06,1.777463e+06,1.307102,0.032515,0.024469,0.007631,0.007631,1.935247
283,76111000,GUADALAJARA DE BUGA VALLE DEL CAUCA,2,Camión Rígido de 2 ejes,002309,PREPARACIONES DEL TIPO DE LAS UTILIZADAS PARA ...,1.372459e+06,2.262742e+06,1.648677,0.010592,0.018750,0.007335,0.007335,1.860307


... (7 outputs left out)

,PERIODO,VAR_FLETE_TORNQVIST_PCT,VAR_FLETE_A_PCT,CELDAS_A_COMPARABLES
15,202105,20.901337,21.676152,74
16,202106,-0.578763,3.094577,78
35,202301,43.024373,47.873909,167
55,202409,4.782095,2.454177,245



Concentración de eslabones críticos:


,ESLABON,N_CELDAS,HHI_PESOS,TOP5_PESO_PCT,TOP5_CONTRIB_ABS_PCT
0,202104->202105,85,0.026907,26.662812,39.640991
1,202105->202106,94,0.027813,28.440092,34.052816
2,202212->202301,415,0.010384,15.917255,20.523212
3,202408->202409,682,0.007432,12.766497,17.145239



PASS — CELDA 8H

La celda NO modifica la Variable 2.

Solo audita:
- mayo de 2021,
- enero de 2023,
- septiembre de 2024,
- concentración,
- sensibilidad de composición.

Si estas pruebas son coherentes,
podemos congelar formalmente
el índice regional de fletes.



In [29]:
# ============================================================
# CELDA 8I — CONGELACIÓN DEFINITIVA DE LA VARIABLE 2
# ÍNDICE REGIONAL DE FLETES HACIA PASTO
# ============================================================
#
# OBJETIVO:
# Dejar formalmente terminada la Variable 2.
#
# GENERA:
# 1. Serie completa del índice: 202001–202412.
# 2. Variable econométrica: 202002–202412 (59 variaciones).
# 3. Resumen de robustez.
# 4. Metadata metodológica.
# 5. SHA-256 de todos los archivos finales.
#
# NO descarga RNDC.
# NO recalcula los 60 Excel.
# NO modifica la metodología.
# ============================================================

import pandas as pd
import numpy as np
import hashlib
import json
import time
from pathlib import Path
from datetime import datetime, timezone


# ------------------------------------------------------------
# 1. RUTAS EXISTENTES
# ------------------------------------------------------------

assert Path(RUTA_MASTER).exists(), (
    "No existe el MASTER compacto."
)

assert Path(RUTA_PANEL_J).exists(), (
    "No existe el panel jerárquico."
)

assert Path(RUTA_VARIABLE2_PARQUET).exists(), (
    "No existe la Variable 2 generada por 8G."
)


# ------------------------------------------------------------
# 2. DIRECTORIO DE PRODUCTOS FINALES
# ------------------------------------------------------------

DIR_VARIABLE2_FINAL = (
    ROOT
    /
    "VARIABLE2_FINAL"
)

DIR_VARIABLE2_FINAL.mkdir(
    parents=True,
    exist_ok=True
)


RUTA_SERIE_FINAL = (
    DIR_VARIABLE2_FINAL
    /
    "indice_flete_regional_pasto_2020_2024.parquet"
)


RUTA_MODELO_FINAL = (
    DIR_VARIABLE2_FINAL
    /
    "variable2_flete_regional_modelo_2020_2024.csv"
)


RUTA_ROBUSTEZ_FINAL = (
    DIR_VARIABLE2_FINAL
    /
    "robustez_variable2.csv"
)


RUTA_AUDITORIA_FINAL = (
    DIR_VARIABLE2_FINAL
    /
    "episodios_auditados_variable2.csv"
)


RUTA_METADATA_FINAL = (
    DIR_VARIABLE2_FINAL
    /
    "metadata_variable2.json"
)


# ------------------------------------------------------------
# 3. CARGAR SERIE FINAL
# ------------------------------------------------------------

serie_v2 = pd.read_parquet(
    RUTA_VARIABLE2_PARQUET
)


serie_v2["PERIODO"] = (
    serie_v2["PERIODO"]
    .astype(str)
    .str.zfill(6)
)


serie_v2 = (
    serie_v2
    .sort_values("PERIODO")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. VALIDACIONES DURAS
# ------------------------------------------------------------

assert len(serie_v2) == 60

assert serie_v2["PERIODO"].nunique() == 60

assert serie_v2["PERIODO"].iloc[0] == "202001"

assert serie_v2["PERIODO"].iloc[-1] == "202412"


assert np.isclose(
    serie_v2.loc[
        serie_v2["PERIODO"].eq("202001"),
        "INDICE_FLETE_TORNQVIST"
    ].iloc[0],
    100.0
)


assert (
    serie_v2[
        "VAR_FLETE_TORNQVIST_PCT"
    ]
    .notna()
    .sum()
    ==
    59
)


assert np.isfinite(
    serie_v2.loc[
        serie_v2["PERIODO"].ne("202001"),
        "VAR_FLETE_TORNQVIST_PCT"
    ]
).all()


# ------------------------------------------------------------
# 5. VARIABLE ECONOMÉTRICA
#
# Enero 2020 queda como base=100.
# La primera variación observable es febrero 2020.
# ------------------------------------------------------------

variable2_modelo = (

    serie_v2.loc[
        serie_v2[
            "VAR_FLETE_TORNQVIST_PCT"
        ].notna(),
        [
            "PERIODO",

            "VAR_FLETE_TORNQVIST_PCT",

            "VAR_FLETE_FISHER_PCT",

            "VAR_FLETE_CRUDO_PCT",

            "VAR_FLETE_SIN_2000_PCT",

            "INDICE_FLETE_TORNQVIST",

            "INDICE_FLETE_FISHER",

            "CELDAS_COMPARABLES",

            "COB_VALOR_T_PCT",

            "COB_VIAJES_T_PCT",

            "COB_KG_T_PCT"
        ]
    ]

    .copy()

    .reset_index(drop=True)
)


assert len(variable2_modelo) == 59


# ------------------------------------------------------------
# 6. FECHA MENSUAL PARA FUSIÓN ECONOMÉTRICA
# ------------------------------------------------------------

variable2_modelo[
    "FECHA"
] = pd.to_datetime(
    variable2_modelo["PERIODO"],
    format="%Y%m"
)


# ------------------------------------------------------------
# 7. EPISODIOS AUDITADOS
#
# No implica eliminación.
# Solo conserva trazabilidad de las auditorías.
# ------------------------------------------------------------

episodios_auditados = pd.DataFrame({

    "PERIODO": [
        "202004",
        "202105",
        "202106",
        "202301",
        "202302",
        "202409"
    ],

    "MOTIVO_AUDITORIA": [

        "Cobertura RNDC inferior al entorno temporal; "
        "contracción simultánea de viajes, kg y valor.",

        "Cobertura excepcionalmente baja durante episodio "
        "de bloqueos; índice robusto a agregación.",

        "Eslabón posterior a mayo de 2021; "
        "sensibilidad de composición revisada.",

        "Variación mensual elevada; "
        "robusta a Fisher y agregación.",

        "Cobertura inferior al entorno temporal; "
        "mes conservado tras auditoría.",

        "Resultado sensible a depuración/composición; "
        "signo positivo estable entre especificaciones."
    ],

    "DECISION": [
        "CONSERVAR",
        "CONSERVAR",
        "CONSERVAR",
        "CONSERVAR",
        "CONSERVAR",
        "CONSERVAR"
    ]
})


# ------------------------------------------------------------
# 8. RESUMEN DE ROBUSTEZ
#
# Usa los resultados ya calculados en 8G y 8H.
# ------------------------------------------------------------

assert "corr_TF" in globals()
assert "mae_TF" in globals()

assert "corr_crudo" in globals()
assert "mae_crudo" in globals()

assert "corr_2000" in globals()
assert "mae_2000" in globals()

assert "corr_AB" in globals()
assert "mae_AB" in globals()


robustez_final = pd.DataFrame({

    "COMPARACION": [

        "Törnqvist principal vs Fisher",

        "Törnqvist principal vs serie cruda",

        "Törnqvist principal vs sensibilidad sin $2.000",

        (
            "Celda B origen×configuración×mercancía "
            "vs celda A origen×configuración"
        )
    ],

    "CORRELACION": [

        float(corr_TF),

        float(corr_crudo),

        float(corr_2000),

        float(corr_AB)
    ],

    "MAE_PP": [

        float(mae_TF),

        float(mae_crudo),

        float(mae_2000),

        float(mae_AB)
    ]
})


# ------------------------------------------------------------
# 9. GUARDAR SERIE FINAL
# ------------------------------------------------------------

serie_v2.to_parquet(
    RUTA_SERIE_FINAL,
    index=False,
    compression="zstd"
)


variable2_modelo.to_csv(
    RUTA_MODELO_FINAL,
    index=False,
    encoding="utf-8-sig"
)


robustez_final.to_csv(
    RUTA_ROBUSTEZ_FINAL,
    index=False,
    encoding="utf-8-sig"
)


episodios_auditados.to_csv(
    RUTA_AUDITORIA_FINAL,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 10. HASH
# ------------------------------------------------------------

def sha256_final(ruta):

    h = hashlib.sha256()

    with open(ruta, "rb") as f:

        for bloque in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            h.update(bloque)

    return h.hexdigest()


HASH_MASTER = sha256_final(
    RUTA_MASTER
)

HASH_PANEL_J = sha256_final(
    RUTA_PANEL_J
)

HASH_SERIE = sha256_final(
    RUTA_SERIE_FINAL
)

HASH_MODELO = sha256_final(
    RUTA_MODELO_FINAL
)


# ------------------------------------------------------------
# 11. METADATA METODOLÓGICA
# ------------------------------------------------------------

metadata_final = {

    "nombre_variable":
        "Indice regional de fletes hacia Pasto",

    "variable_modelo":
        "VAR_FLETE_TORNQVIST_PCT",

    "fuente":
        "Registro Nacional de Despachos de Carga (RNDC), "
        "Ministerio de Transporte de Colombia",

    "destino":
        "Pasto, Nariño",

    "codigo_dane_destino":
        int(CODIGO_PASTO),

    "periodo_indice":
        "2020-01 a 2024-12",

    "n_meses_indice":
        60,

    "periodo_variaciones":
        "2020-02 a 2024-12",

    "n_variaciones":
        59,

    "base_indice":
        "2020-01 = 100",

    "unidad_elemental":
        (
            "municipio de origen × configuración vehicular "
            "× mercancía"
        ),

    "precio_elemental":
        (
            "VALORESPAGADOS / "
            "(VIAJESTOTALES - VIAJESVALORCERO)"
        ),

    "indice_principal":
        "Törnqvist encadenado mensual",

    "ponderacion":
        (
            "promedio de participaciones del valor pagado "
            "de cada celda en t-1 y t"
        ),

    "matching":
        (
            "solo celdas presentes simultáneamente "
            "en t-1 y t"
        ),

    "depuracion":
        (
            "regla jerárquica robusta sin información futura; "
            "solo exclusión de valores anormalmente bajos "
            "cuando existe evidencia conjunta suficiente"
        ),

    "ventana_historia":
        "12 meses anteriores",

    "z_critico":
        3.5,

    "historia_minima":
        4,

    "pares_misma_ruta_minimos":
        5,

    "otras_rutas_minimas":
        3,

    "extremos_altos":
        "conservados",

    "imputacion":
        False,

    "winsorizacion":
        False,

    "uso_informacion_futura":
        False,

    "sensibilidades": [

        "Fisher sobre exactamente las mismas celdas",

        "Törnqvist con datos monetariamente válidos sin "
        "depuración jerárquica",

        "Törnqvist excluyendo precios exactamente iguales "
        "a $2.000",

        "Törnqvist agregado por origen × configuración"
    ],

    "episodios_auditados": [

        "2020-04",
        "2021-05",
        "2021-06",
        "2023-01",
        "2023-02",
        "2024-09"
    ],

    "limitacion_temporal":
        (
            "La fuente mensual reproducible utilizada para "
            "el índice regional está disponible desde enero "
            "de 2020. Enero de 2020 es base 100 y la primera "
            "variación observable es febrero de 2020."
        ),

    "sha256_master":
        HASH_MASTER,

    "sha256_panel_jerarquico":
        HASH_PANEL_J,

    "sha256_serie_final":
        HASH_SERIE,

    "sha256_variable_modelo":
        HASH_MODELO,

    "fecha_congelacion_utc":
        datetime.now(
            timezone.utc
        ).isoformat(
            timespec="seconds"
        )
}


with open(
    RUTA_METADATA_FINAL,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata_final,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 12. PRUEBA DE CARGA RÁPIDA
# ------------------------------------------------------------

inicio = time.perf_counter()


prueba_rapida = pd.read_parquet(
    RUTA_MASTER
)


segundos_carga_master = (
    time.perf_counter()
    -
    inicio
)


assert len(
    prueba_rapida
) == 78819


del prueba_rapida


# ------------------------------------------------------------
# 13. MOSTRAR RESULTADOS FINALES
# ------------------------------------------------------------

print("=" * 110)
print("VARIABLE 2 — CONGELACIÓN DEFINITIVA")
print("=" * 110)


print(
    "\nSerie del índice:"
)

display(
    serie_v2.tail(12)
)


print(
    "\nResumen de robustez:"
)

display(
    robustez_final
)


print(
    "\nEpisodios auditados:"
)

display(
    episodios_auditados
)


print(
    "\nCarga del MASTER compacto:",
    round(
        segundos_carga_master,
        4
    ),
    "segundos"
)


print(
    "\nArchivos finales:"
)

print(
    "MASTER compacto:",
    RUTA_MASTER
)

print(
    "Serie índice:",
    RUTA_SERIE_FINAL
)

print(
    "Variable para modelo:",
    RUTA_MODELO_FINAL
)

print(
    "Robustez:",
    RUTA_ROBUSTEZ_FINAL
)

print(
    "Auditoría:",
    RUTA_AUDITORIA_FINAL
)

print(
    "Metadata:",
    RUTA_METADATA_FINAL
)


print(
    "\nSHA-256 serie final:"
)

print(
    HASH_SERIE
)


print(
    "\nSHA-256 variable econométrica:"
)

print(
    HASH_MODELO
)


# ------------------------------------------------------------
# 14. PASS FINAL
# ------------------------------------------------------------

assert len(variable2_modelo) == 59

assert (
    variable2_modelo[
        "PERIODO"
    ].iloc[0]
    ==
    "202002"
)

assert (
    variable2_modelo[
        "PERIODO"
    ].iloc[-1]
    ==
    "202412"
)


print(
    "\n"
    + "=" * 110
)

print(
    "PASS — VARIABLE 2 CONGELADA"
)

print(
    "=" * 110
)

print(
    """
Variable 2 terminada:

ÍNDICE REGIONAL DE FLETES HACIA PASTO

Principal:
Törnqvist encadenado mensual.

Periodo del índice:
2020-01 a 2024-12.

Variación econométrica:
2020-02 a 2024-12.

No imputación.
No winsorización.
No look-ahead.
Extremos altos conservados.

La reconstrucción pesada RNDC NO forma parte
del flujo normal del notebook final.

A partir de este punto la Variable 2 no debe
redefinirse salvo que cambie explícitamente
el diseño de investigación.
"""
)

VARIABLE 2 — CONGELACIÓN DEFINITIVA

Serie del índice:


,PERIODO,INDICE_FLETE_TORNQVIST,INDICE_FLETE_FISHER,VAR_FLETE_TORNQVIST_PCT,VAR_FLETE_FISHER_PCT,CELDAS_COMPARABLES,COB_VALOR_T_PCT,COB_VIAJES_T_PCT,COB_KG_T_PCT,VAR_FLETE_CRUDO_PCT,VAR_FLETE_SIN_2000_PCT
48,202401,116.075460,115.436282,-0.562039,-0.572960,592.0,80.427594,83.849507,80.047566,-0.586663,-0.562039
49,202402,116.362000,115.545779,0.246857,0.094855,601.0,79.112398,81.485188,79.524147,-0.021832,0.246857
50,202403,116.573839,115.732118,0.182052,0.161268,611.0,82.229751,84.666667,81.001068,0.126039,0.182052
51,202404,116.591723,115.869293,0.015341,0.118528,581.0,79.353807,82.510121,78.467694,-0.003519,0.015341
52,202405,116.176317,115.406872,-0.356291,-0.399089,632.0,83.362948,86.113344,84.323370,-0.212420,-0.356291
53,202406,116.268610,115.519928,0.079441,0.097963,651.0,83.179503,85.559039,82.170712,0.031040,0.079441
54,202407,117.147880,116.341111,0.756240,0.710858,649.0,81.470975,84.444444,82.132564,0.685232,0.756240
55,202408,117.043505,116.394423,-0.089096,0.045824,671.0,82.242830,80.455547,72.675697,0.054354,-0.089096
56,202409,122.640638,121.964921,4.782095,4.785880,682.0,80.776420,81.755151,72.071188,11.594735,4.782095
57,202410,123.535943,122.898745,0.730023,0.765650,685.0,83.264783,85.078667,84.122246,0.686764,0.730023



Resumen de robustez:


,COMPARACION,CORRELACION,MAE_PP
0,Törnqvist principal vs Fisher,0.999096,0.174071
1,Törnqvist principal vs serie cruda,0.991811,0.205234
2,Törnqvist principal vs sensibilidad sin $2.000,1.000000,0.000141
3,Celda B origen×configuración×mercancía vs celd...,0.976188,1.175909



Episodios auditados:


,PERIODO,MOTIVO_AUDITORIA,DECISION
0,202004,Cobertura RNDC inferior al entorno temporal; c...,CONSERVAR
1,202105,Cobertura excepcionalmente baja durante episod...,CONSERVAR
2,202106,Eslabón posterior a mayo de 2021; sensibilidad...,CONSERVAR
3,202301,Variación mensual elevada; robusta a Fisher y ...,CONSERVAR
4,202302,Cobertura inferior al entorno temporal; mes co...,CONSERVAR
5,202409,Resultado sensible a depuración/composición; s...,CONSERVAR



Carga del MASTER compacto: 0.368 segundos

Archivos finales:
MASTER compacto: /content/RNDC_PASTO/RNDC_PASTO_MASTER_2020_2024.parquet
Serie índice: /content/RNDC_PASTO/VARIABLE2_FINAL/indice_flete_regional_pasto_2020_2024.parquet
Variable para modelo: /content/RNDC_PASTO/VARIABLE2_FINAL/variable2_flete_regional_modelo_2020_2024.csv
Robustez: /content/RNDC_PASTO/VARIABLE2_FINAL/robustez_variable2.csv
Auditoría: /content/RNDC_PASTO/VARIABLE2_FINAL/episodios_auditados_variable2.csv
Metadata: /content/RNDC_PASTO/VARIABLE2_FINAL/metadata_variable2.json

SHA-256 serie final:
4336398949cbc5c74a95b7bf982f241a5b4e5564e44b6d67313536c0a4fa0e19

SHA-256 variable econométrica:
aaec180a88d32bbccbfb13b94be4784c9e9b4c6f23d9a93b7475b1acd1bd3891

PASS — VARIABLE 2 CONGELADA

Variable 2 terminada:

ÍNDICE REGIONAL DE FLETES HACIA PASTO

Principal:
Törnqvist encadenado mensual.

Periodo del índice:
2020-01 a 2024-12.

Variación econométrica:
2020-02 a 2024-12.

No imputación.
No winsorización.
No look-ah